# CogAttention — Proactive Interference

**Track:** Attention — Attention Capacity
**Benchmark:** CogAttention v1.0
**Task:** interference

---

## Methodology

Tests proactive interference resistance. A series of updates assigns new values to the same keys repeatedly; the model must report only the final value. Based on PI-LLM (Wang & Sun, 2025; arXiv:2506.08184).

### Cognitive Science Grounding

- **Proactive Interference** (Wang & Sun, 2025; arXiv:2506.08184): old memories inhibit retrieval of new ones
- In Transformers, this maps to KV-cache attention sinks anchoring to initial token states
- Qwen-72B scores 1.0; Llama-8B collapses to 0.0 at Expert — the largest cross-model gap in CogAttention

### Difficulty Scaling

Easy: 3 updates, 1 key | Medium: 8 updates, 2 keys | Hard: 15 updates, 3 keys | Expert: 25 updates, 4 keys | Frontier: 50 updates, 8 keys

### Scoring

One assertion per key — checks whether the model reports the FINAL value, not any earlier (interfering) value.

All instances are procedurally generated from a seed with programmatic ground truth.
No static datasets. 7 layers of contamination resistance including canary strings,
zero lexical overlap (Semantic NIAH), and seed-based regeneration.

---

`<!-- COGATTENTION-BENCH-CANARY-82D29C1863AF -->`


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 2: Imports + Inline Helpers
# CogAttention — Attention Capacity
# ══════════════════════════════════════════════════════════════════════

import kaggle_benchmarks as kbench

import json
import re

def extract_answer_block(response):
    for pat in [r"ANSWER:\s*(.*)", r"Answer:\s*(.*)", r"answer:\s*(.*)"]:
        match = re.search(pat, response, re.DOTALL | re.IGNORECASE)
        if match:
            return match.group(1).strip()
    return response.strip()

def extract_numbered_answers(response):
    answer_block = extract_answer_block(response)
    results = {}
    matches = re.findall(
        r"(\d+)\s*[.):\-]\s*(.+?)(?=\n\d+\s*[.):\-]|\Z)",
        answer_block, re.DOTALL,
    )
    for num, val in matches:
        results[num] = val.strip().rstrip(".")
    return results

def extract_list_items(response):
    answer_block = extract_answer_block(response)
    bullets = re.findall(r"[-\u2022]\s*(.+?)(?:\n|$)", answer_block)
    if bullets:
        return [b.strip().rstrip(".") for b in bullets]
    numeric_items = re.findall(
        r'[\$]?\d{1,3}(?:,\d{3})*(?:\.\d+)?(?:\s*(?:\xb0[CF]|mg/L|%|\$))?',
        answer_block,
    )
    if numeric_items and len(numeric_items) >= 2:
        return [x.strip() for x in numeric_items]
    if "," in answer_block:
        items = [x.strip().rstrip(".") for x in answer_block.split(",")]
        return [x for x in items if x]
    lines = [l.strip().rstrip(".") for l in answer_block.split("\n") if l.strip()]
    return lines if lines else ([answer_block] if answer_block else [])

def extract_person_item_pairs(response):
    answer_block = extract_answer_block(response)
    results = {}
    for pat in [
        r"[-\u2022]?\s*(\w+)\s*:\s*(.+?)(?:\n|$)",
        r"[-\u2022]?\s*(\w+)\s+holds?\s+(?:a\s+)?(.+?)(?:\n|$)",
    ]:
        matches = re.findall(pat, answer_block, re.IGNORECASE)
        if matches:
            for name, item in matches:
                results[name.strip()] = item.strip().rstrip(".")
            break
    return results

def fuzzy_value_match(predicted, gold):
    pred_clean = re.sub(r"\s+", " ", predicted.strip().lower())
    gold_clean = re.sub(r"\s+", " ", gold.strip().lower())
    if pred_clean == gold_clean:
        return True
    if gold_clean in pred_clean:
        return True
    try:
        pred_num = float(re.sub(r"[,$%\xb0]", "", predicted))
        gold_num = float(re.sub(r"[,$%\xb0]", "", gold))
        return pred_num == gold_num
    except (ValueError, TypeError):
        pass
    return False

def _escape_for_regex(s):
    return re.escape(s).replace(r"\ ", r"\s+")


def run_assertions_interference(response, gold, kbench):
    for key in gold["key_names"]:
        gold_val = gold["final_values"][key]
        pattern = rf"(?i){re.escape(key)}\s*[:\-=]\s*.*{_escape_for_regex(gold_val)}"
        kbench.assertions.assert_contains_regex(
            pattern, response,
            expectation=f"Final value of '{key}' should be '{gold_val}'"
        )


print("CogAttention helpers loaded")
print(f"Task types: ['interference']")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 3: Task Definitions + Embedded Dataset
# ══════════════════════════════════════════════════════════════════════


@kbench.task(name="cogattention_interference")
def cogattention_interference(llm, prompt: str, gold_json: str, task_id: str, difficulty: str):
    """CogAttention interference task."""
    response = llm.prompt(prompt)
    gold = json.loads(gold_json)
    run_assertions_interference(response, gold, kbench)


# ── Embedded dataset ──────────────────────────────────────────────────
DATASET = json.loads(r'''
[
 {
  "task_id": "interference_easy_000",
  "task_type": "interference",
  "difficulty": "Easy",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: delegate is now set to 'Kotor'\n  Update: delegate is now set to 'Gdansk'\n  Update: delegate is now set to 'Plovdiv'\n\nWhat is the FINAL value of each record?\nANSWER:\n- delegate: [final value]",
  "gold_json": "{\"final_values\": {\"delegate\": \"Plovdiv\"}, \"key_names\": [\"delegate\"]}"
 },
 {
  "task_id": "interference_easy_001",
  "task_type": "interference",
  "difficulty": "Easy",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: registry is now set to '574'\n  Update: registry is now set to '943'\n  Update: registry is now set to '850'\n\nWhat is the FINAL value of each record?\nANSWER:\n- registry: [final value]",
  "gold_json": "{\"final_values\": {\"registry\": \"850\"}, \"key_names\": [\"registry\"]}"
 },
 {
  "task_id": "interference_easy_002",
  "task_type": "interference",
  "difficulty": "Easy",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: assignment is now set to 'Mandalay'\n  Update: assignment is now set to 'Plovdiv'\n  Update: assignment is now set to 'Jaipur'\n\nWhat is the FINAL value of each record?\nANSWER:\n- assignment: [final value]",
  "gold_json": "{\"final_values\": {\"assignment\": \"Jaipur\"}, \"key_names\": [\"assignment\"]}"
 },
 {
  "task_id": "interference_easy_003",
  "task_type": "interference",
  "difficulty": "Easy",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: coordinator is now set to 'Zain'\n  Update: coordinator is now set to 'Bashir'\n  Update: coordinator is now set to 'Willa'\n\nWhat is the FINAL value of each record?\nANSWER:\n- coordinator: [final value]",
  "gold_json": "{\"final_values\": {\"coordinator\": \"Willa\"}, \"key_names\": [\"coordinator\"]}"
 },
 {
  "task_id": "interference_easy_004",
  "task_type": "interference",
  "difficulty": "Easy",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: dispatch is now set to 'Zanzibar'\n  Update: dispatch is now set to 'Gdansk'\n  Update: dispatch is now set to 'Reykjavik'\n\nWhat is the FINAL value of each record?\nANSWER:\n- dispatch: [final value]",
  "gold_json": "{\"final_values\": {\"dispatch\": \"Reykjavik\"}, \"key_names\": [\"dispatch\"]}"
 },
 {
  "task_id": "interference_easy_005",
  "task_type": "interference",
  "difficulty": "Easy",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: delegate is now set to 'Tallinn'\n  Update: delegate is now set to 'Jaipur'\n  Update: delegate is now set to 'Oulu'\n\nWhat is the FINAL value of each record?\nANSWER:\n- delegate: [final value]",
  "gold_json": "{\"final_values\": {\"delegate\": \"Oulu\"}, \"key_names\": [\"delegate\"]}"
 },
 {
  "task_id": "interference_easy_006",
  "task_type": "interference",
  "difficulty": "Easy",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: assignment is now set to '627'\n  Update: assignment is now set to '694'\n  Update: assignment is now set to '455'\n\nWhat is the FINAL value of each record?\nANSWER:\n- assignment: [final value]",
  "gold_json": "{\"final_values\": {\"assignment\": \"455\"}, \"key_names\": [\"assignment\"]}"
 },
 {
  "task_id": "interference_easy_007",
  "task_type": "interference",
  "difficulty": "Easy",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: coordinator is now set to 'Reykjavik'\n  Update: coordinator is now set to 'Recife'\n  Update: coordinator is now set to 'Oulu'\n\nWhat is the FINAL value of each record?\nANSWER:\n- coordinator: [final value]",
  "gold_json": "{\"final_values\": {\"coordinator\": \"Oulu\"}, \"key_names\": [\"coordinator\"]}"
 },
 {
  "task_id": "interference_medium_008",
  "task_type": "interference",
  "difficulty": "Medium",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: coordinator is now set to 'Ulaanbaatar'\n  Update: delegate is now set to 'Jaipur'\n  Update: coordinator is now set to 'Jaipur'\n  Update: delegate is now set to 'Kotor'\n  Update: coordinator is now set to 'Fez'\n  Update: delegate is now set to 'Cusco'\n  Update: coordinator is now set to 'Tallinn'\n  Update: delegate is now set to 'Jaipur'\n  Update: coordinator is now set to 'Ulaanbaatar'\n  Update: delegate is now set to 'Tbilisi'\n  Update: coordinator is now set to 'Cartagena'\n  Update: delegate is now set to 'Tallinn'\n\nWhat is the FINAL value of each record?\nANSWER:\n- coordinator: [final value]\n- delegate: [final value]",
  "gold_json": "{\"final_values\": {\"coordinator\": \"Cartagena\", \"delegate\": \"Tallinn\"}, \"key_names\": [\"coordinator\", \"delegate\"]}"
 },
 {
  "task_id": "interference_medium_009",
  "task_type": "interference",
  "difficulty": "Medium",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: reference is now set to 'Valetta'\n  Update: assignment is now set to 'Cusco'\n  Update: reference is now set to 'Kumasi'\n  Update: assignment is now set to 'Kumasi'\n  Update: reference is now set to 'Fez'\n  Update: assignment is now set to 'Oulu'\n  Update: reference is now set to 'Cartagena'\n  Update: assignment is now set to 'Gdansk'\n  Update: reference is now set to 'Ulaanbaatar'\n  Update: assignment is now set to 'Fez'\n  Update: reference is now set to 'Mandalay'\n  Update: assignment is now set to 'Gdansk'\n\nWhat is the FINAL value of each record?\nANSWER:\n- reference: [final value]\n- assignment: [final value]",
  "gold_json": "{\"final_values\": {\"reference\": \"Mandalay\", \"assignment\": \"Gdansk\"}, \"key_names\": [\"reference\", \"assignment\"]}"
 },
 {
  "task_id": "interference_medium_010",
  "task_type": "interference",
  "difficulty": "Medium",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: registry is now set to '398'\n  Update: reference is now set to '662'\n  Update: registry is now set to '187'\n  Update: reference is now set to '985'\n  Update: registry is now set to '513'\n  Update: reference is now set to '142'\n  Update: registry is now set to '388'\n  Update: reference is now set to '849'\n  Update: registry is now set to '308'\n  Update: reference is now set to '158'\n  Update: registry is now set to '805'\n  Update: reference is now set to '297'\n\nWhat is the FINAL value of each record?\nANSWER:\n- registry: [final value]\n- reference: [final value]",
  "gold_json": "{\"final_values\": {\"registry\": \"805\", \"reference\": \"297\"}, \"key_names\": [\"registry\", \"reference\"]}"
 },
 {
  "task_id": "interference_medium_011",
  "task_type": "interference",
  "difficulty": "Medium",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: coordinator is now set to 'Maren'\n  Update: delegate is now set to 'Elara'\n  Update: coordinator is now set to 'Wren'\n  Update: delegate is now set to 'Lumi'\n  Update: coordinator is now set to 'Tariq'\n  Update: delegate is now set to 'Adaeze'\n  Update: coordinator is now set to 'Joelle'\n  Update: delegate is now set to 'Bashir'\n  Update: coordinator is now set to 'Ugo'\n  Update: delegate is now set to 'Colette'\n  Update: coordinator is now set to 'Dmitri'\n  Update: delegate is now set to 'Magnus'\n\nWhat is the FINAL value of each record?\nANSWER:\n- coordinator: [final value]\n- delegate: [final value]",
  "gold_json": "{\"final_values\": {\"coordinator\": \"Dmitri\", \"delegate\": \"Magnus\"}, \"key_names\": [\"coordinator\", \"delegate\"]}"
 },
 {
  "task_id": "interference_medium_012",
  "task_type": "interference",
  "difficulty": "Medium",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: reference is now set to '681'\n  Update: registry is now set to '630'\n  Update: reference is now set to '844'\n  Update: registry is now set to '326'\n  Update: reference is now set to '441'\n  Update: registry is now set to '611'\n  Update: reference is now set to '175'\n  Update: registry is now set to '103'\n  Update: reference is now set to '867'\n  Update: registry is now set to '175'\n  Update: reference is now set to '542'\n  Update: registry is now set to '238'\n\nWhat is the FINAL value of each record?\nANSWER:\n- reference: [final value]\n- registry: [final value]",
  "gold_json": "{\"final_values\": {\"reference\": \"542\", \"registry\": \"238\"}, \"key_names\": [\"reference\", \"registry\"]}"
 },
 {
  "task_id": "interference_medium_013",
  "task_type": "interference",
  "difficulty": "Medium",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: destination is now set to 'Colette'\n  Update: reference is now set to 'Freya'\n  Update: destination is now set to 'Freya'\n  Update: reference is now set to 'Bashir'\n  Update: destination is now set to 'Maren'\n  Update: reference is now set to 'Soren'\n  Update: destination is now set to 'Magnus'\n  Update: reference is now set to 'Ravi'\n  Update: destination is now set to 'Adaeze'\n  Update: reference is now set to 'Vesna'\n  Update: destination is now set to 'Gael'\n  Update: reference is now set to 'Idris'\n\nWhat is the FINAL value of each record?\nANSWER:\n- destination: [final value]\n- reference: [final value]",
  "gold_json": "{\"final_values\": {\"destination\": \"Gael\", \"reference\": \"Idris\"}, \"key_names\": [\"destination\", \"reference\"]}"
 },
 {
  "task_id": "interference_medium_014",
  "task_type": "interference",
  "difficulty": "Medium",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: coordinator is now set to '526'\n  Update: dispatch is now set to '209'\n  Update: coordinator is now set to '646'\n  Update: dispatch is now set to '159'\n  Update: coordinator is now set to '885'\n  Update: dispatch is now set to '985'\n  Update: coordinator is now set to '640'\n  Update: dispatch is now set to '507'\n  Update: coordinator is now set to '823'\n  Update: dispatch is now set to '366'\n  Update: coordinator is now set to '530'\n  Update: dispatch is now set to '553'\n\nWhat is the FINAL value of each record?\nANSWER:\n- coordinator: [final value]\n- dispatch: [final value]",
  "gold_json": "{\"final_values\": {\"coordinator\": \"530\", \"dispatch\": \"553\"}, \"key_names\": [\"coordinator\", \"dispatch\"]}"
 },
 {
  "task_id": "interference_medium_015",
  "task_type": "interference",
  "difficulty": "Medium",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: dispatch is now set to 'Vesna'\n  Update: delegate is now set to 'Haruto'\n  Update: dispatch is now set to 'Soren'\n  Update: delegate is now set to 'Tala'\n  Update: dispatch is now set to 'Kaia'\n  Update: delegate is now set to 'Priya'\n  Update: dispatch is now set to 'Nico'\n  Update: delegate is now set to 'Ines'\n  Update: dispatch is now set to 'Zora'\n  Update: delegate is now set to 'Soren'\n  Update: dispatch is now set to 'Tariq'\n  Update: delegate is now set to 'Elara'\n\nWhat is the FINAL value of each record?\nANSWER:\n- dispatch: [final value]\n- delegate: [final value]",
  "gold_json": "{\"final_values\": {\"dispatch\": \"Tariq\", \"delegate\": \"Elara\"}, \"key_names\": [\"dispatch\", \"delegate\"]}"
 },
 {
  "task_id": "interference_hard_016",
  "task_type": "interference",
  "difficulty": "Hard",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: dispatch is now set to 'Nalini'\n  Update: delegate is now set to 'Qadir'\n  Update: assignment is now set to 'Gael'\n  Update: dispatch is now set to 'Uma'\n  Update: delegate is now set to 'Tariq'\n  Update: assignment is now set to 'Ravi'\n  Update: dispatch is now set to 'Dariush'\n  Update: delegate is now set to 'Hana'\n  Update: assignment is now set to 'Elara'\n  Update: dispatch is now set to 'Yuki'\n  Update: delegate is now set to 'Olena'\n  Update: assignment is now set to 'Uma'\n  Update: dispatch is now set to 'Vesna'\n  Update: delegate is now set to 'Amara'\n  Update: assignment is now set to 'Sigrid'\n  Update: dispatch is now set to 'Lumi'\n  Update: delegate is now set to 'Ugo'\n  Update: assignment is now set to 'Viktor'\n  Update: dispatch is now set to 'Bashir'\n  Update: delegate is now set to 'Colette'\n  Update: assignment is now set to 'Orla'\n  Update: dispatch is now set to 'Uma'\n  Update: delegate is now set to 'Maren'\n  Update: assignment is now set to 'Freya'\n  Update: dispatch is now set to 'Tariq'\n  Update: delegate is now set to 'Adaeze'\n  Update: assignment is now set to 'Elara'\n  Update: dispatch is now set to 'Femi'\n  Update: delegate is now set to 'Freya'\n  Update: assignment is now set to 'Vesna'\n  Update: dispatch is now set to 'Sigrid'\n  Update: delegate is now set to 'Dmitri'\n  Update: assignment is now set to 'Ugo'\n  Update: dispatch is now set to 'Joelle'\n  Update: delegate is now set to 'Freya'\n  Update: assignment is now set to 'Nico'\n\nWhat is the FINAL value of each record?\nANSWER:\n- dispatch: [final value]\n- delegate: [final value]\n- assignment: [final value]\n\nAlso answer these verification questions (Yes or No):\nV1. Was 'Bashir' ever assigned to dispatch? [Yes/No]",
  "gold_json": "{\"final_values\": {\"dispatch\": \"Joelle\", \"delegate\": \"Freya\", \"assignment\": \"Nico\"}, \"key_names\": [\"dispatch\", \"delegate\", \"assignment\"]}"
 },
 {
  "task_id": "interference_hard_017",
  "task_type": "interference",
  "difficulty": "Hard",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: assignment is now set to 'Orla'\n  Update: dispatch is now set to 'Dariush'\n  Update: contact is now set to 'Vesna'\n  Update: assignment is now set to 'Wren'\n  Update: dispatch is now set to 'Colette'\n  Update: contact is now set to 'Kenji'\n  Update: assignment is now set to 'Vesna'\n  Update: dispatch is now set to 'Nico'\n  Update: contact is now set to 'Haruto'\n  Update: assignment is now set to 'Viktor'\n  Update: dispatch is now set to 'Wren'\n  Update: contact is now set to 'Nalini'\n  Update: assignment is now set to 'Leif'\n  Update: dispatch is now set to 'Yuki'\n  Update: contact is now set to 'Tala'\n  Update: assignment is now set to 'Sigrid'\n  Update: dispatch is now set to 'Joaquin'\n  Update: contact is now set to 'Celine'\n  Update: assignment is now set to 'Zora'\n  Update: dispatch is now set to 'Kenji'\n  Update: contact is now set to 'Haruto'\n  Update: assignment is now set to 'Femi'\n  Update: dispatch is now set to 'Runa'\n  Update: contact is now set to 'Xander'\n  Update: assignment is now set to 'Yuki'\n  Update: dispatch is now set to 'Priya'\n  Update: contact is now set to 'Priya'\n  Update: assignment is now set to 'Joelle'\n  Update: dispatch is now set to 'Elara'\n  Update: contact is now set to 'Vesna'\n  Update: assignment is now set to 'Dmitri'\n  Update: dispatch is now set to 'Leif'\n  Update: contact is now set to 'Dariush'\n  Update: assignment is now set to 'Lumi'\n  Update: dispatch is now set to 'Viktor'\n  Update: contact is now set to 'Uma'\n\nWhat is the FINAL value of each record?\nANSWER:\n- assignment: [final value]\n- dispatch: [final value]\n- contact: [final value]\n\nAlso answer these verification questions (Yes or No):\nV1. Was 'Wren' ever assigned to assignment? [Yes/No]",
  "gold_json": "{\"final_values\": {\"assignment\": \"Lumi\", \"dispatch\": \"Viktor\", \"contact\": \"Uma\"}, \"key_names\": [\"assignment\", \"dispatch\", \"contact\"]}"
 },
 {
  "task_id": "interference_hard_018",
  "task_type": "interference",
  "difficulty": "Hard",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: registry is now set to 'Kumasi'\n  Update: contact is now set to 'Luang Prabang'\n  Update: destination is now set to 'Fez'\n  Update: registry is now set to 'Zanzibar'\n  Update: contact is now set to 'Plovdiv'\n  Update: destination is now set to 'Valetta'\n  Update: registry is now set to 'Luang Prabang'\n  Update: contact is now set to 'Kumasi'\n  Update: destination is now set to 'Oulu'\n  Update: registry is now set to 'Trieste'\n  Update: contact is now set to 'Ulaanbaatar'\n  Update: destination is now set to 'Tallinn'\n  Update: registry is now set to 'Ulaanbaatar'\n  Update: contact is now set to 'Jaipur'\n  Update: destination is now set to 'Kotor'\n  Update: registry is now set to 'Tbilisi'\n  Update: contact is now set to 'Oulu'\n  Update: destination is now set to 'Jaipur'\n  Update: registry is now set to 'Zanzibar'\n  Update: contact is now set to 'Luang Prabang'\n  Update: destination is now set to 'Tallinn'\n  Update: registry is now set to 'Bruges'\n  Update: contact is now set to 'Reykjavik'\n  Update: destination is now set to 'Zanzibar'\n  Update: registry is now set to 'Zanzibar'\n  Update: contact is now set to 'Jaipur'\n  Update: destination is now set to 'Bruges'\n  Update: registry is now set to 'Cartagena'\n  Update: contact is now set to 'Valetta'\n  Update: destination is now set to 'Kumasi'\n  Update: registry is now set to 'Recife'\n  Update: contact is now set to 'Gdansk'\n  Update: destination is now set to 'Tbilisi'\n  Update: registry is now set to 'Kotor'\n  Update: contact is now set to 'Cartagena'\n  Update: destination is now set to 'Reykjavik'\n\nWhat is the FINAL value of each record?\nANSWER:\n- registry: [final value]\n- contact: [final value]\n- destination: [final value]\n\nAlso answer these verification questions (Yes or No):\nV1. Was 'Zanzibar' ever assigned to registry? [Yes/No]",
  "gold_json": "{\"final_values\": {\"registry\": \"Kotor\", \"contact\": \"Cartagena\", \"destination\": \"Reykjavik\"}, \"key_names\": [\"registry\", \"contact\", \"destination\"]}"
 },
 {
  "task_id": "interference_hard_019",
  "task_type": "interference",
  "difficulty": "Hard",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: location is now set to '247'\n  Update: assignment is now set to '283'\n  Update: registry is now set to '856'\n  Update: location is now set to '958'\n  Update: assignment is now set to '368'\n  Update: registry is now set to '170'\n  Update: location is now set to '908'\n  Update: assignment is now set to '475'\n  Update: registry is now set to '755'\n  Update: location is now set to '803'\n  Update: assignment is now set to '945'\n  Update: registry is now set to '881'\n  Update: location is now set to '131'\n  Update: assignment is now set to '912'\n  Update: registry is now set to '764'\n  Update: location is now set to '461'\n  Update: assignment is now set to '993'\n  Update: registry is now set to '409'\n  Update: location is now set to '170'\n  Update: assignment is now set to '737'\n  Update: registry is now set to '970'\n  Update: location is now set to '147'\n  Update: assignment is now set to '478'\n  Update: registry is now set to '422'\n  Update: location is now set to '762'\n  Update: assignment is now set to '556'\n  Update: registry is now set to '232'\n  Update: location is now set to '484'\n  Update: assignment is now set to '435'\n  Update: registry is now set to '339'\n  Update: location is now set to '206'\n  Update: assignment is now set to '278'\n  Update: registry is now set to '321'\n  Update: location is now set to '354'\n  Update: assignment is now set to '897'\n  Update: registry is now set to '464'\n\nWhat is the FINAL value of each record?\nANSWER:\n- location: [final value]\n- assignment: [final value]\n- registry: [final value]\n\nAlso answer these verification questions (Yes or No):\nV1. Was '484' ever assigned to location? [Yes/No]",
  "gold_json": "{\"final_values\": {\"location\": \"354\", \"assignment\": \"897\", \"registry\": \"464\"}, \"key_names\": [\"location\", \"assignment\", \"registry\"]}"
 },
 {
  "task_id": "interference_hard_020",
  "task_type": "interference",
  "difficulty": "Hard",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: contact is now set to 'Ines'\n  Update: location is now set to 'Vesna'\n  Update: coordinator is now set to 'Orla'\n  Update: contact is now set to 'Dmitri'\n  Update: location is now set to 'Kaia'\n  Update: coordinator is now set to 'Uma'\n  Update: contact is now set to 'Zora'\n  Update: location is now set to 'Tariq'\n  Update: coordinator is now set to 'Dmitri'\n  Update: contact is now set to 'Soren'\n  Update: location is now set to 'Lumi'\n  Update: coordinator is now set to 'Dariush'\n  Update: contact is now set to 'Bram'\n  Update: location is now set to 'Idris'\n  Update: coordinator is now set to 'Greta'\n  Update: contact is now set to 'Freya'\n  Update: location is now set to 'Ines'\n  Update: coordinator is now set to 'Dmitri'\n  Update: contact is now set to 'Dariush'\n  Update: location is now set to 'Elara'\n  Update: coordinator is now set to 'Kenji'\n  Update: contact is now set to 'Bram'\n  Update: location is now set to 'Bashir'\n  Update: coordinator is now set to 'Xander'\n  Update: contact is now set to 'Zain'\n  Update: location is now set to 'Dmitri'\n  Update: coordinator is now set to 'Idris'\n  Update: contact is now set to 'Xander'\n  Update: location is now set to 'Adaeze'\n  Update: coordinator is now set to 'Bashir'\n  Update: contact is now set to 'Viktor'\n  Update: location is now set to 'Orla'\n  Update: coordinator is now set to 'Celine'\n  Update: contact is now set to 'Zain'\n  Update: location is now set to 'Freya'\n  Update: coordinator is now set to 'Nico'\n\nWhat is the FINAL value of each record?\nANSWER:\n- contact: [final value]\n- location: [final value]\n- coordinator: [final value]\n\nAlso answer these verification questions (Yes or No):\nV1. Was 'Dmitri' ever assigned to contact? [Yes/No]",
  "gold_json": "{\"final_values\": {\"contact\": \"Zain\", \"location\": \"Freya\", \"coordinator\": \"Nico\"}, \"key_names\": [\"contact\", \"location\", \"coordinator\"]}"
 },
 {
  "task_id": "interference_hard_021",
  "task_type": "interference",
  "difficulty": "Hard",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: liaison is now set to 'Kotor'\n  Update: reference is now set to 'Tallinn'\n  Update: location is now set to 'Valetta'\n  Update: liaison is now set to 'Reykjavik'\n  Update: reference is now set to 'Reykjavik'\n  Update: location is now set to 'Tallinn'\n  Update: liaison is now set to 'Kumasi'\n  Update: reference is now set to 'Luang Prabang'\n  Update: location is now set to 'Luang Prabang'\n  Update: liaison is now set to 'Ulaanbaatar'\n  Update: reference is now set to 'Valetta'\n  Update: location is now set to 'Tbilisi'\n  Update: liaison is now set to 'Kumasi'\n  Update: reference is now set to 'Cartagena'\n  Update: location is now set to 'Zanzibar'\n  Update: liaison is now set to 'Cusco'\n  Update: reference is now set to 'Gdansk'\n  Update: location is now set to 'Mandalay'\n  Update: liaison is now set to 'Plovdiv'\n  Update: reference is now set to 'Fez'\n  Update: location is now set to 'Trieste'\n  Update: liaison is now set to 'Jaipur'\n  Update: reference is now set to 'Zanzibar'\n  Update: location is now set to 'Valetta'\n  Update: liaison is now set to 'Tbilisi'\n  Update: reference is now set to 'Recife'\n  Update: location is now set to 'Jaipur'\n  Update: liaison is now set to 'Recife'\n  Update: reference is now set to 'Jaipur'\n  Update: location is now set to 'Valetta'\n  Update: liaison is now set to 'Zanzibar'\n  Update: reference is now set to 'Kumasi'\n  Update: location is now set to 'Luang Prabang'\n  Update: liaison is now set to 'Mandalay'\n  Update: reference is now set to 'Kotor'\n  Update: location is now set to 'Plovdiv'\n\nWhat is the FINAL value of each record?\nANSWER:\n- liaison: [final value]\n- reference: [final value]\n- location: [final value]\n\nAlso answer these verification questions (Yes or No):\nV1. Was 'Cusco' ever assigned to liaison? [Yes/No]",
  "gold_json": "{\"final_values\": {\"liaison\": \"Mandalay\", \"reference\": \"Kotor\", \"location\": \"Plovdiv\"}, \"key_names\": [\"liaison\", \"reference\", \"location\"]}"
 },
 {
  "task_id": "interference_hard_022",
  "task_type": "interference",
  "difficulty": "Hard",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: delegate is now set to 'Valetta'\n  Update: dispatch is now set to 'Mandalay'\n  Update: assignment is now set to 'Reykjavik'\n  Update: delegate is now set to 'Cartagena'\n  Update: dispatch is now set to 'Plovdiv'\n  Update: assignment is now set to 'Fez'\n  Update: delegate is now set to 'Tbilisi'\n  Update: dispatch is now set to 'Cusco'\n  Update: assignment is now set to 'Gdansk'\n  Update: delegate is now set to 'Luang Prabang'\n  Update: dispatch is now set to 'Recife'\n  Update: assignment is now set to 'Bruges'\n  Update: delegate is now set to 'Valetta'\n  Update: dispatch is now set to 'Kumasi'\n  Update: assignment is now set to 'Cartagena'\n  Update: delegate is now set to 'Luang Prabang'\n  Update: dispatch is now set to 'Recife'\n  Update: assignment is now set to 'Cusco'\n  Update: delegate is now set to 'Tbilisi'\n  Update: dispatch is now set to 'Cartagena'\n  Update: assignment is now set to 'Valetta'\n  Update: delegate is now set to 'Tallinn'\n  Update: dispatch is now set to 'Trieste'\n  Update: assignment is now set to 'Gdansk'\n  Update: delegate is now set to 'Luang Prabang'\n  Update: dispatch is now set to 'Ulaanbaatar'\n  Update: assignment is now set to 'Oulu'\n  Update: delegate is now set to 'Trieste'\n  Update: dispatch is now set to 'Recife'\n  Update: assignment is now set to 'Tbilisi'\n  Update: delegate is now set to 'Recife'\n  Update: dispatch is now set to 'Oulu'\n  Update: assignment is now set to 'Plovdiv'\n  Update: delegate is now set to 'Plovdiv'\n  Update: dispatch is now set to 'Mandalay'\n  Update: assignment is now set to 'Bruges'\n\nWhat is the FINAL value of each record?\nANSWER:\n- delegate: [final value]\n- dispatch: [final value]\n- assignment: [final value]\n\nAlso answer these verification questions (Yes or No):\nV1. Was 'Tbilisi' ever assigned to delegate? [Yes/No]",
  "gold_json": "{\"final_values\": {\"delegate\": \"Plovdiv\", \"dispatch\": \"Mandalay\", \"assignment\": \"Bruges\"}, \"key_names\": [\"delegate\", \"dispatch\", \"assignment\"]}"
 },
 {
  "task_id": "interference_hard_023",
  "task_type": "interference",
  "difficulty": "Hard",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: delegate is now set to 'Sigrid'\n  Update: dispatch is now set to 'Runa'\n  Update: contact is now set to 'Elio'\n  Update: delegate is now set to 'Uma'\n  Update: dispatch is now set to 'Greta'\n  Update: contact is now set to 'Yara'\n  Update: delegate is now set to 'Runa'\n  Update: dispatch is now set to 'Maren'\n  Update: contact is now set to 'Yuki'\n  Update: delegate is now set to 'Soren'\n  Update: dispatch is now set to 'Hana'\n  Update: contact is now set to 'Nalini'\n  Update: delegate is now set to 'Paloma'\n  Update: dispatch is now set to 'Qadir'\n  Update: contact is now set to 'Olena'\n  Update: delegate is now set to 'Elara'\n  Update: dispatch is now set to 'Olena'\n  Update: contact is now set to 'Viktor'\n  Update: delegate is now set to 'Bashir'\n  Update: dispatch is now set to 'Paloma'\n  Update: contact is now set to 'Orla'\n  Update: delegate is now set to 'Dariush'\n  Update: dispatch is now set to 'Sigrid'\n  Update: contact is now set to 'Femi'\n  Update: delegate is now set to 'Tariq'\n  Update: dispatch is now set to 'Priya'\n  Update: contact is now set to 'Joaquin'\n  Update: delegate is now set to 'Colette'\n  Update: dispatch is now set to 'Sigrid'\n  Update: contact is now set to 'Celine'\n  Update: delegate is now set to 'Maren'\n  Update: dispatch is now set to 'Olena'\n  Update: contact is now set to 'Gael'\n  Update: delegate is now set to 'Tala'\n  Update: dispatch is now set to 'Amara'\n  Update: contact is now set to 'Amara'\n\nWhat is the FINAL value of each record?\nANSWER:\n- delegate: [final value]\n- dispatch: [final value]\n- contact: [final value]\n\nAlso answer these verification questions (Yes or No):\nV1. Was 'Paloma' ever assigned to delegate? [Yes/No]",
  "gold_json": "{\"final_values\": {\"delegate\": \"Tala\", \"dispatch\": \"Amara\", \"contact\": \"Amara\"}, \"key_names\": [\"delegate\", \"dispatch\", \"contact\"]}"
 },
 {
  "task_id": "interference_expert_024",
  "task_type": "interference",
  "difficulty": "Expert",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: assignment is now set to '901'\n  Update: liaison is now set to '728'\n  Update: location is now set to '422'\n  Update: registry is now set to '440'\n  Update: assignment is now set to '961'\n  Update: liaison is now set to '927'\n  Update: location is now set to '758'\n  Update: registry is now set to '518'\n  Update: assignment is now set to '885'\n  Update: liaison is now set to '787'\n  Update: location is now set to '566'\n  Update: registry is now set to '111'\n  Update: assignment is now set to '931'\n  Update: liaison is now set to '832'\n  Update: location is now set to '863'\n  Update: registry is now set to '850'\n  Update: assignment is now set to '775'\n  Update: liaison is now set to '178'\n  Update: location is now set to '576'\n  Update: registry is now set to '264'\n  Update: assignment is now set to '621'\n  Update: liaison is now set to '872'\n  Update: location is now set to '662'\n  Update: registry is now set to '282'\n  Update: assignment is now set to '423'\n  Update: liaison is now set to '924'\n  Update: location is now set to '427'\n  Update: registry is now set to '419'\n  Update: assignment is now set to '138'\n  Update: liaison is now set to '560'\n  Update: location is now set to '393'\n  Update: registry is now set to '252'\n  Update: assignment is now set to '352'\n  Update: liaison is now set to '774'\n  Update: location is now set to '130'\n  Update: registry is now set to '855'\n  Update: assignment is now set to '741'\n  Update: liaison is now set to '470'\n  Update: location is now set to '505'\n  Update: registry is now set to '761'\n  Update: assignment is now set to '631'\n  Update: liaison is now set to '523'\n  Update: location is now set to '647'\n  Update: registry is now set to '594'\n  Update: assignment is now set to '518'\n  Update: liaison is now set to '875'\n  Update: location is now set to '169'\n  Update: registry is now set to '907'\n  Update: assignment is now set to '491'\n  Update: liaison is now set to '598'\n  Update: location is now set to '189'\n  Update: registry is now set to '508'\n  Update: assignment is now set to '450'\n  Update: liaison is now set to '121'\n  Update: location is now set to '708'\n  Update: registry is now set to '678'\n  Update: assignment is now set to '877'\n  Update: liaison is now set to '823'\n  Update: location is now set to '192'\n  Update: registry is now set to '831'\n  Update: assignment is now set to '833'\n  Update: liaison is now set to '533'\n  Update: location is now set to '330'\n  Update: registry is now set to '344'\n  Update: assignment is now set to '335'\n  Update: liaison is now set to '279'\n  Update: location is now set to '552'\n  Update: registry is now set to '614'\n  Update: assignment is now set to '268'\n  Update: liaison is now set to '209'\n  Update: location is now set to '627'\n  Update: registry is now set to '522'\n  Update: assignment is now set to '559'\n  Update: liaison is now set to '399'\n  Update: location is now set to '355'\n  Update: registry is now set to '742'\n  Update: assignment is now set to '608'\n  Update: liaison is now set to '291'\n  Update: location is now set to '633'\n  Update: registry is now set to '697'\n  Update: assignment is now set to '884'\n  Update: liaison is now set to '765'\n  Update: location is now set to '482'\n  Update: registry is now set to '871'\n  Update: assignment is now set to '448'\n  Update: liaison is now set to '812'\n  Update: location is now set to '496'\n  Update: registry is now set to '781'\n  Update: assignment is now set to '791'\n  Update: liaison is now set to '713'\n  Update: location is now set to '274'\n  Update: registry is now set to '767'\n  Update: assignment is now set to '894'\n  Update: liaison is now set to '132'\n  Update: location is now set to '491'\n  Update: registry is now set to '188'\n  Update: assignment is now set to '209'\n  Update: liaison is now set to '765'\n  Update: location is now set to '267'\n  Update: registry is now set to '951'\n\nWhat is the FINAL value of each record?\nANSWER:\n- assignment: [final value]\n- liaison: [final value]\n- location: [final value]\n- registry: [final value]\n\nAlso answer these verification questions (Yes or No):\nV1. Was '518' ever assigned to assignment? [Yes/No]\nV2. Was '872' ever assigned to liaison? [Yes/No]",
  "gold_json": "{\"final_values\": {\"assignment\": \"209\", \"liaison\": \"765\", \"location\": \"267\", \"registry\": \"951\"}, \"key_names\": [\"assignment\", \"liaison\", \"location\", \"registry\"]}"
 },
 {
  "task_id": "interference_expert_025",
  "task_type": "interference",
  "difficulty": "Expert",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: registry is now set to 'Olena'\n  Update: liaison is now set to 'Runa'\n  Update: dispatch is now set to 'Idris'\n  Update: assignment is now set to 'Lumi'\n  Update: registry is now set to 'Freya'\n  Update: liaison is now set to 'Olena'\n  Update: dispatch is now set to 'Nico'\n  Update: assignment is now set to 'Vesna'\n  Update: registry is now set to 'Zora'\n  Update: liaison is now set to 'Elara'\n  Update: dispatch is now set to 'Femi'\n  Update: assignment is now set to 'Ravi'\n  Update: registry is now set to 'Haruto'\n  Update: liaison is now set to 'Ravi'\n  Update: dispatch is now set to 'Adaeze'\n  Update: assignment is now set to 'Adaeze'\n  Update: registry is now set to 'Zain'\n  Update: liaison is now set to 'Adaeze'\n  Update: dispatch is now set to 'Wren'\n  Update: assignment is now set to 'Zora'\n  Update: registry is now set to 'Maren'\n  Update: liaison is now set to 'Joaquin'\n  Update: dispatch is now set to 'Celine'\n  Update: assignment is now set to 'Joelle'\n  Update: registry is now set to 'Gael'\n  Update: liaison is now set to 'Colette'\n  Update: dispatch is now set to 'Hana'\n  Update: assignment is now set to 'Qadir'\n  Update: registry is now set to 'Yuki'\n  Update: liaison is now set to 'Leif'\n  Update: dispatch is now set to 'Yuki'\n  Update: assignment is now set to 'Wren'\n  Update: registry is now set to 'Joaquin'\n  Update: liaison is now set to 'Elio'\n  Update: dispatch is now set to 'Yara'\n  Update: assignment is now set to 'Colette'\n  Update: registry is now set to 'Joelle'\n  Update: liaison is now set to 'Zain'\n  Update: dispatch is now set to 'Bram'\n  Update: assignment is now set to 'Femi'\n  Update: registry is now set to 'Joaquin'\n  Update: liaison is now set to 'Dariush'\n  Update: dispatch is now set to 'Leif'\n  Update: assignment is now set to 'Yuki'\n  Update: registry is now set to 'Magnus'\n  Update: liaison is now set to 'Viktor'\n  Update: dispatch is now set to 'Greta'\n  Update: assignment is now set to 'Hana'\n  Update: registry is now set to 'Celine'\n  Update: liaison is now set to 'Bashir'\n  Update: dispatch is now set to 'Magnus'\n  Update: assignment is now set to 'Ravi'\n  Update: registry is now set to 'Idris'\n  Update: liaison is now set to 'Femi'\n  Update: dispatch is now set to 'Runa'\n  Update: assignment is now set to 'Uma'\n  Update: registry is now set to 'Qadir'\n  Update: liaison is now set to 'Xander'\n  Update: dispatch is now set to 'Zora'\n  Update: assignment is now set to 'Elio'\n  Update: registry is now set to 'Leif'\n  Update: liaison is now set to 'Zain'\n  Update: dispatch is now set to 'Femi'\n  Update: assignment is now set to 'Tariq'\n  Update: registry is now set to 'Adaeze'\n  Update: liaison is now set to 'Gael'\n  Update: dispatch is now set to 'Nalini'\n  Update: assignment is now set to 'Uma'\n  Update: registry is now set to 'Hana'\n  Update: liaison is now set to 'Lumi'\n  Update: dispatch is now set to 'Haruto'\n  Update: assignment is now set to 'Kenji'\n  Update: registry is now set to 'Xander'\n  Update: liaison is now set to 'Tariq'\n  Update: dispatch is now set to 'Kenji'\n  Update: assignment is now set to 'Joaquin'\n  Update: registry is now set to 'Ravi'\n  Update: liaison is now set to 'Xander'\n  Update: dispatch is now set to 'Priya'\n  Update: assignment is now set to 'Kaia'\n  Update: registry is now set to 'Tala'\n  Update: liaison is now set to 'Elio'\n  Update: dispatch is now set to 'Bashir'\n  Update: assignment is now set to 'Zain'\n  Update: registry is now set to 'Gael'\n  Update: liaison is now set to 'Tala'\n  Update: dispatch is now set to 'Tala'\n  Update: assignment is now set to 'Tala'\n  Update: registry is now set to 'Freya'\n  Update: liaison is now set to 'Paloma'\n  Update: dispatch is now set to 'Yuki'\n  Update: assignment is now set to 'Joelle'\n  Update: registry is now set to 'Haruto'\n  Update: liaison is now set to 'Freya'\n  Update: dispatch is now set to 'Dariush'\n  Update: assignment is now set to 'Celine'\n  Update: registry is now set to 'Ines'\n  Update: liaison is now set to 'Viktor'\n  Update: dispatch is now set to 'Dmitri'\n  Update: assignment is now set to 'Dmitri'\n\nWhat is the FINAL value of each record?\nANSWER:\n- registry: [final value]\n- liaison: [final value]\n- dispatch: [final value]\n- assignment: [final value]\n\nAlso answer these verification questions (Yes or No):\nV1. Was 'Freya' ever assigned to registry? [Yes/No]\nV2. Was 'Leif' ever assigned to liaison? [Yes/No]",
  "gold_json": "{\"final_values\": {\"registry\": \"Ines\", \"liaison\": \"Viktor\", \"dispatch\": \"Dmitri\", \"assignment\": \"Dmitri\"}, \"key_names\": [\"registry\", \"liaison\", \"dispatch\", \"assignment\"]}"
 },
 {
  "task_id": "interference_expert_026",
  "task_type": "interference",
  "difficulty": "Expert",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: reference is now set to 'Soren'\n  Update: registry is now set to 'Olena'\n  Update: contact is now set to 'Bashir'\n  Update: location is now set to 'Hana'\n  Update: reference is now set to 'Amara'\n  Update: registry is now set to 'Yara'\n  Update: contact is now set to 'Tariq'\n  Update: location is now set to 'Priya'\n  Update: reference is now set to 'Yara'\n  Update: registry is now set to 'Zain'\n  Update: contact is now set to 'Ines'\n  Update: location is now set to 'Sigrid'\n  Update: reference is now set to 'Wren'\n  Update: registry is now set to 'Orla'\n  Update: contact is now set to 'Hana'\n  Update: location is now set to 'Olena'\n  Update: reference is now set to 'Greta'\n  Update: registry is now set to 'Greta'\n  Update: contact is now set to 'Elara'\n  Update: location is now set to 'Elara'\n  Update: reference is now set to 'Kaia'\n  Update: registry is now set to 'Colette'\n  Update: contact is now set to 'Freya'\n  Update: location is now set to 'Colette'\n  Update: reference is now set to 'Colette'\n  Update: registry is now set to 'Kaia'\n  Update: contact is now set to 'Colette'\n  Update: location is now set to 'Amara'\n  Update: reference is now set to 'Vesna'\n  Update: registry is now set to 'Magnus'\n  Update: contact is now set to 'Vesna'\n  Update: location is now set to 'Elio'\n  Update: reference is now set to 'Femi'\n  Update: registry is now set to 'Gael'\n  Update: contact is now set to 'Runa'\n  Update: location is now set to 'Lumi'\n  Update: reference is now set to 'Nalini'\n  Update: registry is now set to 'Dmitri'\n  Update: contact is now set to 'Bram'\n  Update: location is now set to 'Paloma'\n  Update: reference is now set to 'Joaquin'\n  Update: registry is now set to 'Vesna'\n  Update: contact is now set to 'Wren'\n  Update: location is now set to 'Olena'\n  Update: reference is now set to 'Uma'\n  Update: registry is now set to 'Gael'\n  Update: contact is now set to 'Runa'\n  Update: location is now set to 'Elio'\n  Update: reference is now set to 'Idris'\n  Update: registry is now set to 'Soren'\n  Update: contact is now set to 'Gael'\n  Update: location is now set to 'Lumi'\n  Update: reference is now set to 'Dariush'\n  Update: registry is now set to 'Vesna'\n  Update: contact is now set to 'Qadir'\n  Update: location is now set to 'Dariush'\n  Update: reference is now set to 'Amara'\n  Update: registry is now set to 'Lumi'\n  Update: contact is now set to 'Joelle'\n  Update: location is now set to 'Bashir'\n  Update: reference is now set to 'Dariush'\n  Update: registry is now set to 'Viktor'\n  Update: contact is now set to 'Uma'\n  Update: location is now set to 'Soren'\n  Update: reference is now set to 'Bram'\n  Update: registry is now set to 'Zain'\n  Update: contact is now set to 'Runa'\n  Update: location is now set to 'Lumi'\n  Update: reference is now set to 'Zain'\n  Update: registry is now set to 'Orla'\n  Update: contact is now set to 'Adaeze'\n  Update: location is now set to 'Ines'\n  Update: reference is now set to 'Nico'\n  Update: registry is now set to 'Paloma'\n  Update: contact is now set to 'Paloma'\n  Update: location is now set to 'Wren'\n  Update: reference is now set to 'Adaeze'\n  Update: registry is now set to 'Bashir'\n  Update: contact is now set to 'Nalini'\n  Update: location is now set to 'Olena'\n  Update: reference is now set to 'Ugo'\n  Update: registry is now set to 'Colette'\n  Update: contact is now set to 'Amara'\n  Update: location is now set to 'Kaia'\n  Update: reference is now set to 'Leif'\n  Update: registry is now set to 'Hana'\n  Update: contact is now set to 'Zora'\n  Update: location is now set to 'Lumi'\n  Update: reference is now set to 'Xander'\n  Update: registry is now set to 'Nalini'\n  Update: contact is now set to 'Hana'\n  Update: location is now set to 'Kenji'\n  Update: reference is now set to 'Lumi'\n  Update: registry is now set to 'Hana'\n  Update: contact is now set to 'Kaia'\n  Update: location is now set to 'Dmitri'\n  Update: reference is now set to 'Joaquin'\n  Update: registry is now set to 'Vesna'\n  Update: contact is now set to 'Tala'\n  Update: location is now set to 'Willa'\n\nWhat is the FINAL value of each record?\nANSWER:\n- reference: [final value]\n- registry: [final value]\n- contact: [final value]\n- location: [final value]\n\nAlso answer these verification questions (Yes or No):\nV1. Was 'Adaeze' ever assigned to reference? [Yes/No]\nV2. Was 'Colette' ever assigned to registry? [Yes/No]",
  "gold_json": "{\"final_values\": {\"reference\": \"Joaquin\", \"registry\": \"Vesna\", \"contact\": \"Tala\", \"location\": \"Willa\"}, \"key_names\": [\"reference\", \"registry\", \"contact\", \"location\"]}"
 },
 {
  "task_id": "interference_expert_027",
  "task_type": "interference",
  "difficulty": "Expert",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: delegate is now set to '436'\n  Update: destination is now set to '126'\n  Update: location is now set to '405'\n  Update: dispatch is now set to '960'\n  Update: delegate is now set to '663'\n  Update: destination is now set to '552'\n  Update: location is now set to '199'\n  Update: dispatch is now set to '923'\n  Update: delegate is now set to '731'\n  Update: destination is now set to '212'\n  Update: location is now set to '882'\n  Update: dispatch is now set to '819'\n  Update: delegate is now set to '895'\n  Update: destination is now set to '492'\n  Update: location is now set to '239'\n  Update: dispatch is now set to '918'\n  Update: delegate is now set to '554'\n  Update: destination is now set to '129'\n  Update: location is now set to '866'\n  Update: dispatch is now set to '106'\n  Update: delegate is now set to '466'\n  Update: destination is now set to '112'\n  Update: location is now set to '556'\n  Update: dispatch is now set to '341'\n  Update: delegate is now set to '705'\n  Update: destination is now set to '909'\n  Update: location is now set to '389'\n  Update: dispatch is now set to '759'\n  Update: delegate is now set to '221'\n  Update: destination is now set to '304'\n  Update: location is now set to '173'\n  Update: dispatch is now set to '324'\n  Update: delegate is now set to '469'\n  Update: destination is now set to '278'\n  Update: location is now set to '167'\n  Update: dispatch is now set to '147'\n  Update: delegate is now set to '771'\n  Update: destination is now set to '702'\n  Update: location is now set to '481'\n  Update: dispatch is now set to '932'\n  Update: delegate is now set to '100'\n  Update: destination is now set to '377'\n  Update: location is now set to '139'\n  Update: dispatch is now set to '103'\n  Update: delegate is now set to '788'\n  Update: destination is now set to '486'\n  Update: location is now set to '418'\n  Update: dispatch is now set to '166'\n  Update: delegate is now set to '213'\n  Update: destination is now set to '514'\n  Update: location is now set to '995'\n  Update: dispatch is now set to '547'\n  Update: delegate is now set to '704'\n  Update: destination is now set to '589'\n  Update: location is now set to '762'\n  Update: dispatch is now set to '939'\n  Update: delegate is now set to '470'\n  Update: destination is now set to '146'\n  Update: location is now set to '588'\n  Update: dispatch is now set to '343'\n  Update: delegate is now set to '848'\n  Update: destination is now set to '980'\n  Update: location is now set to '640'\n  Update: dispatch is now set to '408'\n  Update: delegate is now set to '995'\n  Update: destination is now set to '957'\n  Update: location is now set to '276'\n  Update: dispatch is now set to '793'\n  Update: delegate is now set to '180'\n  Update: destination is now set to '674'\n  Update: location is now set to '699'\n  Update: dispatch is now set to '345'\n  Update: delegate is now set to '837'\n  Update: destination is now set to '643'\n  Update: location is now set to '562'\n  Update: dispatch is now set to '181'\n  Update: delegate is now set to '207'\n  Update: destination is now set to '110'\n  Update: location is now set to '514'\n  Update: dispatch is now set to '880'\n  Update: delegate is now set to '954'\n  Update: destination is now set to '586'\n  Update: location is now set to '894'\n  Update: dispatch is now set to '949'\n  Update: delegate is now set to '930'\n  Update: destination is now set to '961'\n  Update: location is now set to '669'\n  Update: dispatch is now set to '641'\n  Update: delegate is now set to '356'\n  Update: destination is now set to '438'\n  Update: location is now set to '195'\n  Update: dispatch is now set to '103'\n  Update: delegate is now set to '157'\n  Update: destination is now set to '708'\n  Update: location is now set to '520'\n  Update: dispatch is now set to '384'\n  Update: delegate is now set to '900'\n  Update: destination is now set to '751'\n  Update: location is now set to '344'\n  Update: dispatch is now set to '260'\n\nWhat is the FINAL value of each record?\nANSWER:\n- delegate: [final value]\n- destination: [final value]\n- location: [final value]\n- dispatch: [final value]\n\nAlso answer these verification questions (Yes or No):\nV1. Was '848' ever assigned to delegate? [Yes/No]\nV2. Was '304' ever assigned to destination? [Yes/No]",
  "gold_json": "{\"final_values\": {\"delegate\": \"900\", \"destination\": \"751\", \"location\": \"344\", \"dispatch\": \"260\"}, \"key_names\": [\"delegate\", \"destination\", \"location\", \"dispatch\"]}"
 },
 {
  "task_id": "interference_expert_028",
  "task_type": "interference",
  "difficulty": "Expert",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: delegate is now set to '595'\n  Update: assignment is now set to '768'\n  Update: coordinator is now set to '792'\n  Update: reference is now set to '413'\n  Update: delegate is now set to '117'\n  Update: assignment is now set to '241'\n  Update: coordinator is now set to '984'\n  Update: reference is now set to '431'\n  Update: delegate is now set to '138'\n  Update: assignment is now set to '608'\n  Update: coordinator is now set to '287'\n  Update: reference is now set to '709'\n  Update: delegate is now set to '902'\n  Update: assignment is now set to '281'\n  Update: coordinator is now set to '722'\n  Update: reference is now set to '561'\n  Update: delegate is now set to '141'\n  Update: assignment is now set to '434'\n  Update: coordinator is now set to '512'\n  Update: reference is now set to '422'\n  Update: delegate is now set to '502'\n  Update: assignment is now set to '848'\n  Update: coordinator is now set to '470'\n  Update: reference is now set to '217'\n  Update: delegate is now set to '961'\n  Update: assignment is now set to '463'\n  Update: coordinator is now set to '929'\n  Update: reference is now set to '189'\n  Update: delegate is now set to '806'\n  Update: assignment is now set to '151'\n  Update: coordinator is now set to '290'\n  Update: reference is now set to '714'\n  Update: delegate is now set to '392'\n  Update: assignment is now set to '333'\n  Update: coordinator is now set to '720'\n  Update: reference is now set to '115'\n  Update: delegate is now set to '310'\n  Update: assignment is now set to '623'\n  Update: coordinator is now set to '935'\n  Update: reference is now set to '910'\n  Update: delegate is now set to '680'\n  Update: assignment is now set to '954'\n  Update: coordinator is now set to '913'\n  Update: reference is now set to '382'\n  Update: delegate is now set to '643'\n  Update: assignment is now set to '801'\n  Update: coordinator is now set to '487'\n  Update: reference is now set to '366'\n  Update: delegate is now set to '365'\n  Update: assignment is now set to '867'\n  Update: coordinator is now set to '376'\n  Update: reference is now set to '690'\n  Update: delegate is now set to '576'\n  Update: assignment is now set to '688'\n  Update: coordinator is now set to '552'\n  Update: reference is now set to '887'\n  Update: delegate is now set to '202'\n  Update: assignment is now set to '226'\n  Update: coordinator is now set to '127'\n  Update: reference is now set to '921'\n  Update: delegate is now set to '781'\n  Update: assignment is now set to '847'\n  Update: coordinator is now set to '720'\n  Update: reference is now set to '149'\n  Update: delegate is now set to '397'\n  Update: assignment is now set to '750'\n  Update: coordinator is now set to '706'\n  Update: reference is now set to '733'\n  Update: delegate is now set to '400'\n  Update: assignment is now set to '583'\n  Update: coordinator is now set to '733'\n  Update: reference is now set to '189'\n  Update: delegate is now set to '205'\n  Update: assignment is now set to '187'\n  Update: coordinator is now set to '484'\n  Update: reference is now set to '383'\n  Update: delegate is now set to '484'\n  Update: assignment is now set to '335'\n  Update: coordinator is now set to '842'\n  Update: reference is now set to '331'\n  Update: delegate is now set to '639'\n  Update: assignment is now set to '545'\n  Update: coordinator is now set to '844'\n  Update: reference is now set to '256'\n  Update: delegate is now set to '677'\n  Update: assignment is now set to '451'\n  Update: coordinator is now set to '166'\n  Update: reference is now set to '360'\n  Update: delegate is now set to '188'\n  Update: assignment is now set to '488'\n  Update: coordinator is now set to '134'\n  Update: reference is now set to '139'\n  Update: delegate is now set to '215'\n  Update: assignment is now set to '354'\n  Update: coordinator is now set to '699'\n  Update: reference is now set to '184'\n  Update: delegate is now set to '759'\n  Update: assignment is now set to '534'\n  Update: coordinator is now set to '110'\n  Update: reference is now set to '364'\n\nWhat is the FINAL value of each record?\nANSWER:\n- delegate: [final value]\n- assignment: [final value]\n- coordinator: [final value]\n- reference: [final value]\n\nAlso answer these verification questions (Yes or No):\nV1. Was '502' ever assigned to delegate? [Yes/No]\nV2. Was '583' ever assigned to assignment? [Yes/No]",
  "gold_json": "{\"final_values\": {\"delegate\": \"759\", \"assignment\": \"534\", \"coordinator\": \"110\", \"reference\": \"364\"}, \"key_names\": [\"delegate\", \"assignment\", \"coordinator\", \"reference\"]}"
 },
 {
  "task_id": "interference_expert_029",
  "task_type": "interference",
  "difficulty": "Expert",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: destination is now set to 'Jaipur'\n  Update: liaison is now set to 'Luang Prabang'\n  Update: coordinator is now set to 'Cartagena'\n  Update: reference is now set to 'Ulaanbaatar'\n  Update: destination is now set to 'Luang Prabang'\n  Update: liaison is now set to 'Tallinn'\n  Update: coordinator is now set to 'Valetta'\n  Update: reference is now set to 'Tallinn'\n  Update: destination is now set to 'Ulaanbaatar'\n  Update: liaison is now set to 'Cusco'\n  Update: coordinator is now set to 'Kotor'\n  Update: reference is now set to 'Recife'\n  Update: destination is now set to 'Tbilisi'\n  Update: liaison is now set to 'Recife'\n  Update: coordinator is now set to 'Recife'\n  Update: reference is now set to 'Tbilisi'\n  Update: destination is now set to 'Oulu'\n  Update: liaison is now set to 'Gdansk'\n  Update: coordinator is now set to 'Tallinn'\n  Update: reference is now set to 'Reykjavik'\n  Update: destination is now set to 'Tbilisi'\n  Update: liaison is now set to 'Kotor'\n  Update: coordinator is now set to 'Valetta'\n  Update: reference is now set to 'Recife'\n  Update: destination is now set to 'Luang Prabang'\n  Update: liaison is now set to 'Fez'\n  Update: coordinator is now set to 'Gdansk'\n  Update: reference is now set to 'Ulaanbaatar'\n  Update: destination is now set to 'Oulu'\n  Update: liaison is now set to 'Oulu'\n  Update: coordinator is now set to 'Cartagena'\n  Update: reference is now set to 'Oulu'\n  Update: destination is now set to 'Kotor'\n  Update: liaison is now set to 'Tbilisi'\n  Update: coordinator is now set to 'Oulu'\n  Update: reference is now set to 'Tbilisi'\n  Update: destination is now set to 'Valetta'\n  Update: liaison is now set to 'Kumasi'\n  Update: coordinator is now set to 'Plovdiv'\n  Update: reference is now set to 'Recife'\n  Update: destination is now set to 'Cartagena'\n  Update: liaison is now set to 'Kotor'\n  Update: coordinator is now set to 'Gdansk'\n  Update: reference is now set to 'Tallinn'\n  Update: destination is now set to 'Tallinn'\n  Update: liaison is now set to 'Luang Prabang'\n  Update: coordinator is now set to 'Kotor'\n  Update: reference is now set to 'Mandalay'\n  Update: destination is now set to 'Reykjavik'\n  Update: liaison is now set to 'Valetta'\n  Update: coordinator is now set to 'Valetta'\n  Update: reference is now set to 'Jaipur'\n  Update: destination is now set to 'Luang Prabang'\n  Update: liaison is now set to 'Kumasi'\n  Update: coordinator is now set to 'Trieste'\n  Update: reference is now set to 'Kotor'\n  Update: destination is now set to 'Mandalay'\n  Update: liaison is now set to 'Valetta'\n  Update: coordinator is now set to 'Mandalay'\n  Update: reference is now set to 'Kumasi'\n  Update: destination is now set to 'Tbilisi'\n  Update: liaison is now set to 'Recife'\n  Update: coordinator is now set to 'Cartagena'\n  Update: reference is now set to 'Jaipur'\n  Update: destination is now set to 'Zanzibar'\n  Update: liaison is now set to 'Fez'\n  Update: coordinator is now set to 'Trieste'\n  Update: reference is now set to 'Recife'\n  Update: destination is now set to 'Cartagena'\n  Update: liaison is now set to 'Luang Prabang'\n  Update: coordinator is now set to 'Valetta'\n  Update: reference is now set to 'Cusco'\n  Update: destination is now set to 'Plovdiv'\n  Update: liaison is now set to 'Ulaanbaatar'\n  Update: coordinator is now set to 'Luang Prabang'\n  Update: reference is now set to 'Trieste'\n  Update: destination is now set to 'Trieste'\n  Update: liaison is now set to 'Trieste'\n  Update: coordinator is now set to 'Mandalay'\n  Update: reference is now set to 'Mandalay'\n  Update: destination is now set to 'Bruges'\n  Update: liaison is now set to 'Recife'\n  Update: coordinator is now set to 'Cartagena'\n  Update: reference is now set to 'Plovdiv'\n  Update: destination is now set to 'Mandalay'\n  Update: liaison is now set to 'Plovdiv'\n  Update: coordinator is now set to 'Plovdiv'\n  Update: reference is now set to 'Kotor'\n  Update: destination is now set to 'Cusco'\n  Update: liaison is now set to 'Recife'\n  Update: coordinator is now set to 'Recife'\n  Update: reference is now set to 'Recife'\n  Update: destination is now set to 'Mandalay'\n  Update: liaison is now set to 'Luang Prabang'\n  Update: coordinator is now set to 'Cusco'\n  Update: reference is now set to 'Gdansk'\n  Update: destination is now set to 'Valetta'\n  Update: liaison is now set to 'Ulaanbaatar'\n  Update: coordinator is now set to 'Mandalay'\n  Update: reference is now set to 'Trieste'\n\nWhat is the FINAL value of each record?\nANSWER:\n- destination: [final value]\n- liaison: [final value]\n- coordinator: [final value]\n- reference: [final value]\n\nAlso answer these verification questions (Yes or No):\nV1. Was 'Luang Prabang' ever assigned to destination? [Yes/No]\nV2. Was 'Tbilisi' ever assigned to liaison? [Yes/No]",
  "gold_json": "{\"final_values\": {\"destination\": \"Valetta\", \"liaison\": \"Ulaanbaatar\", \"coordinator\": \"Mandalay\", \"reference\": \"Trieste\"}, \"key_names\": [\"destination\", \"liaison\", \"coordinator\", \"reference\"]}"
 },
 {
  "task_id": "interference_expert_030",
  "task_type": "interference",
  "difficulty": "Expert",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: destination is now set to '602'\n  Update: dispatch is now set to '528'\n  Update: liaison is now set to '805'\n  Update: contact is now set to '477'\n  Update: destination is now set to '839'\n  Update: dispatch is now set to '257'\n  Update: liaison is now set to '685'\n  Update: contact is now set to '338'\n  Update: destination is now set to '638'\n  Update: dispatch is now set to '956'\n  Update: liaison is now set to '374'\n  Update: contact is now set to '655'\n  Update: destination is now set to '855'\n  Update: dispatch is now set to '463'\n  Update: liaison is now set to '857'\n  Update: contact is now set to '698'\n  Update: destination is now set to '959'\n  Update: dispatch is now set to '533'\n  Update: liaison is now set to '454'\n  Update: contact is now set to '489'\n  Update: destination is now set to '423'\n  Update: dispatch is now set to '682'\n  Update: liaison is now set to '371'\n  Update: contact is now set to '720'\n  Update: destination is now set to '585'\n  Update: dispatch is now set to '524'\n  Update: liaison is now set to '485'\n  Update: contact is now set to '818'\n  Update: destination is now set to '954'\n  Update: dispatch is now set to '691'\n  Update: liaison is now set to '264'\n  Update: contact is now set to '484'\n  Update: destination is now set to '387'\n  Update: dispatch is now set to '791'\n  Update: liaison is now set to '275'\n  Update: contact is now set to '962'\n  Update: destination is now set to '283'\n  Update: dispatch is now set to '887'\n  Update: liaison is now set to '126'\n  Update: contact is now set to '908'\n  Update: destination is now set to '713'\n  Update: dispatch is now set to '125'\n  Update: liaison is now set to '970'\n  Update: contact is now set to '906'\n  Update: destination is now set to '721'\n  Update: dispatch is now set to '234'\n  Update: liaison is now set to '448'\n  Update: contact is now set to '387'\n  Update: destination is now set to '801'\n  Update: dispatch is now set to '712'\n  Update: liaison is now set to '138'\n  Update: contact is now set to '296'\n  Update: destination is now set to '750'\n  Update: dispatch is now set to '198'\n  Update: liaison is now set to '546'\n  Update: contact is now set to '618'\n  Update: destination is now set to '499'\n  Update: dispatch is now set to '185'\n  Update: liaison is now set to '877'\n  Update: contact is now set to '956'\n  Update: destination is now set to '219'\n  Update: dispatch is now set to '393'\n  Update: liaison is now set to '106'\n  Update: contact is now set to '595'\n  Update: destination is now set to '140'\n  Update: dispatch is now set to '928'\n  Update: liaison is now set to '218'\n  Update: contact is now set to '736'\n  Update: destination is now set to '742'\n  Update: dispatch is now set to '112'\n  Update: liaison is now set to '269'\n  Update: contact is now set to '819'\n  Update: destination is now set to '200'\n  Update: dispatch is now set to '558'\n  Update: liaison is now set to '704'\n  Update: contact is now set to '825'\n  Update: destination is now set to '439'\n  Update: dispatch is now set to '127'\n  Update: liaison is now set to '454'\n  Update: contact is now set to '786'\n  Update: destination is now set to '614'\n  Update: dispatch is now set to '302'\n  Update: liaison is now set to '278'\n  Update: contact is now set to '154'\n  Update: destination is now set to '569'\n  Update: dispatch is now set to '844'\n  Update: liaison is now set to '118'\n  Update: contact is now set to '916'\n  Update: destination is now set to '839'\n  Update: dispatch is now set to '886'\n  Update: liaison is now set to '393'\n  Update: contact is now set to '823'\n  Update: destination is now set to '842'\n  Update: dispatch is now set to '557'\n  Update: liaison is now set to '235'\n  Update: contact is now set to '828'\n  Update: destination is now set to '861'\n  Update: dispatch is now set to '358'\n  Update: liaison is now set to '416'\n  Update: contact is now set to '254'\n\nWhat is the FINAL value of each record?\nANSWER:\n- destination: [final value]\n- dispatch: [final value]\n- liaison: [final value]\n- contact: [final value]\n\nAlso answer these verification questions (Yes or No):\nV1. Was '842' ever assigned to destination? [Yes/No]\nV2. Was '185' ever assigned to dispatch? [Yes/No]",
  "gold_json": "{\"final_values\": {\"destination\": \"861\", \"dispatch\": \"358\", \"liaison\": \"416\", \"contact\": \"254\"}, \"key_names\": [\"destination\", \"dispatch\", \"liaison\", \"contact\"]}"
 },
 {
  "task_id": "interference_expert_031",
  "task_type": "interference",
  "difficulty": "Expert",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: reference is now set to '760'\n  Update: liaison is now set to '916'\n  Update: registry is now set to '586'\n  Update: delegate is now set to '172'\n  Update: reference is now set to '542'\n  Update: liaison is now set to '393'\n  Update: registry is now set to '131'\n  Update: delegate is now set to '544'\n  Update: reference is now set to '751'\n  Update: liaison is now set to '973'\n  Update: registry is now set to '990'\n  Update: delegate is now set to '477'\n  Update: reference is now set to '947'\n  Update: liaison is now set to '692'\n  Update: registry is now set to '149'\n  Update: delegate is now set to '378'\n  Update: reference is now set to '799'\n  Update: liaison is now set to '485'\n  Update: registry is now set to '389'\n  Update: delegate is now set to '107'\n  Update: reference is now set to '854'\n  Update: liaison is now set to '137'\n  Update: registry is now set to '802'\n  Update: delegate is now set to '973'\n  Update: reference is now set to '514'\n  Update: liaison is now set to '708'\n  Update: registry is now set to '479'\n  Update: delegate is now set to '212'\n  Update: reference is now set to '453'\n  Update: liaison is now set to '185'\n  Update: registry is now set to '629'\n  Update: delegate is now set to '528'\n  Update: reference is now set to '416'\n  Update: liaison is now set to '766'\n  Update: registry is now set to '640'\n  Update: delegate is now set to '474'\n  Update: reference is now set to '867'\n  Update: liaison is now set to '240'\n  Update: registry is now set to '720'\n  Update: delegate is now set to '444'\n  Update: reference is now set to '171'\n  Update: liaison is now set to '790'\n  Update: registry is now set to '709'\n  Update: delegate is now set to '621'\n  Update: reference is now set to '496'\n  Update: liaison is now set to '144'\n  Update: registry is now set to '502'\n  Update: delegate is now set to '563'\n  Update: reference is now set to '735'\n  Update: liaison is now set to '601'\n  Update: registry is now set to '280'\n  Update: delegate is now set to '633'\n  Update: reference is now set to '446'\n  Update: liaison is now set to '217'\n  Update: registry is now set to '935'\n  Update: delegate is now set to '982'\n  Update: reference is now set to '470'\n  Update: liaison is now set to '221'\n  Update: registry is now set to '340'\n  Update: delegate is now set to '562'\n  Update: reference is now set to '362'\n  Update: liaison is now set to '494'\n  Update: registry is now set to '461'\n  Update: delegate is now set to '402'\n  Update: reference is now set to '767'\n  Update: liaison is now set to '274'\n  Update: registry is now set to '833'\n  Update: delegate is now set to '611'\n  Update: reference is now set to '995'\n  Update: liaison is now set to '127'\n  Update: registry is now set to '772'\n  Update: delegate is now set to '819'\n  Update: reference is now set to '464'\n  Update: liaison is now set to '488'\n  Update: registry is now set to '179'\n  Update: delegate is now set to '448'\n  Update: reference is now set to '354'\n  Update: liaison is now set to '901'\n  Update: registry is now set to '551'\n  Update: delegate is now set to '740'\n  Update: reference is now set to '626'\n  Update: liaison is now set to '457'\n  Update: registry is now set to '534'\n  Update: delegate is now set to '515'\n  Update: reference is now set to '841'\n  Update: liaison is now set to '915'\n  Update: registry is now set to '753'\n  Update: delegate is now set to '171'\n  Update: reference is now set to '184'\n  Update: liaison is now set to '928'\n  Update: registry is now set to '108'\n  Update: delegate is now set to '710'\n  Update: reference is now set to '201'\n  Update: liaison is now set to '286'\n  Update: registry is now set to '389'\n  Update: delegate is now set to '180'\n  Update: reference is now set to '689'\n  Update: liaison is now set to '831'\n  Update: registry is now set to '325'\n  Update: delegate is now set to '444'\n\nWhat is the FINAL value of each record?\nANSWER:\n- reference: [final value]\n- liaison: [final value]\n- registry: [final value]\n- delegate: [final value]\n\nAlso answer these verification questions (Yes or No):\nV1. Was '841' ever assigned to reference? [Yes/No]\nV2. Was '127' ever assigned to liaison? [Yes/No]",
  "gold_json": "{\"final_values\": {\"reference\": \"689\", \"liaison\": \"831\", \"registry\": \"325\", \"delegate\": \"444\"}, \"key_names\": [\"reference\", \"liaison\", \"registry\", \"delegate\"]}"
 },
 {
  "task_id": "interference_frontier_032",
  "task_type": "interference",
  "difficulty": "Frontier",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: registry is now set to 'Tallinn'\n  Update: coordinator is now set to 'Zanzibar'\n  Update: location is now set to 'Tbilisi'\n  Update: reference is now set to 'Kotor'\n  Update: delegate is now set to 'Mandalay'\n  Update: dispatch is now set to 'Gdansk'\n  Update: contact is now set to 'Gdansk'\n  Update: destination is now set to 'Gdansk'\n  Update: registry is now set to 'Tbilisi'\n  Update: coordinator is now set to 'Recife'\n  Update: location is now set to 'Cusco'\n  Update: reference is now set to 'Oulu'\n  Update: delegate is now set to 'Reykjavik'\n  Update: dispatch is now set to 'Tbilisi'\n  Update: contact is now set to 'Jaipur'\n  Update: destination is now set to 'Plovdiv'\n  Update: registry is now set to 'Kumasi'\n  Update: coordinator is now set to 'Ulaanbaatar'\n  Update: location is now set to 'Luang Prabang'\n  Update: reference is now set to 'Bruges'\n  Update: delegate is now set to 'Zanzibar'\n  Update: dispatch is now set to 'Recife'\n  Update: contact is now set to 'Bruges'\n  Update: destination is now set to 'Tallinn'\n  Update: registry is now set to 'Tallinn'\n  Update: coordinator is now set to 'Valetta'\n  Update: location is now set to 'Plovdiv'\n  Update: reference is now set to 'Tbilisi'\n  Update: delegate is now set to 'Kumasi'\n  Update: dispatch is now set to 'Trieste'\n  Update: contact is now set to 'Reykjavik'\n  Update: destination is now set to 'Trieste'\n  Update: registry is now set to 'Mandalay'\n  Update: coordinator is now set to 'Recife'\n  Update: location is now set to 'Kotor'\n  Update: reference is now set to 'Mandalay'\n  Update: delegate is now set to 'Cusco'\n  Update: dispatch is now set to 'Plovdiv'\n  Update: contact is now set to 'Bruges'\n  Update: destination is now set to 'Kotor'\n  Update: registry is now set to 'Tbilisi'\n  Update: coordinator is now set to 'Tallinn'\n  Update: location is now set to 'Oulu'\n  Update: reference is now set to 'Kotor'\n  Update: delegate is now set to 'Reykjavik'\n  Update: dispatch is now set to 'Luang Prabang'\n  Update: contact is now set to 'Recife'\n  Update: destination is now set to 'Cusco'\n  Update: registry is now set to 'Cartagena'\n  Update: coordinator is now set to 'Gdansk'\n  Update: location is now set to 'Luang Prabang'\n  Update: reference is now set to 'Valetta'\n  Update: delegate is now set to 'Oulu'\n  Update: dispatch is now set to 'Kumasi'\n  Update: contact is now set to 'Zanzibar'\n  Update: destination is now set to 'Gdansk'\n  Update: registry is now set to 'Kumasi'\n  Update: coordinator is now set to 'Bruges'\n  Update: location is now set to 'Reykjavik'\n  Update: reference is now set to 'Oulu'\n  Update: delegate is now set to 'Tbilisi'\n  Update: dispatch is now set to 'Luang Prabang'\n  Update: contact is now set to 'Cartagena'\n  Update: destination is now set to 'Kotor'\n  Update: registry is now set to 'Fez'\n  Update: coordinator is now set to 'Zanzibar'\n  Update: location is now set to 'Trieste'\n  Update: reference is now set to 'Mandalay'\n  Update: delegate is now set to 'Oulu'\n  Update: dispatch is now set to 'Cartagena'\n  Update: contact is now set to 'Luang Prabang'\n  Update: destination is now set to 'Zanzibar'\n  Update: registry is now set to 'Oulu'\n  Update: coordinator is now set to 'Kotor'\n  Update: location is now set to 'Tbilisi'\n  Update: reference is now set to 'Recife'\n  Update: delegate is now set to 'Reykjavik'\n  Update: dispatch is now set to 'Zanzibar'\n  Update: contact is now set to 'Zanzibar'\n  Update: destination is now set to 'Kotor'\n  Update: registry is now set to 'Gdansk'\n  Update: coordinator is now set to 'Gdansk'\n  Update: location is now set to 'Gdansk'\n  Update: reference is now set to 'Luang Prabang'\n  Update: delegate is now set to 'Cusco'\n  Update: dispatch is now set to 'Reykjavik'\n  Update: contact is now set to 'Cartagena'\n  Update: destination is now set to 'Valetta'\n  Update: registry is now set to 'Bruges'\n  Update: coordinator is now set to 'Trieste'\n  Update: location is now set to 'Zanzibar'\n  Update: reference is now set to 'Tallinn'\n  Update: delegate is now set to 'Valetta'\n  Update: dispatch is now set to 'Kumasi'\n  Update: contact is now set to 'Mandalay'\n  Update: destination is now set to 'Kumasi'\n  Update: registry is now set to 'Oulu'\n  Update: coordinator is now set to 'Recife'\n  Update: location is now set to 'Oulu'\n  Update: reference is now set to 'Tbilisi'\n  Update: delegate is now set to 'Kotor'\n  Update: dispatch is now set to 'Oulu'\n  Update: contact is now set to 'Jaipur'\n  Update: destination is now set to 'Valetta'\n  Update: registry is now set to 'Gdansk'\n  Update: coordinator is now set to 'Fez'\n  Update: location is now set to 'Ulaanbaatar'\n  Update: reference is now set to 'Gdansk'\n  Update: delegate is now set to 'Luang Prabang'\n  Update: dispatch is now set to 'Bruges'\n  Update: contact is now set to 'Gdansk'\n  Update: destination is now set to 'Tbilisi'\n  Update: registry is now set to 'Bruges'\n  Update: coordinator is now set to 'Mandalay'\n  Update: location is now set to 'Plovdiv'\n  Update: reference is now set to 'Jaipur'\n  Update: delegate is now set to 'Oulu'\n  Update: dispatch is now set to 'Kotor'\n  Update: contact is now set to 'Reykjavik'\n  Update: destination is now set to 'Kumasi'\n  Update: registry is now set to 'Cartagena'\n  Update: coordinator is now set to 'Zanzibar'\n  Update: location is now set to 'Tbilisi'\n  Update: reference is now set to 'Mandalay'\n  Update: delegate is now set to 'Tbilisi'\n  Update: dispatch is now set to 'Ulaanbaatar'\n  Update: contact is now set to 'Jaipur'\n  Update: destination is now set to 'Recife'\n  Update: registry is now set to 'Mandalay'\n  Update: coordinator is now set to 'Trieste'\n  Update: location is now set to 'Mandalay'\n  Update: reference is now set to 'Zanzibar'\n  Update: delegate is now set to 'Plovdiv'\n  Update: dispatch is now set to 'Zanzibar'\n  Update: contact is now set to 'Oulu'\n  Update: destination is now set to 'Reykjavik'\n  Update: registry is now set to 'Ulaanbaatar'\n  Update: coordinator is now set to 'Kumasi'\n  Update: location is now set to 'Plovdiv'\n  Update: reference is now set to 'Oulu'\n  Update: delegate is now set to 'Luang Prabang'\n  Update: dispatch is now set to 'Bruges'\n  Update: contact is now set to 'Cartagena'\n  Update: destination is now set to 'Fez'\n  Update: registry is now set to 'Jaipur'\n  Update: coordinator is now set to 'Fez'\n  Update: location is now set to 'Trieste'\n  Update: reference is now set to 'Cartagena'\n  Update: delegate is now set to 'Fez'\n  Update: dispatch is now set to 'Trieste'\n  Update: contact is now set to 'Reykjavik'\n  Update: destination is now set to 'Valetta'\n  Update: registry is now set to 'Bruges'\n  Update: coordinator is now set to 'Valetta'\n  Update: location is now set to 'Cusco'\n  Update: reference is now set to 'Cusco'\n  Update: delegate is now set to 'Recife'\n  Update: dispatch is now set to 'Zanzibar'\n  Update: contact is now set to 'Zanzibar'\n  Update: destination is now set to 'Cusco'\n  Update: registry is now set to 'Mandalay'\n  Update: coordinator is now set to 'Cusco'\n  Update: location is now set to 'Gdansk'\n  Update: reference is now set to 'Ulaanbaatar'\n  Update: delegate is now set to 'Tbilisi'\n  Update: dispatch is now set to 'Reykjavik'\n  Update: contact is now set to 'Gdansk'\n  Update: destination is now set to 'Cartagena'\n  Update: registry is now set to 'Oulu'\n  Update: coordinator is now set to 'Plovdiv'\n  Update: location is now set to 'Reykjavik'\n  Update: reference is now set to 'Cartagena'\n  Update: delegate is now set to 'Trieste'\n  Update: dispatch is now set to 'Gdansk'\n  Update: contact is now set to 'Tbilisi'\n  Update: destination is now set to 'Oulu'\n  Update: registry is now set to 'Fez'\n  Update: coordinator is now set to 'Oulu'\n  Update: location is now set to 'Fez'\n  Update: reference is now set to 'Fez'\n  Update: delegate is now set to 'Fez'\n  Update: dispatch is now set to 'Trieste'\n  Update: contact is now set to 'Oulu'\n  Update: destination is now set to 'Fez'\n  Update: registry is now set to 'Valetta'\n  Update: coordinator is now set to 'Kumasi'\n  Update: location is now set to 'Jaipur'\n  Update: reference is now set to 'Kumasi'\n  Update: delegate is now set to 'Ulaanbaatar'\n  Update: dispatch is now set to 'Gdansk'\n  Update: contact is now set to 'Tallinn'\n  Update: destination is now set to 'Ulaanbaatar'\n  Update: registry is now set to 'Fez'\n  Update: coordinator is now set to 'Cartagena'\n  Update: location is now set to 'Luang Prabang'\n  Update: reference is now set to 'Tallinn'\n  Update: delegate is now set to 'Jaipur'\n  Update: dispatch is now set to 'Luang Prabang'\n  Update: contact is now set to 'Valetta'\n  Update: destination is now set to 'Plovdiv'\n  Update: registry is now set to 'Mandalay'\n  Update: coordinator is now set to 'Cusco'\n  Update: location is now set to 'Ulaanbaatar'\n  Update: reference is now set to 'Fez'\n  Update: delegate is now set to 'Valetta'\n  Update: dispatch is now set to 'Valetta'\n  Update: contact is now set to 'Recife'\n  Update: destination is now set to 'Cartagena'\n  Update: registry is now set to 'Recife'\n  Update: coordinator is now set to 'Oulu'\n  Update: location is now set to 'Bruges'\n  Update: reference is now set to 'Cusco'\n  Update: delegate is now set to 'Bruges'\n  Update: dispatch is now set to 'Cusco'\n  Update: contact is now set to 'Gdansk'\n  Update: destination is now set to 'Luang Prabang'\n  Update: registry is now set to 'Trieste'\n  Update: coordinator is now set to 'Trieste'\n  Update: location is now set to 'Trieste'\n  Update: reference is now set to 'Plovdiv'\n  Update: delegate is now set to 'Oulu'\n  Update: dispatch is now set to 'Mandalay'\n  Update: contact is now set to 'Zanzibar'\n  Update: destination is now set to 'Oulu'\n  Update: registry is now set to 'Kotor'\n  Update: coordinator is now set to 'Zanzibar'\n  Update: location is now set to 'Cartagena'\n  Update: reference is now set to 'Gdansk'\n  Update: delegate is now set to 'Reykjavik'\n  Update: dispatch is now set to 'Kotor'\n  Update: contact is now set to 'Mandalay'\n  Update: destination is now set to 'Cusco'\n  Update: registry is now set to 'Luang Prabang'\n  Update: coordinator is now set to 'Valetta'\n  Update: location is now set to 'Oulu'\n  Update: reference is now set to 'Valetta'\n  Update: delegate is now set to 'Bruges'\n  Update: dispatch is now set to 'Bruges'\n  Update: contact is now set to 'Trieste'\n  Update: destination is now set to 'Gdansk'\n  Update: registry is now set to 'Zanzibar'\n  Update: coordinator is now set to 'Cusco'\n  Update: location is now set to 'Kotor'\n  Update: reference is now set to 'Plovdiv'\n  Update: delegate is now set to 'Tbilisi'\n  Update: dispatch is now set to 'Kotor'\n  Update: contact is now set to 'Tbilisi'\n  Update: destination is now set to 'Kumasi'\n  Update: registry is now set to 'Recife'\n  Update: coordinator is now set to 'Recife'\n  Update: location is now set to 'Ulaanbaatar'\n  Update: reference is now set to 'Gdansk'\n  Update: delegate is now set to 'Kumasi'\n  Update: dispatch is now set to 'Cartagena'\n  Update: contact is now set to 'Reykjavik'\n  Update: destination is now set to 'Gdansk'\n  Update: registry is now set to 'Kotor'\n  Update: coordinator is now set to 'Bruges'\n  Update: location is now set to 'Bruges'\n  Update: reference is now set to 'Jaipur'\n  Update: delegate is now set to 'Zanzibar'\n  Update: dispatch is now set to 'Fez'\n  Update: contact is now set to 'Kotor'\n  Update: destination is now set to 'Mandalay'\n  Update: registry is now set to 'Recife'\n  Update: coordinator is now set to 'Ulaanbaatar'\n  Update: location is now set to 'Oulu'\n  Update: reference is now set to 'Gdansk'\n  Update: delegate is now set to 'Kumasi'\n  Update: dispatch is now set to 'Mandalay'\n  Update: contact is now set to 'Tallinn'\n  Update: destination is now set to 'Tbilisi'\n  Update: registry is now set to 'Valetta'\n  Update: coordinator is now set to 'Tbilisi'\n  Update: location is now set to 'Ulaanbaatar'\n  Update: reference is now set to 'Reykjavik'\n  Update: delegate is now set to 'Tbilisi'\n  Update: dispatch is now set to 'Cusco'\n  Update: contact is now set to 'Zanzibar'\n  Update: destination is now set to 'Trieste'\n  Update: registry is now set to 'Plovdiv'\n  Update: coordinator is now set to 'Mandalay'\n  Update: location is now set to 'Trieste'\n  Update: reference is now set to 'Recife'\n  Update: delegate is now set to 'Cartagena'\n  Update: dispatch is now set to 'Recife'\n  Update: contact is now set to 'Ulaanbaatar'\n  Update: destination is now set to 'Kumasi'\n  Update: registry is now set to 'Zanzibar'\n  Update: coordinator is now set to 'Trieste'\n  Update: location is now set to 'Fez'\n  Update: reference is now set to 'Trieste'\n  Update: delegate is now set to 'Fez'\n  Update: dispatch is now set to 'Kumasi'\n  Update: contact is now set to 'Tbilisi'\n  Update: destination is now set to 'Gdansk'\n  Update: registry is now set to 'Ulaanbaatar'\n  Update: coordinator is now set to 'Oulu'\n  Update: location is now set to 'Kotor'\n  Update: reference is now set to 'Plovdiv'\n  Update: delegate is now set to 'Kumasi'\n  Update: dispatch is now set to 'Plovdiv'\n  Update: contact is now set to 'Kotor'\n  Update: destination is now set to 'Fez'\n  Update: registry is now set to 'Plovdiv'\n  Update: coordinator is now set to 'Gdansk'\n  Update: location is now set to 'Jaipur'\n  Update: reference is now set to 'Valetta'\n  Update: delegate is now set to 'Cusco'\n  Update: dispatch is now set to 'Luang Prabang'\n  Update: contact is now set to 'Bruges'\n  Update: destination is now set to 'Luang Prabang'\n  Update: registry is now set to 'Recife'\n  Update: coordinator is now set to 'Cartagena'\n  Update: location is now set to 'Zanzibar'\n  Update: reference is now set to 'Fez'\n  Update: delegate is now set to 'Tbilisi'\n  Update: dispatch is now set to 'Mandalay'\n  Update: contact is now set to 'Ulaanbaatar'\n  Update: destination is now set to 'Ulaanbaatar'\n  Update: registry is now set to 'Oulu'\n  Update: coordinator is now set to 'Ulaanbaatar'\n  Update: location is now set to 'Kotor'\n  Update: reference is now set to 'Kotor'\n  Update: delegate is now set to 'Cartagena'\n  Update: dispatch is now set to 'Zanzibar'\n  Update: contact is now set to 'Cusco'\n  Update: destination is now set to 'Trieste'\n  Update: registry is now set to 'Kotor'\n  Update: coordinator is now set to 'Tbilisi'\n  Update: location is now set to 'Gdansk'\n  Update: reference is now set to 'Cusco'\n  Update: delegate is now set to 'Valetta'\n  Update: dispatch is now set to 'Recife'\n  Update: contact is now set to 'Trieste'\n  Update: destination is now set to 'Luang Prabang'\n  Update: registry is now set to 'Luang Prabang'\n  Update: coordinator is now set to 'Oulu'\n  Update: location is now set to 'Mandalay'\n  Update: reference is now set to 'Reykjavik'\n  Update: delegate is now set to 'Jaipur'\n  Update: dispatch is now set to 'Luang Prabang'\n  Update: contact is now set to 'Reykjavik'\n  Update: destination is now set to 'Plovdiv'\n  Update: registry is now set to 'Cartagena'\n  Update: coordinator is now set to 'Cusco'\n  Update: location is now set to 'Gdansk'\n  Update: reference is now set to 'Plovdiv'\n  Update: delegate is now set to 'Gdansk'\n  Update: dispatch is now set to 'Cartagena'\n  Update: contact is now set to 'Valetta'\n  Update: destination is now set to 'Kumasi'\n  Update: registry is now set to 'Ulaanbaatar'\n  Update: coordinator is now set to 'Gdansk'\n  Update: location is now set to 'Cartagena'\n  Update: reference is now set to 'Tallinn'\n  Update: delegate is now set to 'Oulu'\n  Update: dispatch is now set to 'Recife'\n  Update: contact is now set to 'Kumasi'\n  Update: destination is now set to 'Valetta'\n  Update: registry is now set to 'Reykjavik'\n  Update: coordinator is now set to 'Kotor'\n  Update: location is now set to 'Plovdiv'\n  Update: reference is now set to 'Oulu'\n  Update: delegate is now set to 'Tbilisi'\n  Update: dispatch is now set to 'Luang Prabang'\n  Update: contact is now set to 'Jaipur'\n  Update: destination is now set to 'Cartagena'\n  Update: registry is now set to 'Fez'\n  Update: coordinator is now set to 'Zanzibar'\n  Update: location is now set to 'Fez'\n  Update: reference is now set to 'Jaipur'\n  Update: delegate is now set to 'Jaipur'\n  Update: dispatch is now set to 'Kumasi'\n  Update: contact is now set to 'Tbilisi'\n  Update: destination is now set to 'Zanzibar'\n  Update: registry is now set to 'Cartagena'\n  Update: coordinator is now set to 'Trieste'\n  Update: location is now set to 'Gdansk'\n  Update: reference is now set to 'Kumasi'\n  Update: delegate is now set to 'Kotor'\n  Update: dispatch is now set to 'Trieste'\n  Update: contact is now set to 'Ulaanbaatar'\n  Update: destination is now set to 'Bruges'\n  Update: registry is now set to 'Tallinn'\n  Update: coordinator is now set to 'Tallinn'\n  Update: location is now set to 'Mandalay'\n  Update: reference is now set to 'Reykjavik'\n  Update: delegate is now set to 'Luang Prabang'\n  Update: dispatch is now set to 'Oulu'\n  Update: contact is now set to 'Reykjavik'\n  Update: destination is now set to 'Trieste'\n  Update: registry is now set to 'Trieste'\n  Update: coordinator is now set to 'Gdansk'\n  Update: location is now set to 'Tallinn'\n  Update: reference is now set to 'Valetta'\n  Update: delegate is now set to 'Oulu'\n  Update: dispatch is now set to 'Cusco'\n  Update: contact is now set to 'Luang Prabang'\n  Update: destination is now set to 'Reykjavik'\n\nWhat is the FINAL value of each record?\nANSWER:\n- registry: [final value]\n- coordinator: [final value]\n- location: [final value]\n- reference: [final value]\n- delegate: [final value]\n- dispatch: [final value]\n- contact: [final value]\n- destination: [final value]\n\nAlso answer these verification questions (Yes or No):\nV1. Was 'Valetta' ever assigned to registry? [Yes/No]\nV2. Was 'Cusco' ever assigned to coordinator? [Yes/No]\nV3. Was 'Trieste' ever assigned to location? [Yes/No]\nV4. Was 'Tallinn' ever assigned to reference? [Yes/No]",
  "gold_json": "{\"final_values\": {\"registry\": \"Trieste\", \"coordinator\": \"Gdansk\", \"location\": \"Tallinn\", \"reference\": \"Valetta\", \"delegate\": \"Oulu\", \"dispatch\": \"Cusco\", \"contact\": \"Luang Prabang\", \"destination\": \"Reykjavik\"}, \"key_names\": [\"registry\", \"coordinator\", \"location\", \"reference\", \"delegate\", \"dispatch\", \"contact\", \"destination\"]}"
 },
 {
  "task_id": "interference_frontier_033",
  "task_type": "interference",
  "difficulty": "Frontier",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: registry is now set to 'Luang Prabang'\n  Update: dispatch is now set to 'Zanzibar'\n  Update: reference is now set to 'Cartagena'\n  Update: coordinator is now set to 'Trieste'\n  Update: destination is now set to 'Zanzibar'\n  Update: location is now set to 'Valetta'\n  Update: delegate is now set to 'Tbilisi'\n  Update: contact is now set to 'Luang Prabang'\n  Update: registry is now set to 'Mandalay'\n  Update: dispatch is now set to 'Tallinn'\n  Update: reference is now set to 'Valetta'\n  Update: coordinator is now set to 'Tallinn'\n  Update: destination is now set to 'Kotor'\n  Update: location is now set to 'Kumasi'\n  Update: delegate is now set to 'Valetta'\n  Update: contact is now set to 'Trieste'\n  Update: registry is now set to 'Ulaanbaatar'\n  Update: dispatch is now set to 'Trieste'\n  Update: reference is now set to 'Cartagena'\n  Update: coordinator is now set to 'Oulu'\n  Update: destination is now set to 'Zanzibar'\n  Update: location is now set to 'Ulaanbaatar'\n  Update: delegate is now set to 'Cartagena'\n  Update: contact is now set to 'Kotor'\n  Update: registry is now set to 'Kumasi'\n  Update: dispatch is now set to 'Luang Prabang'\n  Update: reference is now set to 'Gdansk'\n  Update: coordinator is now set to 'Kotor'\n  Update: destination is now set to 'Cusco'\n  Update: location is now set to 'Gdansk'\n  Update: delegate is now set to 'Gdansk'\n  Update: contact is now set to 'Cusco'\n  Update: registry is now set to 'Plovdiv'\n  Update: dispatch is now set to 'Jaipur'\n  Update: reference is now set to 'Tallinn'\n  Update: coordinator is now set to 'Fez'\n  Update: destination is now set to 'Ulaanbaatar'\n  Update: location is now set to 'Reykjavik'\n  Update: delegate is now set to 'Trieste'\n  Update: contact is now set to 'Tbilisi'\n  Update: registry is now set to 'Kumasi'\n  Update: dispatch is now set to 'Mandalay'\n  Update: reference is now set to 'Mandalay'\n  Update: coordinator is now set to 'Gdansk'\n  Update: destination is now set to 'Bruges'\n  Update: location is now set to 'Kumasi'\n  Update: delegate is now set to 'Ulaanbaatar'\n  Update: contact is now set to 'Oulu'\n  Update: registry is now set to 'Trieste'\n  Update: dispatch is now set to 'Valetta'\n  Update: reference is now set to 'Kotor'\n  Update: coordinator is now set to 'Cusco'\n  Update: destination is now set to 'Zanzibar'\n  Update: location is now set to 'Cusco'\n  Update: delegate is now set to 'Cartagena'\n  Update: contact is now set to 'Recife'\n  Update: registry is now set to 'Mandalay'\n  Update: dispatch is now set to 'Jaipur'\n  Update: reference is now set to 'Trieste'\n  Update: coordinator is now set to 'Ulaanbaatar'\n  Update: destination is now set to 'Trieste'\n  Update: location is now set to 'Valetta'\n  Update: delegate is now set to 'Plovdiv'\n  Update: contact is now set to 'Mandalay'\n  Update: registry is now set to 'Jaipur'\n  Update: dispatch is now set to 'Valetta'\n  Update: reference is now set to 'Kotor'\n  Update: coordinator is now set to 'Tallinn'\n  Update: destination is now set to 'Recife'\n  Update: location is now set to 'Oulu'\n  Update: delegate is now set to 'Gdansk'\n  Update: contact is now set to 'Tallinn'\n  Update: registry is now set to 'Bruges'\n  Update: dispatch is now set to 'Recife'\n  Update: reference is now set to 'Mandalay'\n  Update: coordinator is now set to 'Luang Prabang'\n  Update: destination is now set to 'Kotor'\n  Update: location is now set to 'Mandalay'\n  Update: delegate is now set to 'Jaipur'\n  Update: contact is now set to 'Luang Prabang'\n  Update: registry is now set to 'Jaipur'\n  Update: dispatch is now set to 'Mandalay'\n  Update: reference is now set to 'Cusco'\n  Update: coordinator is now set to 'Cartagena'\n  Update: destination is now set to 'Jaipur'\n  Update: location is now set to 'Reykjavik'\n  Update: delegate is now set to 'Tbilisi'\n  Update: contact is now set to 'Jaipur'\n  Update: registry is now set to 'Cartagena'\n  Update: dispatch is now set to 'Kumasi'\n  Update: reference is now set to 'Luang Prabang'\n  Update: coordinator is now set to 'Luang Prabang'\n  Update: destination is now set to 'Recife'\n  Update: location is now set to 'Ulaanbaatar'\n  Update: delegate is now set to 'Tallinn'\n  Update: contact is now set to 'Oulu'\n  Update: registry is now set to 'Ulaanbaatar'\n  Update: dispatch is now set to 'Reykjavik'\n  Update: reference is now set to 'Kotor'\n  Update: coordinator is now set to 'Zanzibar'\n  Update: destination is now set to 'Jaipur'\n  Update: location is now set to 'Fez'\n  Update: delegate is now set to 'Trieste'\n  Update: contact is now set to 'Cusco'\n  Update: registry is now set to 'Jaipur'\n  Update: dispatch is now set to 'Recife'\n  Update: reference is now set to 'Zanzibar'\n  Update: coordinator is now set to 'Kumasi'\n  Update: destination is now set to 'Zanzibar'\n  Update: location is now set to 'Valetta'\n  Update: delegate is now set to 'Zanzibar'\n  Update: contact is now set to 'Oulu'\n  Update: registry is now set to 'Recife'\n  Update: dispatch is now set to 'Valetta'\n  Update: reference is now set to 'Reykjavik'\n  Update: coordinator is now set to 'Mandalay'\n  Update: destination is now set to 'Kotor'\n  Update: location is now set to 'Zanzibar'\n  Update: delegate is now set to 'Luang Prabang'\n  Update: contact is now set to 'Recife'\n  Update: registry is now set to 'Zanzibar'\n  Update: dispatch is now set to 'Tbilisi'\n  Update: reference is now set to 'Plovdiv'\n  Update: coordinator is now set to 'Tbilisi'\n  Update: destination is now set to 'Valetta'\n  Update: location is now set to 'Plovdiv'\n  Update: delegate is now set to 'Reykjavik'\n  Update: contact is now set to 'Tallinn'\n  Update: registry is now set to 'Kumasi'\n  Update: dispatch is now set to 'Zanzibar'\n  Update: reference is now set to 'Trieste'\n  Update: coordinator is now set to 'Recife'\n  Update: destination is now set to 'Fez'\n  Update: location is now set to 'Tbilisi'\n  Update: delegate is now set to 'Kumasi'\n  Update: contact is now set to 'Kotor'\n  Update: registry is now set to 'Tallinn'\n  Update: dispatch is now set to 'Recife'\n  Update: reference is now set to 'Kotor'\n  Update: coordinator is now set to 'Cusco'\n  Update: destination is now set to 'Kotor'\n  Update: location is now set to 'Plovdiv'\n  Update: delegate is now set to 'Luang Prabang'\n  Update: contact is now set to 'Valetta'\n  Update: registry is now set to 'Fez'\n  Update: dispatch is now set to 'Luang Prabang'\n  Update: reference is now set to 'Fez'\n  Update: coordinator is now set to 'Ulaanbaatar'\n  Update: destination is now set to 'Tbilisi'\n  Update: location is now set to 'Kotor'\n  Update: delegate is now set to 'Jaipur'\n  Update: contact is now set to 'Trieste'\n  Update: registry is now set to 'Luang Prabang'\n  Update: dispatch is now set to 'Jaipur'\n  Update: reference is now set to 'Gdansk'\n  Update: coordinator is now set to 'Recife'\n  Update: destination is now set to 'Kumasi'\n  Update: location is now set to 'Recife'\n  Update: delegate is now set to 'Cusco'\n  Update: contact is now set to 'Luang Prabang'\n  Update: registry is now set to 'Kumasi'\n  Update: dispatch is now set to 'Tbilisi'\n  Update: reference is now set to 'Recife'\n  Update: coordinator is now set to 'Jaipur'\n  Update: destination is now set to 'Ulaanbaatar'\n  Update: location is now set to 'Cusco'\n  Update: delegate is now set to 'Tbilisi'\n  Update: contact is now set to 'Tbilisi'\n  Update: registry is now set to 'Luang Prabang'\n  Update: dispatch is now set to 'Jaipur'\n  Update: reference is now set to 'Valetta'\n  Update: coordinator is now set to 'Mandalay'\n  Update: destination is now set to 'Reykjavik'\n  Update: location is now set to 'Mandalay'\n  Update: delegate is now set to 'Ulaanbaatar'\n  Update: contact is now set to 'Jaipur'\n  Update: registry is now set to 'Cusco'\n  Update: dispatch is now set to 'Recife'\n  Update: reference is now set to 'Oulu'\n  Update: coordinator is now set to 'Plovdiv'\n  Update: destination is now set to 'Bruges'\n  Update: location is now set to 'Recife'\n  Update: delegate is now set to 'Plovdiv'\n  Update: contact is now set to 'Kumasi'\n  Update: registry is now set to 'Zanzibar'\n  Update: dispatch is now set to 'Valetta'\n  Update: reference is now set to 'Tbilisi'\n  Update: coordinator is now set to 'Kumasi'\n  Update: destination is now set to 'Ulaanbaatar'\n  Update: location is now set to 'Tallinn'\n  Update: delegate is now set to 'Recife'\n  Update: contact is now set to 'Reykjavik'\n  Update: registry is now set to 'Cusco'\n  Update: dispatch is now set to 'Tbilisi'\n  Update: reference is now set to 'Tallinn'\n  Update: coordinator is now set to 'Mandalay'\n  Update: destination is now set to 'Kotor'\n  Update: location is now set to 'Cusco'\n  Update: delegate is now set to 'Tallinn'\n  Update: contact is now set to 'Zanzibar'\n  Update: registry is now set to 'Recife'\n  Update: dispatch is now set to 'Gdansk'\n  Update: reference is now set to 'Kumasi'\n  Update: coordinator is now set to 'Fez'\n  Update: destination is now set to 'Gdansk'\n  Update: location is now set to 'Jaipur'\n  Update: delegate is now set to 'Valetta'\n  Update: contact is now set to 'Cusco'\n  Update: registry is now set to 'Ulaanbaatar'\n  Update: dispatch is now set to 'Valetta'\n  Update: reference is now set to 'Gdansk'\n  Update: coordinator is now set to 'Gdansk'\n  Update: destination is now set to 'Kumasi'\n  Update: location is now set to 'Kotor'\n  Update: delegate is now set to 'Recife'\n  Update: contact is now set to 'Fez'\n  Update: registry is now set to 'Fez'\n  Update: dispatch is now set to 'Reykjavik'\n  Update: reference is now set to 'Zanzibar'\n  Update: coordinator is now set to 'Kotor'\n  Update: destination is now set to 'Mandalay'\n  Update: location is now set to 'Reykjavik'\n  Update: delegate is now set to 'Bruges'\n  Update: contact is now set to 'Trieste'\n  Update: registry is now set to 'Oulu'\n  Update: dispatch is now set to 'Ulaanbaatar'\n  Update: reference is now set to 'Recife'\n  Update: coordinator is now set to 'Trieste'\n  Update: destination is now set to 'Kotor'\n  Update: location is now set to 'Kotor'\n  Update: delegate is now set to 'Recife'\n  Update: contact is now set to 'Tbilisi'\n  Update: registry is now set to 'Ulaanbaatar'\n  Update: dispatch is now set to 'Kumasi'\n  Update: reference is now set to 'Bruges'\n  Update: coordinator is now set to 'Oulu'\n  Update: destination is now set to 'Jaipur'\n  Update: location is now set to 'Jaipur'\n  Update: delegate is now set to 'Bruges'\n  Update: contact is now set to 'Plovdiv'\n  Update: registry is now set to 'Tbilisi'\n  Update: dispatch is now set to 'Gdansk'\n  Update: reference is now set to 'Plovdiv'\n  Update: coordinator is now set to 'Reykjavik'\n  Update: destination is now set to 'Luang Prabang'\n  Update: location is now set to 'Plovdiv'\n  Update: delegate is now set to 'Fez'\n  Update: contact is now set to 'Tallinn'\n  Update: registry is now set to 'Valetta'\n  Update: dispatch is now set to 'Zanzibar'\n  Update: reference is now set to 'Cartagena'\n  Update: coordinator is now set to 'Gdansk'\n  Update: destination is now set to 'Kumasi'\n  Update: location is now set to 'Mandalay'\n  Update: delegate is now set to 'Bruges'\n  Update: contact is now set to 'Recife'\n  Update: registry is now set to 'Fez'\n  Update: dispatch is now set to 'Tbilisi'\n  Update: reference is now set to 'Tallinn'\n  Update: coordinator is now set to 'Kotor'\n  Update: destination is now set to 'Recife'\n  Update: location is now set to 'Bruges'\n  Update: delegate is now set to 'Tallinn'\n  Update: contact is now set to 'Zanzibar'\n  Update: registry is now set to 'Recife'\n  Update: dispatch is now set to 'Tallinn'\n  Update: reference is now set to 'Cusco'\n  Update: coordinator is now set to 'Valetta'\n  Update: destination is now set to 'Trieste'\n  Update: location is now set to 'Kotor'\n  Update: delegate is now set to 'Cartagena'\n  Update: contact is now set to 'Valetta'\n  Update: registry is now set to 'Mandalay'\n  Update: dispatch is now set to 'Recife'\n  Update: reference is now set to 'Oulu'\n  Update: coordinator is now set to 'Bruges'\n  Update: destination is now set to 'Cartagena'\n  Update: location is now set to 'Recife'\n  Update: delegate is now set to 'Cusco'\n  Update: contact is now set to 'Cartagena'\n  Update: registry is now set to 'Jaipur'\n  Update: dispatch is now set to 'Bruges'\n  Update: reference is now set to 'Kotor'\n  Update: coordinator is now set to 'Ulaanbaatar'\n  Update: destination is now set to 'Zanzibar'\n  Update: location is now set to 'Cusco'\n  Update: delegate is now set to 'Tallinn'\n  Update: contact is now set to 'Reykjavik'\n  Update: registry is now set to 'Cartagena'\n  Update: dispatch is now set to 'Tallinn'\n  Update: reference is now set to 'Valetta'\n  Update: coordinator is now set to 'Jaipur'\n  Update: destination is now set to 'Tallinn'\n  Update: location is now set to 'Kotor'\n  Update: delegate is now set to 'Zanzibar'\n  Update: contact is now set to 'Gdansk'\n  Update: registry is now set to 'Cusco'\n  Update: dispatch is now set to 'Ulaanbaatar'\n  Update: reference is now set to 'Oulu'\n  Update: coordinator is now set to 'Cusco'\n  Update: destination is now set to 'Plovdiv'\n  Update: location is now set to 'Gdansk'\n  Update: delegate is now set to 'Oulu'\n  Update: contact is now set to 'Reykjavik'\n  Update: registry is now set to 'Ulaanbaatar'\n  Update: dispatch is now set to 'Tallinn'\n  Update: reference is now set to 'Kotor'\n  Update: coordinator is now set to 'Kumasi'\n  Update: destination is now set to 'Cusco'\n  Update: location is now set to 'Reykjavik'\n  Update: delegate is now set to 'Trieste'\n  Update: contact is now set to 'Ulaanbaatar'\n  Update: registry is now set to 'Oulu'\n  Update: dispatch is now set to 'Valetta'\n  Update: reference is now set to 'Tbilisi'\n  Update: coordinator is now set to 'Mandalay'\n  Update: destination is now set to 'Tbilisi'\n  Update: location is now set to 'Tbilisi'\n  Update: delegate is now set to 'Ulaanbaatar'\n  Update: contact is now set to 'Mandalay'\n  Update: registry is now set to 'Recife'\n  Update: dispatch is now set to 'Tallinn'\n  Update: reference is now set to 'Valetta'\n  Update: coordinator is now set to 'Tallinn'\n  Update: destination is now set to 'Valetta'\n  Update: location is now set to 'Kotor'\n  Update: delegate is now set to 'Fez'\n  Update: contact is now set to 'Bruges'\n  Update: registry is now set to 'Valetta'\n  Update: dispatch is now set to 'Trieste'\n  Update: reference is now set to 'Gdansk'\n  Update: coordinator is now set to 'Trieste'\n  Update: destination is now set to 'Luang Prabang'\n  Update: location is now set to 'Ulaanbaatar'\n  Update: delegate is now set to 'Tallinn'\n  Update: contact is now set to 'Trieste'\n  Update: registry is now set to 'Zanzibar'\n  Update: dispatch is now set to 'Gdansk'\n  Update: reference is now set to 'Mandalay'\n  Update: coordinator is now set to 'Cusco'\n  Update: destination is now set to 'Zanzibar'\n  Update: location is now set to 'Luang Prabang'\n  Update: delegate is now set to 'Jaipur'\n  Update: contact is now set to 'Valetta'\n  Update: registry is now set to 'Mandalay'\n  Update: dispatch is now set to 'Tbilisi'\n  Update: reference is now set to 'Cusco'\n  Update: coordinator is now set to 'Cartagena'\n  Update: destination is now set to 'Cusco'\n  Update: location is now set to 'Plovdiv'\n  Update: delegate is now set to 'Kumasi'\n  Update: contact is now set to 'Zanzibar'\n  Update: registry is now set to 'Valetta'\n  Update: dispatch is now set to 'Luang Prabang'\n  Update: reference is now set to 'Jaipur'\n  Update: coordinator is now set to 'Oulu'\n  Update: destination is now set to 'Oulu'\n  Update: location is now set to 'Mandalay'\n  Update: delegate is now set to 'Gdansk'\n  Update: contact is now set to 'Gdansk'\n  Update: registry is now set to 'Ulaanbaatar'\n  Update: dispatch is now set to 'Kotor'\n  Update: reference is now set to 'Tbilisi'\n  Update: coordinator is now set to 'Luang Prabang'\n  Update: destination is now set to 'Cartagena'\n  Update: location is now set to 'Gdansk'\n  Update: delegate is now set to 'Mandalay'\n  Update: contact is now set to 'Trieste'\n  Update: registry is now set to 'Oulu'\n  Update: dispatch is now set to 'Recife'\n  Update: reference is now set to 'Valetta'\n  Update: coordinator is now set to 'Plovdiv'\n  Update: destination is now set to 'Plovdiv'\n  Update: location is now set to 'Zanzibar'\n  Update: delegate is now set to 'Cusco'\n  Update: contact is now set to 'Kumasi'\n  Update: registry is now set to 'Gdansk'\n  Update: dispatch is now set to 'Kumasi'\n  Update: reference is now set to 'Tallinn'\n  Update: coordinator is now set to 'Gdansk'\n  Update: destination is now set to 'Tbilisi'\n  Update: location is now set to 'Tallinn'\n  Update: delegate is now set to 'Luang Prabang'\n  Update: contact is now set to 'Valetta'\n  Update: registry is now set to 'Valetta'\n  Update: dispatch is now set to 'Zanzibar'\n  Update: reference is now set to 'Kotor'\n  Update: coordinator is now set to 'Zanzibar'\n  Update: destination is now set to 'Ulaanbaatar'\n  Update: location is now set to 'Kumasi'\n  Update: delegate is now set to 'Mandalay'\n  Update: contact is now set to 'Trieste'\n  Update: registry is now set to 'Kumasi'\n  Update: dispatch is now set to 'Gdansk'\n  Update: reference is now set to 'Jaipur'\n  Update: coordinator is now set to 'Cusco'\n  Update: destination is now set to 'Mandalay'\n  Update: location is now set to 'Tbilisi'\n  Update: delegate is now set to 'Fez'\n  Update: contact is now set to 'Kotor'\n\nWhat is the FINAL value of each record?\nANSWER:\n- registry: [final value]\n- dispatch: [final value]\n- reference: [final value]\n- coordinator: [final value]\n- destination: [final value]\n- location: [final value]\n- delegate: [final value]\n- contact: [final value]\n\nAlso answer these verification questions (Yes or No):\nV1. Was 'Cusco' ever assigned to registry? [Yes/No]\nV2. Was 'Tbilisi' ever assigned to dispatch? [Yes/No]\nV3. Was 'Oulu' ever assigned to reference? [Yes/No]\nV4. Was 'Valetta' ever assigned to coordinator? [Yes/No]",
  "gold_json": "{\"final_values\": {\"registry\": \"Kumasi\", \"dispatch\": \"Gdansk\", \"reference\": \"Jaipur\", \"coordinator\": \"Cusco\", \"destination\": \"Mandalay\", \"location\": \"Tbilisi\", \"delegate\": \"Fez\", \"contact\": \"Kotor\"}, \"key_names\": [\"registry\", \"dispatch\", \"reference\", \"coordinator\", \"destination\", \"location\", \"delegate\", \"contact\"]}"
 },
 {
  "task_id": "interference_frontier_034",
  "task_type": "interference",
  "difficulty": "Frontier",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: coordinator is now set to '498'\n  Update: liaison is now set to '737'\n  Update: location is now set to '205'\n  Update: dispatch is now set to '927'\n  Update: reference is now set to '212'\n  Update: delegate is now set to '571'\n  Update: contact is now set to '473'\n  Update: assignment is now set to '114'\n  Update: coordinator is now set to '810'\n  Update: liaison is now set to '548'\n  Update: location is now set to '562'\n  Update: dispatch is now set to '801'\n  Update: reference is now set to '330'\n  Update: delegate is now set to '343'\n  Update: contact is now set to '686'\n  Update: assignment is now set to '862'\n  Update: coordinator is now set to '551'\n  Update: liaison is now set to '931'\n  Update: location is now set to '792'\n  Update: dispatch is now set to '682'\n  Update: reference is now set to '396'\n  Update: delegate is now set to '665'\n  Update: contact is now set to '269'\n  Update: assignment is now set to '658'\n  Update: coordinator is now set to '366'\n  Update: liaison is now set to '630'\n  Update: location is now set to '585'\n  Update: dispatch is now set to '782'\n  Update: reference is now set to '375'\n  Update: delegate is now set to '555'\n  Update: contact is now set to '703'\n  Update: assignment is now set to '895'\n  Update: coordinator is now set to '327'\n  Update: liaison is now set to '465'\n  Update: location is now set to '446'\n  Update: dispatch is now set to '934'\n  Update: reference is now set to '304'\n  Update: delegate is now set to '240'\n  Update: contact is now set to '295'\n  Update: assignment is now set to '806'\n  Update: coordinator is now set to '189'\n  Update: liaison is now set to '954'\n  Update: location is now set to '927'\n  Update: dispatch is now set to '586'\n  Update: reference is now set to '561'\n  Update: delegate is now set to '536'\n  Update: contact is now set to '823'\n  Update: assignment is now set to '501'\n  Update: coordinator is now set to '510'\n  Update: liaison is now set to '885'\n  Update: location is now set to '426'\n  Update: dispatch is now set to '873'\n  Update: reference is now set to '844'\n  Update: delegate is now set to '646'\n  Update: contact is now set to '954'\n  Update: assignment is now set to '273'\n  Update: coordinator is now set to '171'\n  Update: liaison is now set to '866'\n  Update: location is now set to '876'\n  Update: dispatch is now set to '469'\n  Update: reference is now set to '131'\n  Update: delegate is now set to '512'\n  Update: contact is now set to '598'\n  Update: assignment is now set to '842'\n  Update: coordinator is now set to '158'\n  Update: liaison is now set to '923'\n  Update: location is now set to '591'\n  Update: dispatch is now set to '756'\n  Update: reference is now set to '461'\n  Update: delegate is now set to '968'\n  Update: contact is now set to '981'\n  Update: assignment is now set to '319'\n  Update: coordinator is now set to '880'\n  Update: liaison is now set to '488'\n  Update: location is now set to '666'\n  Update: dispatch is now set to '713'\n  Update: reference is now set to '830'\n  Update: delegate is now set to '491'\n  Update: contact is now set to '507'\n  Update: assignment is now set to '536'\n  Update: coordinator is now set to '304'\n  Update: liaison is now set to '248'\n  Update: location is now set to '499'\n  Update: dispatch is now set to '448'\n  Update: reference is now set to '736'\n  Update: delegate is now set to '647'\n  Update: contact is now set to '683'\n  Update: assignment is now set to '718'\n  Update: coordinator is now set to '234'\n  Update: liaison is now set to '394'\n  Update: location is now set to '697'\n  Update: dispatch is now set to '795'\n  Update: reference is now set to '944'\n  Update: delegate is now set to '546'\n  Update: contact is now set to '346'\n  Update: assignment is now set to '550'\n  Update: coordinator is now set to '364'\n  Update: liaison is now set to '927'\n  Update: location is now set to '474'\n  Update: dispatch is now set to '359'\n  Update: reference is now set to '828'\n  Update: delegate is now set to '511'\n  Update: contact is now set to '957'\n  Update: assignment is now set to '839'\n  Update: coordinator is now set to '689'\n  Update: liaison is now set to '579'\n  Update: location is now set to '173'\n  Update: dispatch is now set to '391'\n  Update: reference is now set to '673'\n  Update: delegate is now set to '799'\n  Update: contact is now set to '679'\n  Update: assignment is now set to '927'\n  Update: coordinator is now set to '996'\n  Update: liaison is now set to '393'\n  Update: location is now set to '129'\n  Update: dispatch is now set to '840'\n  Update: reference is now set to '615'\n  Update: delegate is now set to '685'\n  Update: contact is now set to '366'\n  Update: assignment is now set to '279'\n  Update: coordinator is now set to '192'\n  Update: liaison is now set to '325'\n  Update: location is now set to '997'\n  Update: dispatch is now set to '868'\n  Update: reference is now set to '463'\n  Update: delegate is now set to '820'\n  Update: contact is now set to '817'\n  Update: assignment is now set to '739'\n  Update: coordinator is now set to '124'\n  Update: liaison is now set to '848'\n  Update: location is now set to '824'\n  Update: dispatch is now set to '545'\n  Update: reference is now set to '319'\n  Update: delegate is now set to '927'\n  Update: contact is now set to '563'\n  Update: assignment is now set to '794'\n  Update: coordinator is now set to '479'\n  Update: liaison is now set to '876'\n  Update: location is now set to '133'\n  Update: dispatch is now set to '812'\n  Update: reference is now set to '721'\n  Update: delegate is now set to '713'\n  Update: contact is now set to '804'\n  Update: assignment is now set to '228'\n  Update: coordinator is now set to '188'\n  Update: liaison is now set to '623'\n  Update: location is now set to '444'\n  Update: dispatch is now set to '986'\n  Update: reference is now set to '200'\n  Update: delegate is now set to '790'\n  Update: contact is now set to '523'\n  Update: assignment is now set to '237'\n  Update: coordinator is now set to '412'\n  Update: liaison is now set to '395'\n  Update: location is now set to '448'\n  Update: dispatch is now set to '472'\n  Update: reference is now set to '545'\n  Update: delegate is now set to '146'\n  Update: contact is now set to '422'\n  Update: assignment is now set to '356'\n  Update: coordinator is now set to '283'\n  Update: liaison is now set to '730'\n  Update: location is now set to '975'\n  Update: dispatch is now set to '405'\n  Update: reference is now set to '736'\n  Update: delegate is now set to '483'\n  Update: contact is now set to '810'\n  Update: assignment is now set to '551'\n  Update: coordinator is now set to '694'\n  Update: liaison is now set to '131'\n  Update: location is now set to '995'\n  Update: dispatch is now set to '441'\n  Update: reference is now set to '314'\n  Update: delegate is now set to '963'\n  Update: contact is now set to '712'\n  Update: assignment is now set to '669'\n  Update: coordinator is now set to '149'\n  Update: liaison is now set to '303'\n  Update: location is now set to '419'\n  Update: dispatch is now set to '484'\n  Update: reference is now set to '813'\n  Update: delegate is now set to '488'\n  Update: contact is now set to '517'\n  Update: assignment is now set to '129'\n  Update: coordinator is now set to '316'\n  Update: liaison is now set to '478'\n  Update: location is now set to '942'\n  Update: dispatch is now set to '895'\n  Update: reference is now set to '695'\n  Update: delegate is now set to '523'\n  Update: contact is now set to '126'\n  Update: assignment is now set to '568'\n  Update: coordinator is now set to '105'\n  Update: liaison is now set to '684'\n  Update: location is now set to '227'\n  Update: dispatch is now set to '685'\n  Update: reference is now set to '688'\n  Update: delegate is now set to '957'\n  Update: contact is now set to '322'\n  Update: assignment is now set to '394'\n  Update: coordinator is now set to '759'\n  Update: liaison is now set to '320'\n  Update: location is now set to '431'\n  Update: dispatch is now set to '663'\n  Update: reference is now set to '527'\n  Update: delegate is now set to '964'\n  Update: contact is now set to '485'\n  Update: assignment is now set to '148'\n  Update: coordinator is now set to '574'\n  Update: liaison is now set to '952'\n  Update: location is now set to '207'\n  Update: dispatch is now set to '994'\n  Update: reference is now set to '542'\n  Update: delegate is now set to '555'\n  Update: contact is now set to '105'\n  Update: assignment is now set to '490'\n  Update: coordinator is now set to '629'\n  Update: liaison is now set to '617'\n  Update: location is now set to '843'\n  Update: dispatch is now set to '490'\n  Update: reference is now set to '759'\n  Update: delegate is now set to '536'\n  Update: contact is now set to '214'\n  Update: assignment is now set to '665'\n  Update: coordinator is now set to '280'\n  Update: liaison is now set to '717'\n  Update: location is now set to '488'\n  Update: dispatch is now set to '600'\n  Update: reference is now set to '961'\n  Update: delegate is now set to '508'\n  Update: contact is now set to '511'\n  Update: assignment is now set to '647'\n  Update: coordinator is now set to '372'\n  Update: liaison is now set to '572'\n  Update: location is now set to '973'\n  Update: dispatch is now set to '396'\n  Update: reference is now set to '230'\n  Update: delegate is now set to '630'\n  Update: contact is now set to '302'\n  Update: assignment is now set to '673'\n  Update: coordinator is now set to '125'\n  Update: liaison is now set to '503'\n  Update: location is now set to '552'\n  Update: dispatch is now set to '376'\n  Update: reference is now set to '157'\n  Update: delegate is now set to '974'\n  Update: contact is now set to '767'\n  Update: assignment is now set to '861'\n  Update: coordinator is now set to '370'\n  Update: liaison is now set to '247'\n  Update: location is now set to '924'\n  Update: dispatch is now set to '935'\n  Update: reference is now set to '418'\n  Update: delegate is now set to '226'\n  Update: contact is now set to '925'\n  Update: assignment is now set to '260'\n  Update: coordinator is now set to '712'\n  Update: liaison is now set to '382'\n  Update: location is now set to '818'\n  Update: dispatch is now set to '832'\n  Update: reference is now set to '864'\n  Update: delegate is now set to '269'\n  Update: contact is now set to '618'\n  Update: assignment is now set to '442'\n  Update: coordinator is now set to '556'\n  Update: liaison is now set to '269'\n  Update: location is now set to '584'\n  Update: dispatch is now set to '162'\n  Update: reference is now set to '459'\n  Update: delegate is now set to '398'\n  Update: contact is now set to '458'\n  Update: assignment is now set to '856'\n  Update: coordinator is now set to '480'\n  Update: liaison is now set to '564'\n  Update: location is now set to '154'\n  Update: dispatch is now set to '151'\n  Update: reference is now set to '327'\n  Update: delegate is now set to '960'\n  Update: contact is now set to '568'\n  Update: assignment is now set to '273'\n  Update: coordinator is now set to '741'\n  Update: liaison is now set to '991'\n  Update: location is now set to '368'\n  Update: dispatch is now set to '288'\n  Update: reference is now set to '630'\n  Update: delegate is now set to '457'\n  Update: contact is now set to '292'\n  Update: assignment is now set to '661'\n  Update: coordinator is now set to '460'\n  Update: liaison is now set to '271'\n  Update: location is now set to '583'\n  Update: dispatch is now set to '909'\n  Update: reference is now set to '537'\n  Update: delegate is now set to '644'\n  Update: contact is now set to '768'\n  Update: assignment is now set to '677'\n  Update: coordinator is now set to '186'\n  Update: liaison is now set to '154'\n  Update: location is now set to '220'\n  Update: dispatch is now set to '145'\n  Update: reference is now set to '870'\n  Update: delegate is now set to '114'\n  Update: contact is now set to '279'\n  Update: assignment is now set to '898'\n  Update: coordinator is now set to '756'\n  Update: liaison is now set to '710'\n  Update: location is now set to '391'\n  Update: dispatch is now set to '125'\n  Update: reference is now set to '451'\n  Update: delegate is now set to '837'\n  Update: contact is now set to '229'\n  Update: assignment is now set to '569'\n  Update: coordinator is now set to '735'\n  Update: liaison is now set to '995'\n  Update: location is now set to '696'\n  Update: dispatch is now set to '871'\n  Update: reference is now set to '741'\n  Update: delegate is now set to '452'\n  Update: contact is now set to '412'\n  Update: assignment is now set to '729'\n  Update: coordinator is now set to '109'\n  Update: liaison is now set to '358'\n  Update: location is now set to '258'\n  Update: dispatch is now set to '492'\n  Update: reference is now set to '236'\n  Update: delegate is now set to '939'\n  Update: contact is now set to '849'\n  Update: assignment is now set to '511'\n  Update: coordinator is now set to '255'\n  Update: liaison is now set to '827'\n  Update: location is now set to '350'\n  Update: dispatch is now set to '862'\n  Update: reference is now set to '500'\n  Update: delegate is now set to '697'\n  Update: contact is now set to '293'\n  Update: assignment is now set to '548'\n  Update: coordinator is now set to '829'\n  Update: liaison is now set to '652'\n  Update: location is now set to '990'\n  Update: dispatch is now set to '844'\n  Update: reference is now set to '848'\n  Update: delegate is now set to '174'\n  Update: contact is now set to '696'\n  Update: assignment is now set to '996'\n  Update: coordinator is now set to '202'\n  Update: liaison is now set to '233'\n  Update: location is now set to '379'\n  Update: dispatch is now set to '376'\n  Update: reference is now set to '799'\n  Update: delegate is now set to '249'\n  Update: contact is now set to '259'\n  Update: assignment is now set to '782'\n  Update: coordinator is now set to '672'\n  Update: liaison is now set to '703'\n  Update: location is now set to '195'\n  Update: dispatch is now set to '540'\n  Update: reference is now set to '439'\n  Update: delegate is now set to '973'\n  Update: contact is now set to '121'\n  Update: assignment is now set to '137'\n  Update: coordinator is now set to '519'\n  Update: liaison is now set to '736'\n  Update: location is now set to '303'\n  Update: dispatch is now set to '542'\n  Update: reference is now set to '183'\n  Update: delegate is now set to '949'\n  Update: contact is now set to '372'\n  Update: assignment is now set to '481'\n  Update: coordinator is now set to '821'\n  Update: liaison is now set to '846'\n  Update: location is now set to '684'\n  Update: dispatch is now set to '645'\n  Update: reference is now set to '437'\n  Update: delegate is now set to '504'\n  Update: contact is now set to '200'\n  Update: assignment is now set to '848'\n  Update: coordinator is now set to '232'\n  Update: liaison is now set to '336'\n  Update: location is now set to '734'\n  Update: dispatch is now set to '310'\n  Update: reference is now set to '740'\n  Update: delegate is now set to '796'\n  Update: contact is now set to '391'\n  Update: assignment is now set to '772'\n  Update: coordinator is now set to '827'\n  Update: liaison is now set to '977'\n  Update: location is now set to '360'\n  Update: dispatch is now set to '274'\n  Update: reference is now set to '171'\n  Update: delegate is now set to '890'\n  Update: contact is now set to '480'\n  Update: assignment is now set to '379'\n  Update: coordinator is now set to '643'\n  Update: liaison is now set to '806'\n  Update: location is now set to '599'\n  Update: dispatch is now set to '638'\n  Update: reference is now set to '186'\n  Update: delegate is now set to '700'\n  Update: contact is now set to '847'\n  Update: assignment is now set to '189'\n\nWhat is the FINAL value of each record?\nANSWER:\n- coordinator: [final value]\n- liaison: [final value]\n- location: [final value]\n- dispatch: [final value]\n- reference: [final value]\n- delegate: [final value]\n- contact: [final value]\n- assignment: [final value]\n\nAlso answer these verification questions (Yes or No):\nV1. Was '186' ever assigned to coordinator? [Yes/No]\nV2. Was '154' ever assigned to liaison? [Yes/No]\nV3. Was '666' ever assigned to location? [Yes/No]\nV4. Was '713' ever assigned to dispatch? [Yes/No]",
  "gold_json": "{\"final_values\": {\"coordinator\": \"643\", \"liaison\": \"806\", \"location\": \"599\", \"dispatch\": \"638\", \"reference\": \"186\", \"delegate\": \"700\", \"contact\": \"847\", \"assignment\": \"189\"}, \"key_names\": [\"coordinator\", \"liaison\", \"location\", \"dispatch\", \"reference\", \"delegate\", \"contact\", \"assignment\"]}"
 },
 {
  "task_id": "interference_frontier_035",
  "task_type": "interference",
  "difficulty": "Frontier",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: assignment is now set to 'Joaquin'\n  Update: location is now set to 'Orla'\n  Update: destination is now set to 'Freya'\n  Update: reference is now set to 'Ugo'\n  Update: dispatch is now set to 'Yara'\n  Update: contact is now set to 'Elara'\n  Update: registry is now set to 'Uma'\n  Update: liaison is now set to 'Paloma'\n  Update: assignment is now set to 'Willa'\n  Update: location is now set to 'Magnus'\n  Update: destination is now set to 'Celine'\n  Update: reference is now set to 'Wren'\n  Update: dispatch is now set to 'Amara'\n  Update: contact is now set to 'Willa'\n  Update: registry is now set to 'Ravi'\n  Update: liaison is now set to 'Sigrid'\n  Update: assignment is now set to 'Orla'\n  Update: location is now set to 'Bram'\n  Update: destination is now set to 'Kenji'\n  Update: reference is now set to 'Joaquin'\n  Update: dispatch is now set to 'Kaia'\n  Update: contact is now set to 'Elara'\n  Update: registry is now set to 'Xander'\n  Update: liaison is now set to 'Leif'\n  Update: assignment is now set to 'Maren'\n  Update: location is now set to 'Uma'\n  Update: destination is now set to 'Kaia'\n  Update: reference is now set to 'Lumi'\n  Update: dispatch is now set to 'Zain'\n  Update: contact is now set to 'Bashir'\n  Update: registry is now set to 'Greta'\n  Update: liaison is now set to 'Ugo'\n  Update: assignment is now set to 'Dmitri'\n  Update: location is now set to 'Orla'\n  Update: destination is now set to 'Gael'\n  Update: reference is now set to 'Magnus'\n  Update: dispatch is now set to 'Kaia'\n  Update: contact is now set to 'Xander'\n  Update: registry is now set to 'Maren'\n  Update: liaison is now set to 'Paloma'\n  Update: assignment is now set to 'Yara'\n  Update: location is now set to 'Tala'\n  Update: destination is now set to 'Nico'\n  Update: reference is now set to 'Bram'\n  Update: dispatch is now set to 'Orla'\n  Update: contact is now set to 'Soren'\n  Update: registry is now set to 'Ines'\n  Update: liaison is now set to 'Kaia'\n  Update: assignment is now set to 'Orla'\n  Update: location is now set to 'Qadir'\n  Update: destination is now set to 'Ines'\n  Update: reference is now set to 'Willa'\n  Update: dispatch is now set to 'Ines'\n  Update: contact is now set to 'Priya'\n  Update: registry is now set to 'Tariq'\n  Update: liaison is now set to 'Wren'\n  Update: assignment is now set to 'Sigrid'\n  Update: location is now set to 'Viktor'\n  Update: destination is now set to 'Haruto'\n  Update: reference is now set to 'Idris'\n  Update: dispatch is now set to 'Zora'\n  Update: contact is now set to 'Elio'\n  Update: registry is now set to 'Viktor'\n  Update: liaison is now set to 'Kenji'\n  Update: assignment is now set to 'Bram'\n  Update: location is now set to 'Vesna'\n  Update: destination is now set to 'Willa'\n  Update: reference is now set to 'Yara'\n  Update: dispatch is now set to 'Qadir'\n  Update: contact is now set to 'Joaquin'\n  Update: registry is now set to 'Maren'\n  Update: liaison is now set to 'Magnus'\n  Update: assignment is now set to 'Kenji'\n  Update: location is now set to 'Paloma'\n  Update: destination is now set to 'Dmitri'\n  Update: reference is now set to 'Tariq'\n  Update: dispatch is now set to 'Gael'\n  Update: contact is now set to 'Vesna'\n  Update: registry is now set to 'Hana'\n  Update: liaison is now set to 'Nico'\n  Update: assignment is now set to 'Nico'\n  Update: location is now set to 'Xander'\n  Update: destination is now set to 'Joelle'\n  Update: reference is now set to 'Leif'\n  Update: dispatch is now set to 'Magnus'\n  Update: contact is now set to 'Dariush'\n  Update: registry is now set to 'Amara'\n  Update: liaison is now set to 'Freya'\n  Update: assignment is now set to 'Kenji'\n  Update: location is now set to 'Hana'\n  Update: destination is now set to 'Femi'\n  Update: reference is now set to 'Lumi'\n  Update: dispatch is now set to 'Zain'\n  Update: contact is now set to 'Viktor'\n  Update: registry is now set to 'Paloma'\n  Update: liaison is now set to 'Bram'\n  Update: assignment is now set to 'Magnus'\n  Update: location is now set to 'Femi'\n  Update: destination is now set to 'Lumi'\n  Update: reference is now set to 'Zora'\n  Update: dispatch is now set to 'Vesna'\n  Update: contact is now set to 'Qadir'\n  Update: registry is now set to 'Ugo'\n  Update: liaison is now set to 'Colette'\n  Update: assignment is now set to 'Nico'\n  Update: location is now set to 'Kenji'\n  Update: destination is now set to 'Elara'\n  Update: reference is now set to 'Magnus'\n  Update: dispatch is now set to 'Dmitri'\n  Update: contact is now set to 'Soren'\n  Update: registry is now set to 'Nico'\n  Update: liaison is now set to 'Hana'\n  Update: assignment is now set to 'Sigrid'\n  Update: location is now set to 'Joelle'\n  Update: destination is now set to 'Joaquin'\n  Update: reference is now set to 'Colette'\n  Update: dispatch is now set to 'Zora'\n  Update: contact is now set to 'Yuki'\n  Update: registry is now set to 'Dariush'\n  Update: liaison is now set to 'Orla'\n  Update: assignment is now set to 'Paloma'\n  Update: location is now set to 'Paloma'\n  Update: destination is now set to 'Viktor'\n  Update: reference is now set to 'Kenji'\n  Update: dispatch is now set to 'Freya'\n  Update: contact is now set to 'Greta'\n  Update: registry is now set to 'Runa'\n  Update: liaison is now set to 'Celine'\n  Update: assignment is now set to 'Hana'\n  Update: location is now set to 'Magnus'\n  Update: destination is now set to 'Bashir'\n  Update: reference is now set to 'Tala'\n  Update: dispatch is now set to 'Bashir'\n  Update: contact is now set to 'Viktor'\n  Update: registry is now set to 'Yara'\n  Update: liaison is now set to 'Joelle'\n  Update: assignment is now set to 'Paloma'\n  Update: location is now set to 'Femi'\n  Update: destination is now set to 'Celine'\n  Update: reference is now set to 'Dmitri'\n  Update: dispatch is now set to 'Celine'\n  Update: contact is now set to 'Kaia'\n  Update: registry is now set to 'Sigrid'\n  Update: liaison is now set to 'Ugo'\n  Update: assignment is now set to 'Joelle'\n  Update: location is now set to 'Haruto'\n  Update: destination is now set to 'Idris'\n  Update: reference is now set to 'Adaeze'\n  Update: dispatch is now set to 'Haruto'\n  Update: contact is now set to 'Hana'\n  Update: registry is now set to 'Hana'\n  Update: liaison is now set to 'Xander'\n  Update: assignment is now set to 'Dariush'\n  Update: location is now set to 'Bram'\n  Update: destination is now set to 'Kaia'\n  Update: reference is now set to 'Nico'\n  Update: dispatch is now set to 'Kaia'\n  Update: contact is now set to 'Runa'\n  Update: registry is now set to 'Maren'\n  Update: liaison is now set to 'Gael'\n  Update: assignment is now set to 'Ines'\n  Update: location is now set to 'Leif'\n  Update: destination is now set to 'Joelle'\n  Update: reference is now set to 'Ravi'\n  Update: dispatch is now set to 'Sigrid'\n  Update: contact is now set to 'Yara'\n  Update: registry is now set to 'Qadir'\n  Update: liaison is now set to 'Hana'\n  Update: assignment is now set to 'Lumi'\n  Update: location is now set to 'Uma'\n  Update: destination is now set to 'Bram'\n  Update: reference is now set to 'Gael'\n  Update: dispatch is now set to 'Amara'\n  Update: contact is now set to 'Zora'\n  Update: registry is now set to 'Yara'\n  Update: liaison is now set to 'Maren'\n  Update: assignment is now set to 'Ravi'\n  Update: location is now set to 'Joaquin'\n  Update: destination is now set to 'Magnus'\n  Update: reference is now set to 'Amara'\n  Update: dispatch is now set to 'Elara'\n  Update: contact is now set to 'Yara'\n  Update: registry is now set to 'Tariq'\n  Update: liaison is now set to 'Wren'\n  Update: assignment is now set to 'Colette'\n  Update: location is now set to 'Willa'\n  Update: destination is now set to 'Celine'\n  Update: reference is now set to 'Dmitri'\n  Update: dispatch is now set to 'Tala'\n  Update: contact is now set to 'Uma'\n  Update: registry is now set to 'Xander'\n  Update: liaison is now set to 'Zain'\n  Update: assignment is now set to 'Ravi'\n  Update: location is now set to 'Zora'\n  Update: destination is now set to 'Priya'\n  Update: reference is now set to 'Wren'\n  Update: dispatch is now set to 'Orla'\n  Update: contact is now set to 'Elio'\n  Update: registry is now set to 'Tala'\n  Update: liaison is now set to 'Ravi'\n  Update: assignment is now set to 'Kenji'\n  Update: location is now set to 'Dariush'\n  Update: destination is now set to 'Tariq'\n  Update: reference is now set to 'Ravi'\n  Update: dispatch is now set to 'Elio'\n  Update: contact is now set to 'Amara'\n  Update: registry is now set to 'Bashir'\n  Update: liaison is now set to 'Dariush'\n  Update: assignment is now set to 'Ines'\n  Update: location is now set to 'Amara'\n  Update: destination is now set to 'Orla'\n  Update: reference is now set to 'Wren'\n  Update: dispatch is now set to 'Celine'\n  Update: contact is now set to 'Kaia'\n  Update: registry is now set to 'Dariush'\n  Update: liaison is now set to 'Ines'\n  Update: assignment is now set to 'Willa'\n  Update: location is now set to 'Maren'\n  Update: destination is now set to 'Qadir'\n  Update: reference is now set to 'Joelle'\n  Update: dispatch is now set to 'Kaia'\n  Update: contact is now set to 'Tala'\n  Update: registry is now set to 'Tala'\n  Update: liaison is now set to 'Kaia'\n  Update: assignment is now set to 'Dmitri'\n  Update: location is now set to 'Nalini'\n  Update: destination is now set to 'Nico'\n  Update: reference is now set to 'Sigrid'\n  Update: dispatch is now set to 'Elio'\n  Update: contact is now set to 'Hana'\n  Update: registry is now set to 'Hana'\n  Update: liaison is now set to 'Joaquin'\n  Update: assignment is now set to 'Dariush'\n  Update: location is now set to 'Tariq'\n  Update: destination is now set to 'Kenji'\n  Update: reference is now set to 'Ravi'\n  Update: dispatch is now set to 'Priya'\n  Update: contact is now set to 'Dmitri'\n  Update: registry is now set to 'Willa'\n  Update: liaison is now set to 'Ravi'\n  Update: assignment is now set to 'Bashir'\n  Update: location is now set to 'Lumi'\n  Update: destination is now set to 'Colette'\n  Update: reference is now set to 'Bashir'\n  Update: dispatch is now set to 'Leif'\n  Update: contact is now set to 'Yara'\n  Update: registry is now set to 'Sigrid'\n  Update: liaison is now set to 'Bram'\n  Update: assignment is now set to 'Femi'\n  Update: location is now set to 'Zora'\n  Update: destination is now set to 'Elio'\n  Update: reference is now set to 'Lumi'\n  Update: dispatch is now set to 'Joelle'\n  Update: contact is now set to 'Ravi'\n  Update: registry is now set to 'Willa'\n  Update: liaison is now set to 'Gael'\n  Update: assignment is now set to 'Soren'\n  Update: location is now set to 'Kenji'\n  Update: destination is now set to 'Wren'\n  Update: reference is now set to 'Olena'\n  Update: dispatch is now set to 'Bram'\n  Update: contact is now set to 'Elara'\n  Update: registry is now set to 'Leif'\n  Update: liaison is now set to 'Magnus'\n  Update: assignment is now set to 'Gael'\n  Update: location is now set to 'Qadir'\n  Update: destination is now set to 'Priya'\n  Update: reference is now set to 'Ines'\n  Update: dispatch is now set to 'Joaquin'\n  Update: contact is now set to 'Yuki'\n  Update: registry is now set to 'Bashir'\n  Update: liaison is now set to 'Zain'\n  Update: assignment is now set to 'Freya'\n  Update: location is now set to 'Idris'\n  Update: destination is now set to 'Elio'\n  Update: reference is now set to 'Ugo'\n  Update: dispatch is now set to 'Joelle'\n  Update: contact is now set to 'Willa'\n  Update: registry is now set to 'Elio'\n  Update: liaison is now set to 'Yuki'\n  Update: assignment is now set to 'Paloma'\n  Update: location is now set to 'Elio'\n  Update: destination is now set to 'Magnus'\n  Update: reference is now set to 'Tala'\n  Update: dispatch is now set to 'Runa'\n  Update: contact is now set to 'Dmitri'\n  Update: registry is now set to 'Uma'\n  Update: liaison is now set to 'Yara'\n  Update: assignment is now set to 'Idris'\n  Update: location is now set to 'Bram'\n  Update: destination is now set to 'Orla'\n  Update: reference is now set to 'Paloma'\n  Update: dispatch is now set to 'Dariush'\n  Update: contact is now set to 'Kaia'\n  Update: registry is now set to 'Priya'\n  Update: liaison is now set to 'Lumi'\n  Update: assignment is now set to 'Uma'\n  Update: location is now set to 'Nico'\n  Update: destination is now set to 'Sigrid'\n  Update: reference is now set to 'Leif'\n  Update: dispatch is now set to 'Magnus'\n  Update: contact is now set to 'Qadir'\n  Update: registry is now set to 'Bram'\n  Update: liaison is now set to 'Leif'\n  Update: assignment is now set to 'Bashir'\n  Update: location is now set to 'Tala'\n  Update: destination is now set to 'Xander'\n  Update: reference is now set to 'Wren'\n  Update: dispatch is now set to 'Femi'\n  Update: contact is now set to 'Lumi'\n  Update: registry is now set to 'Uma'\n  Update: liaison is now set to 'Amara'\n  Update: assignment is now set to 'Ravi'\n  Update: location is now set to 'Zora'\n  Update: destination is now set to 'Nalini'\n  Update: reference is now set to 'Adaeze'\n  Update: dispatch is now set to 'Bram'\n  Update: contact is now set to 'Zain'\n  Update: registry is now set to 'Yara'\n  Update: liaison is now set to 'Ugo'\n  Update: assignment is now set to 'Wren'\n  Update: location is now set to 'Haruto'\n  Update: destination is now set to 'Kenji'\n  Update: reference is now set to 'Magnus'\n  Update: dispatch is now set to 'Joaquin'\n  Update: contact is now set to 'Tala'\n  Update: registry is now set to 'Greta'\n  Update: liaison is now set to 'Kaia'\n  Update: assignment is now set to 'Zain'\n  Update: location is now set to 'Dariush'\n  Update: destination is now set to 'Priya'\n  Update: reference is now set to 'Runa'\n  Update: dispatch is now set to 'Bram'\n  Update: contact is now set to 'Bashir'\n  Update: registry is now set to 'Femi'\n  Update: liaison is now set to 'Nalini'\n  Update: assignment is now set to 'Adaeze'\n  Update: location is now set to 'Gael'\n  Update: destination is now set to 'Yara'\n  Update: reference is now set to 'Wren'\n  Update: dispatch is now set to 'Paloma'\n  Update: contact is now set to 'Adaeze'\n  Update: registry is now set to 'Maren'\n  Update: liaison is now set to 'Uma'\n  Update: assignment is now set to 'Haruto'\n  Update: location is now set to 'Amara'\n  Update: destination is now set to 'Lumi'\n  Update: reference is now set to 'Kaia'\n  Update: dispatch is now set to 'Idris'\n  Update: contact is now set to 'Tala'\n  Update: registry is now set to 'Ravi'\n  Update: liaison is now set to 'Hana'\n  Update: assignment is now set to 'Amara'\n  Update: location is now set to 'Magnus'\n  Update: destination is now set to 'Ugo'\n  Update: reference is now set to 'Ines'\n  Update: dispatch is now set to 'Dariush'\n  Update: contact is now set to 'Zora'\n  Update: registry is now set to 'Olena'\n  Update: liaison is now set to 'Ines'\n  Update: assignment is now set to 'Idris'\n  Update: location is now set to 'Runa'\n  Update: destination is now set to 'Kenji'\n  Update: reference is now set to 'Elio'\n  Update: dispatch is now set to 'Vesna'\n  Update: contact is now set to 'Idris'\n  Update: registry is now set to 'Colette'\n  Update: liaison is now set to 'Amara'\n  Update: assignment is now set to 'Femi'\n  Update: location is now set to 'Vesna'\n  Update: destination is now set to 'Ravi'\n  Update: reference is now set to 'Tala'\n  Update: dispatch is now set to 'Kenji'\n  Update: contact is now set to 'Joaquin'\n  Update: registry is now set to 'Tariq'\n  Update: liaison is now set to 'Gael'\n  Update: assignment is now set to 'Ugo'\n  Update: location is now set to 'Freya'\n  Update: destination is now set to 'Kaia'\n  Update: reference is now set to 'Magnus'\n  Update: dispatch is now set to 'Vesna'\n  Update: contact is now set to 'Zain'\n  Update: registry is now set to 'Celine'\n  Update: liaison is now set to 'Zora'\n  Update: assignment is now set to 'Colette'\n  Update: location is now set to 'Sigrid'\n  Update: destination is now set to 'Magnus'\n  Update: reference is now set to 'Uma'\n  Update: dispatch is now set to 'Ugo'\n  Update: contact is now set to 'Yuki'\n  Update: registry is now set to 'Ugo'\n  Update: liaison is now set to 'Bram'\n  Update: assignment is now set to 'Nalini'\n  Update: location is now set to 'Gael'\n  Update: destination is now set to 'Tala'\n  Update: reference is now set to 'Priya'\n  Update: dispatch is now set to 'Uma'\n  Update: contact is now set to 'Celine'\n  Update: registry is now set to 'Nico'\n  Update: liaison is now set to 'Kenji'\n\nWhat is the FINAL value of each record?\nANSWER:\n- assignment: [final value]\n- location: [final value]\n- destination: [final value]\n- reference: [final value]\n- dispatch: [final value]\n- contact: [final value]\n- registry: [final value]\n- liaison: [final value]\n\nAlso answer these verification questions (Yes or No):\nV1. Was 'Paloma' ever assigned to assignment? [Yes/No]\nV2. Was 'Dariush' ever assigned to location? [Yes/No]\nV3. Was 'Dmitri' ever assigned to destination? [Yes/No]\nV4. Was 'Yara' ever assigned to reference? [Yes/No]",
  "gold_json": "{\"final_values\": {\"assignment\": \"Nalini\", \"location\": \"Gael\", \"destination\": \"Tala\", \"reference\": \"Priya\", \"dispatch\": \"Uma\", \"contact\": \"Celine\", \"registry\": \"Nico\", \"liaison\": \"Kenji\"}, \"key_names\": [\"assignment\", \"location\", \"destination\", \"reference\", \"dispatch\", \"contact\", \"registry\", \"liaison\"]}"
 },
 {
  "task_id": "interference_frontier_036",
  "task_type": "interference",
  "difficulty": "Frontier",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: liaison is now set to '373'\n  Update: contact is now set to '405'\n  Update: assignment is now set to '496'\n  Update: registry is now set to '714'\n  Update: delegate is now set to '536'\n  Update: reference is now set to '959'\n  Update: dispatch is now set to '765'\n  Update: location is now set to '495'\n  Update: liaison is now set to '642'\n  Update: contact is now set to '881'\n  Update: assignment is now set to '572'\n  Update: registry is now set to '934'\n  Update: delegate is now set to '103'\n  Update: reference is now set to '505'\n  Update: dispatch is now set to '758'\n  Update: location is now set to '539'\n  Update: liaison is now set to '461'\n  Update: contact is now set to '172'\n  Update: assignment is now set to '429'\n  Update: registry is now set to '506'\n  Update: delegate is now set to '351'\n  Update: reference is now set to '166'\n  Update: dispatch is now set to '521'\n  Update: location is now set to '643'\n  Update: liaison is now set to '719'\n  Update: contact is now set to '695'\n  Update: assignment is now set to '119'\n  Update: registry is now set to '960'\n  Update: delegate is now set to '971'\n  Update: reference is now set to '375'\n  Update: dispatch is now set to '917'\n  Update: location is now set to '379'\n  Update: liaison is now set to '479'\n  Update: contact is now set to '931'\n  Update: assignment is now set to '643'\n  Update: registry is now set to '415'\n  Update: delegate is now set to '175'\n  Update: reference is now set to '711'\n  Update: dispatch is now set to '506'\n  Update: location is now set to '315'\n  Update: liaison is now set to '122'\n  Update: contact is now set to '823'\n  Update: assignment is now set to '407'\n  Update: registry is now set to '641'\n  Update: delegate is now set to '775'\n  Update: reference is now set to '683'\n  Update: dispatch is now set to '192'\n  Update: location is now set to '241'\n  Update: liaison is now set to '878'\n  Update: contact is now set to '413'\n  Update: assignment is now set to '680'\n  Update: registry is now set to '929'\n  Update: delegate is now set to '485'\n  Update: reference is now set to '632'\n  Update: dispatch is now set to '915'\n  Update: location is now set to '565'\n  Update: liaison is now set to '450'\n  Update: contact is now set to '220'\n  Update: assignment is now set to '559'\n  Update: registry is now set to '658'\n  Update: delegate is now set to '527'\n  Update: reference is now set to '752'\n  Update: dispatch is now set to '963'\n  Update: location is now set to '338'\n  Update: liaison is now set to '618'\n  Update: contact is now set to '824'\n  Update: assignment is now set to '209'\n  Update: registry is now set to '554'\n  Update: delegate is now set to '306'\n  Update: reference is now set to '310'\n  Update: dispatch is now set to '354'\n  Update: location is now set to '584'\n  Update: liaison is now set to '197'\n  Update: contact is now set to '599'\n  Update: assignment is now set to '564'\n  Update: registry is now set to '503'\n  Update: delegate is now set to '296'\n  Update: reference is now set to '295'\n  Update: dispatch is now set to '872'\n  Update: location is now set to '331'\n  Update: liaison is now set to '489'\n  Update: contact is now set to '551'\n  Update: assignment is now set to '205'\n  Update: registry is now set to '255'\n  Update: delegate is now set to '303'\n  Update: reference is now set to '848'\n  Update: dispatch is now set to '379'\n  Update: location is now set to '504'\n  Update: liaison is now set to '511'\n  Update: contact is now set to '271'\n  Update: assignment is now set to '313'\n  Update: registry is now set to '651'\n  Update: delegate is now set to '315'\n  Update: reference is now set to '638'\n  Update: dispatch is now set to '588'\n  Update: location is now set to '407'\n  Update: liaison is now set to '568'\n  Update: contact is now set to '700'\n  Update: assignment is now set to '774'\n  Update: registry is now set to '772'\n  Update: delegate is now set to '908'\n  Update: reference is now set to '261'\n  Update: dispatch is now set to '100'\n  Update: location is now set to '732'\n  Update: liaison is now set to '359'\n  Update: contact is now set to '718'\n  Update: assignment is now set to '736'\n  Update: registry is now set to '304'\n  Update: delegate is now set to '550'\n  Update: reference is now set to '444'\n  Update: dispatch is now set to '791'\n  Update: location is now set to '961'\n  Update: liaison is now set to '246'\n  Update: contact is now set to '183'\n  Update: assignment is now set to '802'\n  Update: registry is now set to '906'\n  Update: delegate is now set to '312'\n  Update: reference is now set to '486'\n  Update: dispatch is now set to '331'\n  Update: location is now set to '383'\n  Update: liaison is now set to '756'\n  Update: contact is now set to '769'\n  Update: assignment is now set to '424'\n  Update: registry is now set to '621'\n  Update: delegate is now set to '697'\n  Update: reference is now set to '689'\n  Update: dispatch is now set to '338'\n  Update: location is now set to '983'\n  Update: liaison is now set to '345'\n  Update: contact is now set to '533'\n  Update: assignment is now set to '662'\n  Update: registry is now set to '547'\n  Update: delegate is now set to '946'\n  Update: reference is now set to '980'\n  Update: dispatch is now set to '878'\n  Update: location is now set to '338'\n  Update: liaison is now set to '159'\n  Update: contact is now set to '633'\n  Update: assignment is now set to '288'\n  Update: registry is now set to '767'\n  Update: delegate is now set to '106'\n  Update: reference is now set to '285'\n  Update: dispatch is now set to '889'\n  Update: location is now set to '159'\n  Update: liaison is now set to '144'\n  Update: contact is now set to '600'\n  Update: assignment is now set to '107'\n  Update: registry is now set to '443'\n  Update: delegate is now set to '454'\n  Update: reference is now set to '487'\n  Update: dispatch is now set to '137'\n  Update: location is now set to '981'\n  Update: liaison is now set to '998'\n  Update: contact is now set to '789'\n  Update: assignment is now set to '961'\n  Update: registry is now set to '336'\n  Update: delegate is now set to '875'\n  Update: reference is now set to '428'\n  Update: dispatch is now set to '815'\n  Update: location is now set to '219'\n  Update: liaison is now set to '581'\n  Update: contact is now set to '313'\n  Update: assignment is now set to '279'\n  Update: registry is now set to '703'\n  Update: delegate is now set to '236'\n  Update: reference is now set to '172'\n  Update: dispatch is now set to '969'\n  Update: location is now set to '120'\n  Update: liaison is now set to '345'\n  Update: contact is now set to '678'\n  Update: assignment is now set to '561'\n  Update: registry is now set to '799'\n  Update: delegate is now set to '551'\n  Update: reference is now set to '688'\n  Update: dispatch is now set to '379'\n  Update: location is now set to '693'\n  Update: liaison is now set to '612'\n  Update: contact is now set to '539'\n  Update: assignment is now set to '773'\n  Update: registry is now set to '769'\n  Update: delegate is now set to '447'\n  Update: reference is now set to '223'\n  Update: dispatch is now set to '230'\n  Update: location is now set to '226'\n  Update: liaison is now set to '382'\n  Update: contact is now set to '621'\n  Update: assignment is now set to '294'\n  Update: registry is now set to '116'\n  Update: delegate is now set to '911'\n  Update: reference is now set to '810'\n  Update: dispatch is now set to '161'\n  Update: location is now set to '713'\n  Update: liaison is now set to '660'\n  Update: contact is now set to '997'\n  Update: assignment is now set to '645'\n  Update: registry is now set to '753'\n  Update: delegate is now set to '449'\n  Update: reference is now set to '539'\n  Update: dispatch is now set to '166'\n  Update: location is now set to '381'\n  Update: liaison is now set to '147'\n  Update: contact is now set to '838'\n  Update: assignment is now set to '437'\n  Update: registry is now set to '177'\n  Update: delegate is now set to '154'\n  Update: reference is now set to '574'\n  Update: dispatch is now set to '112'\n  Update: location is now set to '272'\n  Update: liaison is now set to '441'\n  Update: contact is now set to '886'\n  Update: assignment is now set to '887'\n  Update: registry is now set to '974'\n  Update: delegate is now set to '534'\n  Update: reference is now set to '166'\n  Update: dispatch is now set to '911'\n  Update: location is now set to '125'\n  Update: liaison is now set to '946'\n  Update: contact is now set to '192'\n  Update: assignment is now set to '200'\n  Update: registry is now set to '479'\n  Update: delegate is now set to '649'\n  Update: reference is now set to '675'\n  Update: dispatch is now set to '549'\n  Update: location is now set to '152'\n  Update: liaison is now set to '809'\n  Update: contact is now set to '481'\n  Update: assignment is now set to '226'\n  Update: registry is now set to '235'\n  Update: delegate is now set to '928'\n  Update: reference is now set to '885'\n  Update: dispatch is now set to '828'\n  Update: location is now set to '404'\n  Update: liaison is now set to '532'\n  Update: contact is now set to '307'\n  Update: assignment is now set to '758'\n  Update: registry is now set to '105'\n  Update: delegate is now set to '457'\n  Update: reference is now set to '211'\n  Update: dispatch is now set to '176'\n  Update: location is now set to '372'\n  Update: liaison is now set to '339'\n  Update: contact is now set to '604'\n  Update: assignment is now set to '116'\n  Update: registry is now set to '807'\n  Update: delegate is now set to '143'\n  Update: reference is now set to '558'\n  Update: dispatch is now set to '483'\n  Update: location is now set to '855'\n  Update: liaison is now set to '766'\n  Update: contact is now set to '458'\n  Update: assignment is now set to '352'\n  Update: registry is now set to '528'\n  Update: delegate is now set to '253'\n  Update: reference is now set to '206'\n  Update: dispatch is now set to '142'\n  Update: location is now set to '301'\n  Update: liaison is now set to '251'\n  Update: contact is now set to '750'\n  Update: assignment is now set to '578'\n  Update: registry is now set to '929'\n  Update: delegate is now set to '829'\n  Update: reference is now set to '518'\n  Update: dispatch is now set to '237'\n  Update: location is now set to '149'\n  Update: liaison is now set to '989'\n  Update: contact is now set to '993'\n  Update: assignment is now set to '731'\n  Update: registry is now set to '947'\n  Update: delegate is now set to '172'\n  Update: reference is now set to '606'\n  Update: dispatch is now set to '180'\n  Update: location is now set to '175'\n  Update: liaison is now set to '417'\n  Update: contact is now set to '870'\n  Update: assignment is now set to '368'\n  Update: registry is now set to '581'\n  Update: delegate is now set to '863'\n  Update: reference is now set to '743'\n  Update: dispatch is now set to '907'\n  Update: location is now set to '280'\n  Update: liaison is now set to '700'\n  Update: contact is now set to '299'\n  Update: assignment is now set to '120'\n  Update: registry is now set to '532'\n  Update: delegate is now set to '170'\n  Update: reference is now set to '718'\n  Update: dispatch is now set to '425'\n  Update: location is now set to '722'\n  Update: liaison is now set to '997'\n  Update: contact is now set to '592'\n  Update: assignment is now set to '222'\n  Update: registry is now set to '127'\n  Update: delegate is now set to '597'\n  Update: reference is now set to '680'\n  Update: dispatch is now set to '986'\n  Update: location is now set to '773'\n  Update: liaison is now set to '434'\n  Update: contact is now set to '633'\n  Update: assignment is now set to '381'\n  Update: registry is now set to '165'\n  Update: delegate is now set to '246'\n  Update: reference is now set to '707'\n  Update: dispatch is now set to '710'\n  Update: location is now set to '222'\n  Update: liaison is now set to '243'\n  Update: contact is now set to '393'\n  Update: assignment is now set to '628'\n  Update: registry is now set to '285'\n  Update: delegate is now set to '606'\n  Update: reference is now set to '422'\n  Update: dispatch is now set to '825'\n  Update: location is now set to '144'\n  Update: liaison is now set to '868'\n  Update: contact is now set to '492'\n  Update: assignment is now set to '496'\n  Update: registry is now set to '719'\n  Update: delegate is now set to '192'\n  Update: reference is now set to '317'\n  Update: dispatch is now set to '649'\n  Update: location is now set to '243'\n  Update: liaison is now set to '305'\n  Update: contact is now set to '920'\n  Update: assignment is now set to '321'\n  Update: registry is now set to '908'\n  Update: delegate is now set to '612'\n  Update: reference is now set to '749'\n  Update: dispatch is now set to '702'\n  Update: location is now set to '782'\n  Update: liaison is now set to '128'\n  Update: contact is now set to '166'\n  Update: assignment is now set to '688'\n  Update: registry is now set to '439'\n  Update: delegate is now set to '955'\n  Update: reference is now set to '325'\n  Update: dispatch is now set to '644'\n  Update: location is now set to '150'\n  Update: liaison is now set to '786'\n  Update: contact is now set to '732'\n  Update: assignment is now set to '242'\n  Update: registry is now set to '840'\n  Update: delegate is now set to '257'\n  Update: reference is now set to '167'\n  Update: dispatch is now set to '860'\n  Update: location is now set to '594'\n  Update: liaison is now set to '893'\n  Update: contact is now set to '309'\n  Update: assignment is now set to '729'\n  Update: registry is now set to '928'\n  Update: delegate is now set to '307'\n  Update: reference is now set to '566'\n  Update: dispatch is now set to '234'\n  Update: location is now set to '498'\n  Update: liaison is now set to '492'\n  Update: contact is now set to '458'\n  Update: assignment is now set to '215'\n  Update: registry is now set to '807'\n  Update: delegate is now set to '448'\n  Update: reference is now set to '420'\n  Update: dispatch is now set to '492'\n  Update: location is now set to '344'\n  Update: liaison is now set to '425'\n  Update: contact is now set to '226'\n  Update: assignment is now set to '355'\n  Update: registry is now set to '273'\n  Update: delegate is now set to '264'\n  Update: reference is now set to '527'\n  Update: dispatch is now set to '844'\n  Update: location is now set to '680'\n  Update: liaison is now set to '601'\n  Update: contact is now set to '758'\n  Update: assignment is now set to '590'\n  Update: registry is now set to '699'\n  Update: delegate is now set to '960'\n  Update: reference is now set to '909'\n  Update: dispatch is now set to '146'\n  Update: location is now set to '451'\n  Update: liaison is now set to '341'\n  Update: contact is now set to '497'\n  Update: assignment is now set to '415'\n  Update: registry is now set to '933'\n  Update: delegate is now set to '438'\n  Update: reference is now set to '952'\n  Update: dispatch is now set to '707'\n  Update: location is now set to '596'\n  Update: liaison is now set to '149'\n  Update: contact is now set to '590'\n  Update: assignment is now set to '904'\n  Update: registry is now set to '817'\n  Update: delegate is now set to '288'\n  Update: reference is now set to '530'\n  Update: dispatch is now set to '535'\n  Update: location is now set to '348'\n  Update: liaison is now set to '187'\n  Update: contact is now set to '416'\n  Update: assignment is now set to '693'\n  Update: registry is now set to '841'\n  Update: delegate is now set to '551'\n  Update: reference is now set to '188'\n  Update: dispatch is now set to '480'\n  Update: location is now set to '512'\n\nWhat is the FINAL value of each record?\nANSWER:\n- liaison: [final value]\n- contact: [final value]\n- assignment: [final value]\n- registry: [final value]\n- delegate: [final value]\n- reference: [final value]\n- dispatch: [final value]\n- location: [final value]\n\nAlso answer these verification questions (Yes or No):\nV1. Was '809' ever assigned to liaison? [Yes/No]\nV2. Was '700' ever assigned to contact? [Yes/No]\nV3. Was '564' ever assigned to assignment? [Yes/No]\nV4. Was '479' ever assigned to registry? [Yes/No]",
  "gold_json": "{\"final_values\": {\"liaison\": \"187\", \"contact\": \"416\", \"assignment\": \"693\", \"registry\": \"841\", \"delegate\": \"551\", \"reference\": \"188\", \"dispatch\": \"480\", \"location\": \"512\"}, \"key_names\": [\"liaison\", \"contact\", \"assignment\", \"registry\", \"delegate\", \"reference\", \"dispatch\", \"location\"]}"
 },
 {
  "task_id": "interference_frontier_037",
  "task_type": "interference",
  "difficulty": "Frontier",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: assignment is now set to '254'\n  Update: location is now set to '243'\n  Update: destination is now set to '948'\n  Update: dispatch is now set to '385'\n  Update: contact is now set to '350'\n  Update: coordinator is now set to '320'\n  Update: registry is now set to '250'\n  Update: delegate is now set to '139'\n  Update: assignment is now set to '577'\n  Update: location is now set to '826'\n  Update: destination is now set to '929'\n  Update: dispatch is now set to '531'\n  Update: contact is now set to '827'\n  Update: coordinator is now set to '185'\n  Update: registry is now set to '492'\n  Update: delegate is now set to '206'\n  Update: assignment is now set to '502'\n  Update: location is now set to '614'\n  Update: destination is now set to '715'\n  Update: dispatch is now set to '983'\n  Update: contact is now set to '657'\n  Update: coordinator is now set to '709'\n  Update: registry is now set to '419'\n  Update: delegate is now set to '512'\n  Update: assignment is now set to '123'\n  Update: location is now set to '150'\n  Update: destination is now set to '834'\n  Update: dispatch is now set to '561'\n  Update: contact is now set to '283'\n  Update: coordinator is now set to '472'\n  Update: registry is now set to '224'\n  Update: delegate is now set to '967'\n  Update: assignment is now set to '497'\n  Update: location is now set to '432'\n  Update: destination is now set to '368'\n  Update: dispatch is now set to '542'\n  Update: contact is now set to '191'\n  Update: coordinator is now set to '950'\n  Update: registry is now set to '742'\n  Update: delegate is now set to '200'\n  Update: assignment is now set to '683'\n  Update: location is now set to '195'\n  Update: destination is now set to '468'\n  Update: dispatch is now set to '856'\n  Update: contact is now set to '250'\n  Update: coordinator is now set to '827'\n  Update: registry is now set to '996'\n  Update: delegate is now set to '646'\n  Update: assignment is now set to '701'\n  Update: location is now set to '939'\n  Update: destination is now set to '283'\n  Update: dispatch is now set to '982'\n  Update: contact is now set to '630'\n  Update: coordinator is now set to '547'\n  Update: registry is now set to '419'\n  Update: delegate is now set to '737'\n  Update: assignment is now set to '138'\n  Update: location is now set to '764'\n  Update: destination is now set to '545'\n  Update: dispatch is now set to '810'\n  Update: contact is now set to '159'\n  Update: coordinator is now set to '919'\n  Update: registry is now set to '804'\n  Update: delegate is now set to '559'\n  Update: assignment is now set to '581'\n  Update: location is now set to '671'\n  Update: destination is now set to '994'\n  Update: dispatch is now set to '343'\n  Update: contact is now set to '246'\n  Update: coordinator is now set to '906'\n  Update: registry is now set to '238'\n  Update: delegate is now set to '158'\n  Update: assignment is now set to '590'\n  Update: location is now set to '683'\n  Update: destination is now set to '635'\n  Update: dispatch is now set to '122'\n  Update: contact is now set to '984'\n  Update: coordinator is now set to '146'\n  Update: registry is now set to '839'\n  Update: delegate is now set to '404'\n  Update: assignment is now set to '384'\n  Update: location is now set to '988'\n  Update: destination is now set to '357'\n  Update: dispatch is now set to '932'\n  Update: contact is now set to '443'\n  Update: coordinator is now set to '431'\n  Update: registry is now set to '932'\n  Update: delegate is now set to '238'\n  Update: assignment is now set to '852'\n  Update: location is now set to '925'\n  Update: destination is now set to '346'\n  Update: dispatch is now set to '108'\n  Update: contact is now set to '937'\n  Update: coordinator is now set to '464'\n  Update: registry is now set to '317'\n  Update: delegate is now set to '972'\n  Update: assignment is now set to '318'\n  Update: location is now set to '192'\n  Update: destination is now set to '831'\n  Update: dispatch is now set to '562'\n  Update: contact is now set to '613'\n  Update: coordinator is now set to '884'\n  Update: registry is now set to '628'\n  Update: delegate is now set to '606'\n  Update: assignment is now set to '789'\n  Update: location is now set to '784'\n  Update: destination is now set to '160'\n  Update: dispatch is now set to '512'\n  Update: contact is now set to '905'\n  Update: coordinator is now set to '735'\n  Update: registry is now set to '255'\n  Update: delegate is now set to '984'\n  Update: assignment is now set to '787'\n  Update: location is now set to '365'\n  Update: destination is now set to '879'\n  Update: dispatch is now set to '797'\n  Update: contact is now set to '137'\n  Update: coordinator is now set to '786'\n  Update: registry is now set to '530'\n  Update: delegate is now set to '763'\n  Update: assignment is now set to '449'\n  Update: location is now set to '538'\n  Update: destination is now set to '477'\n  Update: dispatch is now set to '268'\n  Update: contact is now set to '796'\n  Update: coordinator is now set to '848'\n  Update: registry is now set to '570'\n  Update: delegate is now set to '571'\n  Update: assignment is now set to '414'\n  Update: location is now set to '444'\n  Update: destination is now set to '738'\n  Update: dispatch is now set to '219'\n  Update: contact is now set to '914'\n  Update: coordinator is now set to '284'\n  Update: registry is now set to '698'\n  Update: delegate is now set to '901'\n  Update: assignment is now set to '593'\n  Update: location is now set to '172'\n  Update: destination is now set to '927'\n  Update: dispatch is now set to '973'\n  Update: contact is now set to '109'\n  Update: coordinator is now set to '826'\n  Update: registry is now set to '212'\n  Update: delegate is now set to '990'\n  Update: assignment is now set to '992'\n  Update: location is now set to '523'\n  Update: destination is now set to '904'\n  Update: dispatch is now set to '905'\n  Update: contact is now set to '110'\n  Update: coordinator is now set to '870'\n  Update: registry is now set to '810'\n  Update: delegate is now set to '545'\n  Update: assignment is now set to '429'\n  Update: location is now set to '269'\n  Update: destination is now set to '757'\n  Update: dispatch is now set to '549'\n  Update: contact is now set to '493'\n  Update: coordinator is now set to '868'\n  Update: registry is now set to '531'\n  Update: delegate is now set to '577'\n  Update: assignment is now set to '353'\n  Update: location is now set to '744'\n  Update: destination is now set to '740'\n  Update: dispatch is now set to '938'\n  Update: contact is now set to '355'\n  Update: coordinator is now set to '899'\n  Update: registry is now set to '526'\n  Update: delegate is now set to '509'\n  Update: assignment is now set to '860'\n  Update: location is now set to '656'\n  Update: destination is now set to '648'\n  Update: dispatch is now set to '620'\n  Update: contact is now set to '720'\n  Update: coordinator is now set to '155'\n  Update: registry is now set to '444'\n  Update: delegate is now set to '823'\n  Update: assignment is now set to '465'\n  Update: location is now set to '925'\n  Update: destination is now set to '284'\n  Update: dispatch is now set to '604'\n  Update: contact is now set to '983'\n  Update: coordinator is now set to '545'\n  Update: registry is now set to '377'\n  Update: delegate is now set to '799'\n  Update: assignment is now set to '309'\n  Update: location is now set to '851'\n  Update: destination is now set to '304'\n  Update: dispatch is now set to '821'\n  Update: contact is now set to '497'\n  Update: coordinator is now set to '880'\n  Update: registry is now set to '955'\n  Update: delegate is now set to '886'\n  Update: assignment is now set to '196'\n  Update: location is now set to '961'\n  Update: destination is now set to '869'\n  Update: dispatch is now set to '242'\n  Update: contact is now set to '659'\n  Update: coordinator is now set to '805'\n  Update: registry is now set to '858'\n  Update: delegate is now set to '815'\n  Update: assignment is now set to '669'\n  Update: location is now set to '835'\n  Update: destination is now set to '235'\n  Update: dispatch is now set to '153'\n  Update: contact is now set to '490'\n  Update: coordinator is now set to '627'\n  Update: registry is now set to '640'\n  Update: delegate is now set to '581'\n  Update: assignment is now set to '690'\n  Update: location is now set to '713'\n  Update: destination is now set to '139'\n  Update: dispatch is now set to '972'\n  Update: contact is now set to '133'\n  Update: coordinator is now set to '973'\n  Update: registry is now set to '144'\n  Update: delegate is now set to '400'\n  Update: assignment is now set to '861'\n  Update: location is now set to '850'\n  Update: destination is now set to '581'\n  Update: dispatch is now set to '338'\n  Update: contact is now set to '619'\n  Update: coordinator is now set to '666'\n  Update: registry is now set to '211'\n  Update: delegate is now set to '288'\n  Update: assignment is now set to '215'\n  Update: location is now set to '288'\n  Update: destination is now set to '387'\n  Update: dispatch is now set to '469'\n  Update: contact is now set to '860'\n  Update: coordinator is now set to '876'\n  Update: registry is now set to '993'\n  Update: delegate is now set to '608'\n  Update: assignment is now set to '761'\n  Update: location is now set to '199'\n  Update: destination is now set to '357'\n  Update: dispatch is now set to '994'\n  Update: contact is now set to '337'\n  Update: coordinator is now set to '505'\n  Update: registry is now set to '876'\n  Update: delegate is now set to '198'\n  Update: assignment is now set to '454'\n  Update: location is now set to '329'\n  Update: destination is now set to '454'\n  Update: dispatch is now set to '799'\n  Update: contact is now set to '292'\n  Update: coordinator is now set to '165'\n  Update: registry is now set to '849'\n  Update: delegate is now set to '558'\n  Update: assignment is now set to '219'\n  Update: location is now set to '310'\n  Update: destination is now set to '607'\n  Update: dispatch is now set to '586'\n  Update: contact is now set to '634'\n  Update: coordinator is now set to '962'\n  Update: registry is now set to '815'\n  Update: delegate is now set to '616'\n  Update: assignment is now set to '421'\n  Update: location is now set to '266'\n  Update: destination is now set to '236'\n  Update: dispatch is now set to '326'\n  Update: contact is now set to '534'\n  Update: coordinator is now set to '164'\n  Update: registry is now set to '228'\n  Update: delegate is now set to '308'\n  Update: assignment is now set to '121'\n  Update: location is now set to '156'\n  Update: destination is now set to '511'\n  Update: dispatch is now set to '314'\n  Update: contact is now set to '289'\n  Update: coordinator is now set to '282'\n  Update: registry is now set to '661'\n  Update: delegate is now set to '913'\n  Update: assignment is now set to '255'\n  Update: location is now set to '174'\n  Update: destination is now set to '465'\n  Update: dispatch is now set to '518'\n  Update: contact is now set to '273'\n  Update: coordinator is now set to '758'\n  Update: registry is now set to '409'\n  Update: delegate is now set to '789'\n  Update: assignment is now set to '839'\n  Update: location is now set to '975'\n  Update: destination is now set to '827'\n  Update: dispatch is now set to '210'\n  Update: contact is now set to '638'\n  Update: coordinator is now set to '788'\n  Update: registry is now set to '754'\n  Update: delegate is now set to '672'\n  Update: assignment is now set to '806'\n  Update: location is now set to '930'\n  Update: destination is now set to '394'\n  Update: dispatch is now set to '529'\n  Update: contact is now set to '169'\n  Update: coordinator is now set to '982'\n  Update: registry is now set to '421'\n  Update: delegate is now set to '914'\n  Update: assignment is now set to '337'\n  Update: location is now set to '588'\n  Update: destination is now set to '952'\n  Update: dispatch is now set to '277'\n  Update: contact is now set to '297'\n  Update: coordinator is now set to '141'\n  Update: registry is now set to '286'\n  Update: delegate is now set to '175'\n  Update: assignment is now set to '149'\n  Update: location is now set to '511'\n  Update: destination is now set to '607'\n  Update: dispatch is now set to '592'\n  Update: contact is now set to '569'\n  Update: coordinator is now set to '734'\n  Update: registry is now set to '328'\n  Update: delegate is now set to '824'\n  Update: assignment is now set to '828'\n  Update: location is now set to '572'\n  Update: destination is now set to '525'\n  Update: dispatch is now set to '423'\n  Update: contact is now set to '440'\n  Update: coordinator is now set to '443'\n  Update: registry is now set to '564'\n  Update: delegate is now set to '453'\n  Update: assignment is now set to '817'\n  Update: location is now set to '501'\n  Update: destination is now set to '831'\n  Update: dispatch is now set to '779'\n  Update: contact is now set to '631'\n  Update: coordinator is now set to '160'\n  Update: registry is now set to '755'\n  Update: delegate is now set to '525'\n  Update: assignment is now set to '979'\n  Update: location is now set to '505'\n  Update: destination is now set to '314'\n  Update: dispatch is now set to '272'\n  Update: contact is now set to '469'\n  Update: coordinator is now set to '528'\n  Update: registry is now set to '324'\n  Update: delegate is now set to '351'\n  Update: assignment is now set to '443'\n  Update: location is now set to '606'\n  Update: destination is now set to '245'\n  Update: dispatch is now set to '693'\n  Update: contact is now set to '808'\n  Update: coordinator is now set to '482'\n  Update: registry is now set to '809'\n  Update: delegate is now set to '391'\n  Update: assignment is now set to '936'\n  Update: location is now set to '760'\n  Update: destination is now set to '331'\n  Update: dispatch is now set to '762'\n  Update: contact is now set to '776'\n  Update: coordinator is now set to '789'\n  Update: registry is now set to '732'\n  Update: delegate is now set to '837'\n  Update: assignment is now set to '206'\n  Update: location is now set to '989'\n  Update: destination is now set to '153'\n  Update: dispatch is now set to '676'\n  Update: contact is now set to '642'\n  Update: coordinator is now set to '377'\n  Update: registry is now set to '736'\n  Update: delegate is now set to '384'\n  Update: assignment is now set to '701'\n  Update: location is now set to '459'\n  Update: destination is now set to '329'\n  Update: dispatch is now set to '980'\n  Update: contact is now set to '574'\n  Update: coordinator is now set to '577'\n  Update: registry is now set to '564'\n  Update: delegate is now set to '387'\n  Update: assignment is now set to '783'\n  Update: location is now set to '328'\n  Update: destination is now set to '984'\n  Update: dispatch is now set to '917'\n  Update: contact is now set to '604'\n  Update: coordinator is now set to '720'\n  Update: registry is now set to '918'\n  Update: delegate is now set to '774'\n  Update: assignment is now set to '538'\n  Update: location is now set to '520'\n  Update: destination is now set to '990'\n  Update: dispatch is now set to '183'\n  Update: contact is now set to '869'\n  Update: coordinator is now set to '103'\n  Update: registry is now set to '392'\n  Update: delegate is now set to '419'\n  Update: assignment is now set to '739'\n  Update: location is now set to '748'\n  Update: destination is now set to '839'\n  Update: dispatch is now set to '776'\n  Update: contact is now set to '901'\n  Update: coordinator is now set to '328'\n  Update: registry is now set to '862'\n  Update: delegate is now set to '288'\n  Update: assignment is now set to '986'\n  Update: location is now set to '143'\n  Update: destination is now set to '279'\n  Update: dispatch is now set to '531'\n  Update: contact is now set to '290'\n  Update: coordinator is now set to '872'\n  Update: registry is now set to '929'\n  Update: delegate is now set to '285'\n\nWhat is the FINAL value of each record?\nANSWER:\n- assignment: [final value]\n- location: [final value]\n- destination: [final value]\n- dispatch: [final value]\n- contact: [final value]\n- coordinator: [final value]\n- registry: [final value]\n- delegate: [final value]\n\nAlso answer these verification questions (Yes or No):\nV1. Was '538' ever assigned to assignment? [Yes/No]\nV2. Was '243' ever assigned to location? [Yes/No]\nV3. Was '990' ever assigned to destination? [Yes/No]\nV4. Was '518' ever assigned to dispatch? [Yes/No]",
  "gold_json": "{\"final_values\": {\"assignment\": \"986\", \"location\": \"143\", \"destination\": \"279\", \"dispatch\": \"531\", \"contact\": \"290\", \"coordinator\": \"872\", \"registry\": \"929\", \"delegate\": \"285\"}, \"key_names\": [\"assignment\", \"location\", \"destination\", \"dispatch\", \"contact\", \"coordinator\", \"registry\", \"delegate\"]}"
 },
 {
  "task_id": "interference_frontier_038",
  "task_type": "interference",
  "difficulty": "Frontier",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: location is now set to 'Joelle'\n  Update: reference is now set to 'Hana'\n  Update: liaison is now set to 'Willa'\n  Update: assignment is now set to 'Olena'\n  Update: delegate is now set to 'Paloma'\n  Update: registry is now set to 'Idris'\n  Update: dispatch is now set to 'Amara'\n  Update: contact is now set to 'Lumi'\n  Update: location is now set to 'Runa'\n  Update: reference is now set to 'Joelle'\n  Update: liaison is now set to 'Lumi'\n  Update: assignment is now set to 'Zain'\n  Update: delegate is now set to 'Bashir'\n  Update: registry is now set to 'Celine'\n  Update: dispatch is now set to 'Soren'\n  Update: contact is now set to 'Tariq'\n  Update: location is now set to 'Greta'\n  Update: reference is now set to 'Leif'\n  Update: liaison is now set to 'Tariq'\n  Update: assignment is now set to 'Vesna'\n  Update: delegate is now set to 'Orla'\n  Update: registry is now set to 'Orla'\n  Update: dispatch is now set to 'Femi'\n  Update: contact is now set to 'Paloma'\n  Update: location is now set to 'Priya'\n  Update: reference is now set to 'Wren'\n  Update: liaison is now set to 'Runa'\n  Update: assignment is now set to 'Nalini'\n  Update: delegate is now set to 'Olena'\n  Update: registry is now set to 'Tariq'\n  Update: dispatch is now set to 'Tala'\n  Update: contact is now set to 'Soren'\n  Update: location is now set to 'Willa'\n  Update: reference is now set to 'Olena'\n  Update: liaison is now set to 'Sigrid'\n  Update: assignment is now set to 'Vesna'\n  Update: delegate is now set to 'Gael'\n  Update: registry is now set to 'Leif'\n  Update: dispatch is now set to 'Yuki'\n  Update: contact is now set to 'Amara'\n  Update: location is now set to 'Qadir'\n  Update: reference is now set to 'Zora'\n  Update: liaison is now set to 'Adaeze'\n  Update: assignment is now set to 'Runa'\n  Update: delegate is now set to 'Adaeze'\n  Update: registry is now set to 'Nalini'\n  Update: dispatch is now set to 'Adaeze'\n  Update: contact is now set to 'Tala'\n  Update: location is now set to 'Orla'\n  Update: reference is now set to 'Adaeze'\n  Update: liaison is now set to 'Gael'\n  Update: assignment is now set to 'Joaquin'\n  Update: delegate is now set to 'Femi'\n  Update: registry is now set to 'Joelle'\n  Update: dispatch is now set to 'Tala'\n  Update: contact is now set to 'Gael'\n  Update: location is now set to 'Bashir'\n  Update: reference is now set to 'Ravi'\n  Update: liaison is now set to 'Viktor'\n  Update: assignment is now set to 'Priya'\n  Update: delegate is now set to 'Magnus'\n  Update: registry is now set to 'Ines'\n  Update: dispatch is now set to 'Gael'\n  Update: contact is now set to 'Zora'\n  Update: location is now set to 'Wren'\n  Update: reference is now set to 'Lumi'\n  Update: liaison is now set to 'Leif'\n  Update: assignment is now set to 'Soren'\n  Update: delegate is now set to 'Joaquin'\n  Update: registry is now set to 'Orla'\n  Update: dispatch is now set to 'Haruto'\n  Update: contact is now set to 'Hana'\n  Update: location is now set to 'Yuki'\n  Update: reference is now set to 'Idris'\n  Update: liaison is now set to 'Colette'\n  Update: assignment is now set to 'Yara'\n  Update: delegate is now set to 'Amara'\n  Update: registry is now set to 'Maren'\n  Update: dispatch is now set to 'Tala'\n  Update: contact is now set to 'Wren'\n  Update: location is now set to 'Leif'\n  Update: reference is now set to 'Gael'\n  Update: liaison is now set to 'Sigrid'\n  Update: assignment is now set to 'Kaia'\n  Update: delegate is now set to 'Freya'\n  Update: registry is now set to 'Qadir'\n  Update: dispatch is now set to 'Runa'\n  Update: contact is now set to 'Magnus'\n  Update: location is now set to 'Nalini'\n  Update: reference is now set to 'Uma'\n  Update: liaison is now set to 'Kaia'\n  Update: assignment is now set to 'Sigrid'\n  Update: delegate is now set to 'Ines'\n  Update: registry is now set to 'Paloma'\n  Update: dispatch is now set to 'Haruto'\n  Update: contact is now set to 'Runa'\n  Update: location is now set to 'Elara'\n  Update: reference is now set to 'Joaquin'\n  Update: liaison is now set to 'Runa'\n  Update: assignment is now set to 'Xander'\n  Update: delegate is now set to 'Kaia'\n  Update: registry is now set to 'Xander'\n  Update: dispatch is now set to 'Elio'\n  Update: contact is now set to 'Soren'\n  Update: location is now set to 'Colette'\n  Update: reference is now set to 'Olena'\n  Update: liaison is now set to 'Soren'\n  Update: assignment is now set to 'Joaquin'\n  Update: delegate is now set to 'Ines'\n  Update: registry is now set to 'Greta'\n  Update: dispatch is now set to 'Wren'\n  Update: contact is now set to 'Nalini'\n  Update: location is now set to 'Dariush'\n  Update: reference is now set to 'Tala'\n  Update: liaison is now set to 'Orla'\n  Update: assignment is now set to 'Qadir'\n  Update: delegate is now set to 'Runa'\n  Update: registry is now set to 'Zora'\n  Update: dispatch is now set to 'Vesna'\n  Update: contact is now set to 'Zora'\n  Update: location is now set to 'Adaeze'\n  Update: reference is now set to 'Elio'\n  Update: liaison is now set to 'Maren'\n  Update: assignment is now set to 'Joelle'\n  Update: delegate is now set to 'Dariush'\n  Update: registry is now set to 'Bram'\n  Update: dispatch is now set to 'Gael'\n  Update: contact is now set to 'Nico'\n  Update: location is now set to 'Freya'\n  Update: reference is now set to 'Nalini'\n  Update: liaison is now set to 'Colette'\n  Update: assignment is now set to 'Nalini'\n  Update: delegate is now set to 'Runa'\n  Update: registry is now set to 'Zora'\n  Update: dispatch is now set to 'Dmitri'\n  Update: contact is now set to 'Maren'\n  Update: location is now set to 'Nalini'\n  Update: reference is now set to 'Bashir'\n  Update: liaison is now set to 'Ines'\n  Update: assignment is now set to 'Maren'\n  Update: delegate is now set to 'Dariush'\n  Update: registry is now set to 'Idris'\n  Update: dispatch is now set to 'Qadir'\n  Update: contact is now set to 'Paloma'\n  Update: location is now set to 'Orla'\n  Update: reference is now set to 'Sigrid'\n  Update: liaison is now set to 'Orla'\n  Update: assignment is now set to 'Nalini'\n  Update: delegate is now set to 'Elio'\n  Update: registry is now set to 'Hana'\n  Update: dispatch is now set to 'Adaeze'\n  Update: contact is now set to 'Maren'\n  Update: location is now set to 'Vesna'\n  Update: reference is now set to 'Femi'\n  Update: liaison is now set to 'Nalini'\n  Update: assignment is now set to 'Nico'\n  Update: delegate is now set to 'Ugo'\n  Update: registry is now set to 'Haruto'\n  Update: dispatch is now set to 'Greta'\n  Update: contact is now set to 'Olena'\n  Update: location is now set to 'Elara'\n  Update: reference is now set to 'Joaquin'\n  Update: liaison is now set to 'Greta'\n  Update: assignment is now set to 'Lumi'\n  Update: delegate is now set to 'Colette'\n  Update: registry is now set to 'Yara'\n  Update: dispatch is now set to 'Elara'\n  Update: contact is now set to 'Nalini'\n  Update: location is now set to 'Femi'\n  Update: reference is now set to 'Zora'\n  Update: liaison is now set to 'Yuki'\n  Update: assignment is now set to 'Colette'\n  Update: delegate is now set to 'Elara'\n  Update: registry is now set to 'Priya'\n  Update: dispatch is now set to 'Dariush'\n  Update: contact is now set to 'Vesna'\n  Update: location is now set to 'Amara'\n  Update: reference is now set to 'Bram'\n  Update: liaison is now set to 'Willa'\n  Update: assignment is now set to 'Elio'\n  Update: delegate is now set to 'Soren'\n  Update: registry is now set to 'Magnus'\n  Update: dispatch is now set to 'Amara'\n  Update: contact is now set to 'Amara'\n  Update: location is now set to 'Greta'\n  Update: reference is now set to 'Zain'\n  Update: liaison is now set to 'Dmitri'\n  Update: assignment is now set to 'Vesna'\n  Update: delegate is now set to 'Magnus'\n  Update: registry is now set to 'Priya'\n  Update: dispatch is now set to 'Lumi'\n  Update: contact is now set to 'Gael'\n  Update: location is now set to 'Leif'\n  Update: reference is now set to 'Tala'\n  Update: liaison is now set to 'Tala'\n  Update: assignment is now set to 'Elio'\n  Update: delegate is now set to 'Kenji'\n  Update: registry is now set to 'Paloma'\n  Update: dispatch is now set to 'Femi'\n  Update: contact is now set to 'Ines'\n  Update: location is now set to 'Adaeze'\n  Update: reference is now set to 'Uma'\n  Update: liaison is now set to 'Ugo'\n  Update: assignment is now set to 'Dmitri'\n  Update: delegate is now set to 'Bram'\n  Update: registry is now set to 'Greta'\n  Update: dispatch is now set to 'Elio'\n  Update: contact is now set to 'Colette'\n  Update: location is now set to 'Qadir'\n  Update: reference is now set to 'Yuki'\n  Update: liaison is now set to 'Freya'\n  Update: assignment is now set to 'Dariush'\n  Update: delegate is now set to 'Yara'\n  Update: registry is now set to 'Amara'\n  Update: dispatch is now set to 'Tala'\n  Update: contact is now set to 'Soren'\n  Update: location is now set to 'Joaquin'\n  Update: reference is now set to 'Qadir'\n  Update: liaison is now set to 'Zain'\n  Update: assignment is now set to 'Leif'\n  Update: delegate is now set to 'Olena'\n  Update: registry is now set to 'Bram'\n  Update: dispatch is now set to 'Ugo'\n  Update: contact is now set to 'Zora'\n  Update: location is now set to 'Femi'\n  Update: reference is now set to 'Ines'\n  Update: liaison is now set to 'Uma'\n  Update: assignment is now set to 'Tariq'\n  Update: delegate is now set to 'Yuki'\n  Update: registry is now set to 'Sigrid'\n  Update: dispatch is now set to 'Elio'\n  Update: contact is now set to 'Olena'\n  Update: location is now set to 'Hana'\n  Update: reference is now set to 'Lumi'\n  Update: liaison is now set to 'Soren'\n  Update: assignment is now set to 'Adaeze'\n  Update: delegate is now set to 'Kaia'\n  Update: registry is now set to 'Bashir'\n  Update: dispatch is now set to 'Yara'\n  Update: contact is now set to 'Priya'\n  Update: location is now set to 'Yuki'\n  Update: reference is now set to 'Kenji'\n  Update: liaison is now set to 'Kenji'\n  Update: assignment is now set to 'Dmitri'\n  Update: delegate is now set to 'Ravi'\n  Update: registry is now set to 'Greta'\n  Update: dispatch is now set to 'Nalini'\n  Update: contact is now set to 'Amara'\n  Update: location is now set to 'Bram'\n  Update: reference is now set to 'Haruto'\n  Update: liaison is now set to 'Femi'\n  Update: assignment is now set to 'Priya'\n  Update: delegate is now set to 'Greta'\n  Update: registry is now set to 'Elara'\n  Update: dispatch is now set to 'Tariq'\n  Update: contact is now set to 'Freya'\n  Update: location is now set to 'Ines'\n  Update: reference is now set to 'Ines'\n  Update: liaison is now set to 'Willa'\n  Update: assignment is now set to 'Xander'\n  Update: delegate is now set to 'Qadir'\n  Update: registry is now set to 'Zain'\n  Update: dispatch is now set to 'Amara'\n  Update: contact is now set to 'Uma'\n  Update: location is now set to 'Kaia'\n  Update: reference is now set to 'Hana'\n  Update: liaison is now set to 'Kaia'\n  Update: assignment is now set to 'Elara'\n  Update: delegate is now set to 'Idris'\n  Update: registry is now set to 'Celine'\n  Update: dispatch is now set to 'Soren'\n  Update: contact is now set to 'Zora'\n  Update: location is now set to 'Zora'\n  Update: reference is now set to 'Orla'\n  Update: liaison is now set to 'Bram'\n  Update: assignment is now set to 'Yara'\n  Update: delegate is now set to 'Bashir'\n  Update: registry is now set to 'Bram'\n  Update: dispatch is now set to 'Viktor'\n  Update: contact is now set to 'Lumi'\n  Update: location is now set to 'Adaeze'\n  Update: reference is now set to 'Leif'\n  Update: liaison is now set to 'Zora'\n  Update: assignment is now set to 'Yuki'\n  Update: delegate is now set to 'Nico'\n  Update: registry is now set to 'Tariq'\n  Update: dispatch is now set to 'Maren'\n  Update: contact is now set to 'Olena'\n  Update: location is now set to 'Olena'\n  Update: reference is now set to 'Joaquin'\n  Update: liaison is now set to 'Idris'\n  Update: assignment is now set to 'Kaia'\n  Update: delegate is now set to 'Nalini'\n  Update: registry is now set to 'Wren'\n  Update: dispatch is now set to 'Dmitri'\n  Update: contact is now set to 'Qadir'\n  Update: location is now set to 'Hana'\n  Update: reference is now set to 'Gael'\n  Update: liaison is now set to 'Femi'\n  Update: assignment is now set to 'Viktor'\n  Update: delegate is now set to 'Amara'\n  Update: registry is now set to 'Bashir'\n  Update: dispatch is now set to 'Ugo'\n  Update: contact is now set to 'Runa'\n  Update: location is now set to 'Ines'\n  Update: reference is now set to 'Ines'\n  Update: liaison is now set to 'Elio'\n  Update: assignment is now set to 'Sigrid'\n  Update: delegate is now set to 'Nico'\n  Update: registry is now set to 'Kenji'\n  Update: dispatch is now set to 'Femi'\n  Update: contact is now set to 'Lumi'\n  Update: location is now set to 'Adaeze'\n  Update: reference is now set to 'Maren'\n  Update: liaison is now set to 'Greta'\n  Update: assignment is now set to 'Olena'\n  Update: delegate is now set to 'Willa'\n  Update: registry is now set to 'Sigrid'\n  Update: dispatch is now set to 'Celine'\n  Update: contact is now set to 'Viktor'\n  Update: location is now set to 'Maren'\n  Update: reference is now set to 'Sigrid'\n  Update: liaison is now set to 'Zora'\n  Update: assignment is now set to 'Haruto'\n  Update: delegate is now set to 'Adaeze'\n  Update: registry is now set to 'Lumi'\n  Update: dispatch is now set to 'Willa'\n  Update: contact is now set to 'Maren'\n  Update: location is now set to 'Zain'\n  Update: reference is now set to 'Greta'\n  Update: liaison is now set to 'Greta'\n  Update: assignment is now set to 'Joelle'\n  Update: delegate is now set to 'Joelle'\n  Update: registry is now set to 'Wren'\n  Update: dispatch is now set to 'Idris'\n  Update: contact is now set to 'Leif'\n  Update: location is now set to 'Priya'\n  Update: reference is now set to 'Hana'\n  Update: liaison is now set to 'Colette'\n  Update: assignment is now set to 'Tala'\n  Update: delegate is now set to 'Lumi'\n  Update: registry is now set to 'Elara'\n  Update: dispatch is now set to 'Ravi'\n  Update: contact is now set to 'Haruto'\n  Update: location is now set to 'Elara'\n  Update: reference is now set to 'Olena'\n  Update: liaison is now set to 'Lumi'\n  Update: assignment is now set to 'Orla'\n  Update: delegate is now set to 'Kaia'\n  Update: registry is now set to 'Ravi'\n  Update: dispatch is now set to 'Magnus'\n  Update: contact is now set to 'Vesna'\n  Update: location is now set to 'Bashir'\n  Update: reference is now set to 'Adaeze'\n  Update: liaison is now set to 'Joaquin'\n  Update: assignment is now set to 'Vesna'\n  Update: delegate is now set to 'Adaeze'\n  Update: registry is now set to 'Idris'\n  Update: dispatch is now set to 'Soren'\n  Update: contact is now set to 'Leif'\n  Update: location is now set to 'Sigrid'\n  Update: reference is now set to 'Yuki'\n  Update: liaison is now set to 'Tariq'\n  Update: assignment is now set to 'Willa'\n  Update: delegate is now set to 'Viktor'\n  Update: registry is now set to 'Magnus'\n  Update: dispatch is now set to 'Tariq'\n  Update: contact is now set to 'Celine'\n  Update: location is now set to 'Colette'\n  Update: reference is now set to 'Lumi'\n  Update: liaison is now set to 'Nalini'\n  Update: assignment is now set to 'Haruto'\n  Update: delegate is now set to 'Ravi'\n  Update: registry is now set to 'Uma'\n  Update: dispatch is now set to 'Femi'\n  Update: contact is now set to 'Wren'\n  Update: location is now set to 'Elara'\n  Update: reference is now set to 'Yuki'\n  Update: liaison is now set to 'Viktor'\n  Update: assignment is now set to 'Ines'\n  Update: delegate is now set to 'Xander'\n  Update: registry is now set to 'Elara'\n  Update: dispatch is now set to 'Hana'\n  Update: contact is now set to 'Femi'\n  Update: location is now set to 'Kaia'\n  Update: reference is now set to 'Qadir'\n  Update: liaison is now set to 'Maren'\n  Update: assignment is now set to 'Elio'\n  Update: delegate is now set to 'Idris'\n  Update: registry is now set to 'Bram'\n  Update: dispatch is now set to 'Dmitri'\n  Update: contact is now set to 'Kaia'\n  Update: location is now set to 'Xander'\n  Update: reference is now set to 'Paloma'\n  Update: liaison is now set to 'Leif'\n  Update: assignment is now set to 'Haruto'\n  Update: delegate is now set to 'Haruto'\n  Update: registry is now set to 'Nalini'\n  Update: dispatch is now set to 'Amara'\n  Update: contact is now set to 'Dmitri'\n\nWhat is the FINAL value of each record?\nANSWER:\n- location: [final value]\n- reference: [final value]\n- liaison: [final value]\n- assignment: [final value]\n- delegate: [final value]\n- registry: [final value]\n- dispatch: [final value]\n- contact: [final value]\n\nAlso answer these verification questions (Yes or No):\nV1. Was 'Amara' ever assigned to location? [Yes/No]\nV2. Was 'Hana' ever assigned to reference? [Yes/No]\nV3. Was 'Ines' ever assigned to liaison? [Yes/No]\nV4. Was 'Dariush' ever assigned to assignment? [Yes/No]",
  "gold_json": "{\"final_values\": {\"location\": \"Xander\", \"reference\": \"Paloma\", \"liaison\": \"Leif\", \"assignment\": \"Haruto\", \"delegate\": \"Haruto\", \"registry\": \"Nalini\", \"dispatch\": \"Amara\", \"contact\": \"Dmitri\"}, \"key_names\": [\"location\", \"reference\", \"liaison\", \"assignment\", \"delegate\", \"registry\", \"dispatch\", \"contact\"]}"
 },
 {
  "task_id": "interference_frontier_039",
  "task_type": "interference",
  "difficulty": "Frontier",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: contact is now set to 'Reykjavik'\n  Update: coordinator is now set to 'Cusco'\n  Update: liaison is now set to 'Recife'\n  Update: delegate is now set to 'Mandalay'\n  Update: location is now set to 'Tallinn'\n  Update: reference is now set to 'Valetta'\n  Update: assignment is now set to 'Kotor'\n  Update: registry is now set to 'Oulu'\n  Update: contact is now set to 'Zanzibar'\n  Update: coordinator is now set to 'Cartagena'\n  Update: liaison is now set to 'Fez'\n  Update: delegate is now set to 'Gdansk'\n  Update: location is now set to 'Luang Prabang'\n  Update: reference is now set to 'Reykjavik'\n  Update: assignment is now set to 'Kumasi'\n  Update: registry is now set to 'Reykjavik'\n  Update: contact is now set to 'Valetta'\n  Update: coordinator is now set to 'Fez'\n  Update: liaison is now set to 'Zanzibar'\n  Update: delegate is now set to 'Oulu'\n  Update: location is now set to 'Kumasi'\n  Update: reference is now set to 'Cartagena'\n  Update: assignment is now set to 'Oulu'\n  Update: registry is now set to 'Kotor'\n  Update: contact is now set to 'Mandalay'\n  Update: coordinator is now set to 'Zanzibar'\n  Update: liaison is now set to 'Plovdiv'\n  Update: delegate is now set to 'Fez'\n  Update: location is now set to 'Oulu'\n  Update: reference is now set to 'Ulaanbaatar'\n  Update: assignment is now set to 'Zanzibar'\n  Update: registry is now set to 'Tbilisi'\n  Update: contact is now set to 'Cusco'\n  Update: coordinator is now set to 'Tbilisi'\n  Update: liaison is now set to 'Tbilisi'\n  Update: delegate is now set to 'Gdansk'\n  Update: location is now set to 'Bruges'\n  Update: reference is now set to 'Plovdiv'\n  Update: assignment is now set to 'Kumasi'\n  Update: registry is now set to 'Jaipur'\n  Update: contact is now set to 'Cartagena'\n  Update: coordinator is now set to 'Kumasi'\n  Update: liaison is now set to 'Luang Prabang'\n  Update: delegate is now set to 'Ulaanbaatar'\n  Update: location is now set to 'Ulaanbaatar'\n  Update: reference is now set to 'Ulaanbaatar'\n  Update: assignment is now set to 'Gdansk'\n  Update: registry is now set to 'Kumasi'\n  Update: contact is now set to 'Cusco'\n  Update: coordinator is now set to 'Jaipur'\n  Update: liaison is now set to 'Trieste'\n  Update: delegate is now set to 'Reykjavik'\n  Update: location is now set to 'Fez'\n  Update: reference is now set to 'Tallinn'\n  Update: assignment is now set to 'Tbilisi'\n  Update: registry is now set to 'Jaipur'\n  Update: contact is now set to 'Tallinn'\n  Update: coordinator is now set to 'Mandalay'\n  Update: liaison is now set to 'Jaipur'\n  Update: delegate is now set to 'Cartagena'\n  Update: location is now set to 'Kotor'\n  Update: reference is now set to 'Kotor'\n  Update: assignment is now set to 'Cusco'\n  Update: registry is now set to 'Mandalay'\n  Update: contact is now set to 'Jaipur'\n  Update: coordinator is now set to 'Luang Prabang'\n  Update: liaison is now set to 'Reykjavik'\n  Update: delegate is now set to 'Tbilisi'\n  Update: location is now set to 'Ulaanbaatar'\n  Update: reference is now set to 'Valetta'\n  Update: assignment is now set to 'Reykjavik'\n  Update: registry is now set to 'Trieste'\n  Update: contact is now set to 'Zanzibar'\n  Update: coordinator is now set to 'Oulu'\n  Update: liaison is now set to 'Recife'\n  Update: delegate is now set to 'Mandalay'\n  Update: location is now set to 'Cartagena'\n  Update: reference is now set to 'Ulaanbaatar'\n  Update: assignment is now set to 'Fez'\n  Update: registry is now set to 'Jaipur'\n  Update: contact is now set to 'Plovdiv'\n  Update: coordinator is now set to 'Tbilisi'\n  Update: liaison is now set to 'Reykjavik'\n  Update: delegate is now set to 'Luang Prabang'\n  Update: location is now set to 'Reykjavik'\n  Update: reference is now set to 'Reykjavik'\n  Update: assignment is now set to 'Luang Prabang'\n  Update: registry is now set to 'Luang Prabang'\n  Update: contact is now set to 'Mandalay'\n  Update: coordinator is now set to 'Tallinn'\n  Update: liaison is now set to 'Mandalay'\n  Update: delegate is now set to 'Tbilisi'\n  Update: location is now set to 'Tbilisi'\n  Update: reference is now set to 'Gdansk'\n  Update: assignment is now set to 'Fez'\n  Update: registry is now set to 'Valetta'\n  Update: contact is now set to 'Ulaanbaatar'\n  Update: coordinator is now set to 'Kumasi'\n  Update: liaison is now set to 'Luang Prabang'\n  Update: delegate is now set to 'Jaipur'\n  Update: location is now set to 'Fez'\n  Update: reference is now set to 'Fez'\n  Update: assignment is now set to 'Tallinn'\n  Update: registry is now set to 'Trieste'\n  Update: contact is now set to 'Valetta'\n  Update: coordinator is now set to 'Gdansk'\n  Update: liaison is now set to 'Gdansk'\n  Update: delegate is now set to 'Fez'\n  Update: location is now set to 'Recife'\n  Update: reference is now set to 'Luang Prabang'\n  Update: assignment is now set to 'Kumasi'\n  Update: registry is now set to 'Recife'\n  Update: contact is now set to 'Kumasi'\n  Update: coordinator is now set to 'Bruges'\n  Update: liaison is now set to 'Luang Prabang'\n  Update: delegate is now set to 'Recife'\n  Update: location is now set to 'Mandalay'\n  Update: reference is now set to 'Fez'\n  Update: assignment is now set to 'Reykjavik'\n  Update: registry is now set to 'Gdansk'\n  Update: contact is now set to 'Reykjavik'\n  Update: coordinator is now set to 'Ulaanbaatar'\n  Update: liaison is now set to 'Tbilisi'\n  Update: delegate is now set to 'Kotor'\n  Update: location is now set to 'Tallinn'\n  Update: reference is now set to 'Cartagena'\n  Update: assignment is now set to 'Oulu'\n  Update: registry is now set to 'Cartagena'\n  Update: contact is now set to 'Cartagena'\n  Update: coordinator is now set to 'Luang Prabang'\n  Update: liaison is now set to 'Bruges'\n  Update: delegate is now set to 'Kumasi'\n  Update: location is now set to 'Fez'\n  Update: reference is now set to 'Ulaanbaatar'\n  Update: assignment is now set to 'Cartagena'\n  Update: registry is now set to 'Gdansk'\n  Update: contact is now set to 'Jaipur'\n  Update: coordinator is now set to 'Bruges'\n  Update: liaison is now set to 'Kotor'\n  Update: delegate is now set to 'Bruges'\n  Update: location is now set to 'Kumasi'\n  Update: reference is now set to 'Kumasi'\n  Update: assignment is now set to 'Mandalay'\n  Update: registry is now set to 'Cartagena'\n  Update: contact is now set to 'Kumasi'\n  Update: coordinator is now set to 'Valetta'\n  Update: liaison is now set to 'Valetta'\n  Update: delegate is now set to 'Mandalay'\n  Update: location is now set to 'Jaipur'\n  Update: reference is now set to 'Cartagena'\n  Update: assignment is now set to 'Ulaanbaatar'\n  Update: registry is now set to 'Fez'\n  Update: contact is now set to 'Bruges'\n  Update: coordinator is now set to 'Tallinn'\n  Update: liaison is now set to 'Fez'\n  Update: delegate is now set to 'Fez'\n  Update: location is now set to 'Fez'\n  Update: reference is now set to 'Cusco'\n  Update: assignment is now set to 'Kotor'\n  Update: registry is now set to 'Oulu'\n  Update: contact is now set to 'Plovdiv'\n  Update: coordinator is now set to 'Gdansk'\n  Update: liaison is now set to 'Mandalay'\n  Update: delegate is now set to 'Bruges'\n  Update: location is now set to 'Luang Prabang'\n  Update: reference is now set to 'Ulaanbaatar'\n  Update: assignment is now set to 'Valetta'\n  Update: registry is now set to 'Reykjavik'\n  Update: contact is now set to 'Cusco'\n  Update: coordinator is now set to 'Ulaanbaatar'\n  Update: liaison is now set to 'Reykjavik'\n  Update: delegate is now set to 'Cartagena'\n  Update: location is now set to 'Zanzibar'\n  Update: reference is now set to 'Cusco'\n  Update: assignment is now set to 'Bruges'\n  Update: registry is now set to 'Mandalay'\n  Update: contact is now set to 'Oulu'\n  Update: coordinator is now set to 'Tbilisi'\n  Update: liaison is now set to 'Mandalay'\n  Update: delegate is now set to 'Cusco'\n  Update: location is now set to 'Cartagena'\n  Update: reference is now set to 'Gdansk'\n  Update: assignment is now set to 'Trieste'\n  Update: registry is now set to 'Fez'\n  Update: contact is now set to 'Cusco'\n  Update: coordinator is now set to 'Recife'\n  Update: liaison is now set to 'Recife'\n  Update: delegate is now set to 'Ulaanbaatar'\n  Update: location is now set to 'Ulaanbaatar'\n  Update: reference is now set to 'Zanzibar'\n  Update: assignment is now set to 'Jaipur'\n  Update: registry is now set to 'Kumasi'\n  Update: contact is now set to 'Luang Prabang'\n  Update: coordinator is now set to 'Oulu'\n  Update: liaison is now set to 'Kotor'\n  Update: delegate is now set to 'Kumasi'\n  Update: location is now set to 'Luang Prabang'\n  Update: reference is now set to 'Mandalay'\n  Update: assignment is now set to 'Cartagena'\n  Update: registry is now set to 'Trieste'\n  Update: contact is now set to 'Tbilisi'\n  Update: coordinator is now set to 'Jaipur'\n  Update: liaison is now set to 'Tallinn'\n  Update: delegate is now set to 'Cartagena'\n  Update: location is now set to 'Jaipur'\n  Update: reference is now set to 'Zanzibar'\n  Update: assignment is now set to 'Oulu'\n  Update: registry is now set to 'Recife'\n  Update: contact is now set to 'Mandalay'\n  Update: coordinator is now set to 'Zanzibar'\n  Update: liaison is now set to 'Zanzibar'\n  Update: delegate is now set to 'Ulaanbaatar'\n  Update: location is now set to 'Tallinn'\n  Update: reference is now set to 'Bruges'\n  Update: assignment is now set to 'Recife'\n  Update: registry is now set to 'Kotor'\n  Update: contact is now set to 'Kumasi'\n  Update: coordinator is now set to 'Oulu'\n  Update: liaison is now set to 'Recife'\n  Update: delegate is now set to 'Tbilisi'\n  Update: location is now set to 'Ulaanbaatar'\n  Update: reference is now set to 'Tbilisi'\n  Update: assignment is now set to 'Cusco'\n  Update: registry is now set to 'Ulaanbaatar'\n  Update: contact is now set to 'Kotor'\n  Update: coordinator is now set to 'Kotor'\n  Update: liaison is now set to 'Oulu'\n  Update: delegate is now set to 'Recife'\n  Update: location is now set to 'Bruges'\n  Update: reference is now set to 'Jaipur'\n  Update: assignment is now set to 'Tbilisi'\n  Update: registry is now set to 'Cartagena'\n  Update: contact is now set to 'Fez'\n  Update: coordinator is now set to 'Tbilisi'\n  Update: liaison is now set to 'Fez'\n  Update: delegate is now set to 'Zanzibar'\n  Update: location is now set to 'Gdansk'\n  Update: reference is now set to 'Cartagena'\n  Update: assignment is now set to 'Zanzibar'\n  Update: registry is now set to 'Cusco'\n  Update: contact is now set to 'Mandalay'\n  Update: coordinator is now set to 'Fez'\n  Update: liaison is now set to 'Gdansk'\n  Update: delegate is now set to 'Kumasi'\n  Update: location is now set to 'Cusco'\n  Update: reference is now set to 'Trieste'\n  Update: assignment is now set to 'Luang Prabang'\n  Update: registry is now set to 'Valetta'\n  Update: contact is now set to 'Luang Prabang'\n  Update: coordinator is now set to 'Plovdiv'\n  Update: liaison is now set to 'Luang Prabang'\n  Update: delegate is now set to 'Jaipur'\n  Update: location is now set to 'Recife'\n  Update: reference is now set to 'Reykjavik'\n  Update: assignment is now set to 'Cusco'\n  Update: registry is now set to 'Ulaanbaatar'\n  Update: contact is now set to 'Ulaanbaatar'\n  Update: coordinator is now set to 'Tallinn'\n  Update: liaison is now set to 'Ulaanbaatar'\n  Update: delegate is now set to 'Trieste'\n  Update: location is now set to 'Tallinn'\n  Update: reference is now set to 'Oulu'\n  Update: assignment is now set to 'Jaipur'\n  Update: registry is now set to 'Tbilisi'\n  Update: contact is now set to 'Mandalay'\n  Update: coordinator is now set to 'Bruges'\n  Update: liaison is now set to 'Trieste'\n  Update: delegate is now set to 'Tbilisi'\n  Update: location is now set to 'Zanzibar'\n  Update: reference is now set to 'Recife'\n  Update: assignment is now set to 'Fez'\n  Update: registry is now set to 'Kumasi'\n  Update: contact is now set to 'Oulu'\n  Update: coordinator is now set to 'Plovdiv'\n  Update: liaison is now set to 'Valetta'\n  Update: delegate is now set to 'Mandalay'\n  Update: location is now set to 'Kotor'\n  Update: reference is now set to 'Tallinn'\n  Update: assignment is now set to 'Bruges'\n  Update: registry is now set to 'Recife'\n  Update: contact is now set to 'Tallinn'\n  Update: coordinator is now set to 'Bruges'\n  Update: liaison is now set to 'Trieste'\n  Update: delegate is now set to 'Oulu'\n  Update: location is now set to 'Valetta'\n  Update: reference is now set to 'Jaipur'\n  Update: assignment is now set to 'Reykjavik'\n  Update: registry is now set to 'Reykjavik'\n  Update: contact is now set to 'Kotor'\n  Update: coordinator is now set to 'Tallinn'\n  Update: liaison is now set to 'Tbilisi'\n  Update: delegate is now set to 'Tallinn'\n  Update: location is now set to 'Tbilisi'\n  Update: reference is now set to 'Bruges'\n  Update: assignment is now set to 'Cartagena'\n  Update: registry is now set to 'Trieste'\n  Update: contact is now set to 'Tbilisi'\n  Update: coordinator is now set to 'Gdansk'\n  Update: liaison is now set to 'Tallinn'\n  Update: delegate is now set to 'Cusco'\n  Update: location is now set to 'Cartagena'\n  Update: reference is now set to 'Ulaanbaatar'\n  Update: assignment is now set to 'Ulaanbaatar'\n  Update: registry is now set to 'Recife'\n  Update: contact is now set to 'Bruges'\n  Update: coordinator is now set to 'Kotor'\n  Update: liaison is now set to 'Gdansk'\n  Update: delegate is now set to 'Plovdiv'\n  Update: location is now set to 'Luang Prabang'\n  Update: reference is now set to 'Oulu'\n  Update: assignment is now set to 'Jaipur'\n  Update: registry is now set to 'Fez'\n  Update: contact is now set to 'Plovdiv'\n  Update: coordinator is now set to 'Cartagena'\n  Update: liaison is now set to 'Fez'\n  Update: delegate is now set to 'Gdansk'\n  Update: location is now set to 'Plovdiv'\n  Update: reference is now set to 'Luang Prabang'\n  Update: assignment is now set to 'Kumasi'\n  Update: registry is now set to 'Mandalay'\n  Update: contact is now set to 'Oulu'\n  Update: coordinator is now set to 'Valetta'\n  Update: liaison is now set to 'Reykjavik'\n  Update: delegate is now set to 'Luang Prabang'\n  Update: location is now set to 'Trieste'\n  Update: reference is now set to 'Jaipur'\n  Update: assignment is now set to 'Trieste'\n  Update: registry is now set to 'Reykjavik'\n  Update: contact is now set to 'Trieste'\n  Update: coordinator is now set to 'Bruges'\n  Update: liaison is now set to 'Recife'\n  Update: delegate is now set to 'Reykjavik'\n  Update: location is now set to 'Recife'\n  Update: reference is now set to 'Fez'\n  Update: assignment is now set to 'Zanzibar'\n  Update: registry is now set to 'Plovdiv'\n  Update: contact is now set to 'Tallinn'\n  Update: coordinator is now set to 'Gdansk'\n  Update: liaison is now set to 'Kumasi'\n  Update: delegate is now set to 'Cartagena'\n  Update: location is now set to 'Fez'\n  Update: reference is now set to 'Trieste'\n  Update: assignment is now set to 'Recife'\n  Update: registry is now set to 'Valetta'\n  Update: contact is now set to 'Recife'\n  Update: coordinator is now set to 'Ulaanbaatar'\n  Update: liaison is now set to 'Tbilisi'\n  Update: delegate is now set to 'Bruges'\n  Update: location is now set to 'Jaipur'\n  Update: reference is now set to 'Oulu'\n  Update: assignment is now set to 'Tallinn'\n  Update: registry is now set to 'Mandalay'\n  Update: contact is now set to 'Reykjavik'\n  Update: coordinator is now set to 'Fez'\n  Update: liaison is now set to 'Recife'\n  Update: delegate is now set to 'Mandalay'\n  Update: location is now set to 'Oulu'\n  Update: reference is now set to 'Valetta'\n  Update: assignment is now set to 'Zanzibar'\n  Update: registry is now set to 'Cartagena'\n  Update: contact is now set to 'Fez'\n  Update: coordinator is now set to 'Gdansk'\n  Update: liaison is now set to 'Reykjavik'\n  Update: delegate is now set to 'Recife'\n  Update: location is now set to 'Mandalay'\n  Update: reference is now set to 'Reykjavik'\n  Update: assignment is now set to 'Recife'\n  Update: registry is now set to 'Luang Prabang'\n  Update: contact is now set to 'Oulu'\n  Update: coordinator is now set to 'Ulaanbaatar'\n  Update: liaison is now set to 'Mandalay'\n  Update: delegate is now set to 'Bruges'\n  Update: location is now set to 'Trieste'\n  Update: reference is now set to 'Tallinn'\n  Update: assignment is now set to 'Cartagena'\n  Update: registry is now set to 'Kumasi'\n  Update: contact is now set to 'Gdansk'\n  Update: coordinator is now set to 'Recife'\n  Update: liaison is now set to 'Recife'\n  Update: delegate is now set to 'Luang Prabang'\n  Update: location is now set to 'Kotor'\n  Update: reference is now set to 'Tbilisi'\n  Update: assignment is now set to 'Cusco'\n  Update: registry is now set to 'Cartagena'\n  Update: contact is now set to 'Mandalay'\n  Update: coordinator is now set to 'Mandalay'\n  Update: liaison is now set to 'Cartagena'\n  Update: delegate is now set to 'Bruges'\n  Update: location is now set to 'Kumasi'\n  Update: reference is now set to 'Gdansk'\n  Update: assignment is now set to 'Valetta'\n  Update: registry is now set to 'Valetta'\n  Update: contact is now set to 'Oulu'\n  Update: coordinator is now set to 'Zanzibar'\n  Update: liaison is now set to 'Recife'\n  Update: delegate is now set to 'Luang Prabang'\n  Update: location is now set to 'Kotor'\n  Update: reference is now set to 'Valetta'\n  Update: assignment is now set to 'Reykjavik'\n  Update: registry is now set to 'Fez'\n\nWhat is the FINAL value of each record?\nANSWER:\n- contact: [final value]\n- coordinator: [final value]\n- liaison: [final value]\n- delegate: [final value]\n- location: [final value]\n- reference: [final value]\n- assignment: [final value]\n- registry: [final value]\n\nAlso answer these verification questions (Yes or No):\nV1. Was 'Tbilisi' ever assigned to contact? [Yes/No]\nV2. Was 'Tbilisi' ever assigned to coordinator? [Yes/No]\nV3. Was 'Tbilisi' ever assigned to liaison? [Yes/No]\nV4. Was 'Tallinn' ever assigned to delegate? [Yes/No]",
  "gold_json": "{\"final_values\": {\"contact\": \"Oulu\", \"coordinator\": \"Zanzibar\", \"liaison\": \"Recife\", \"delegate\": \"Luang Prabang\", \"location\": \"Kotor\", \"reference\": \"Valetta\", \"assignment\": \"Reykjavik\", \"registry\": \"Fez\"}, \"key_names\": [\"contact\", \"coordinator\", \"liaison\", \"delegate\", \"location\", \"reference\", \"assignment\", \"registry\"]}"
 }
]
''')

print(f"Loaded {len(DATASET)} items")
for tt in ['interference']:
    count = sum(1 for d in DATASET if d["task_type"] == tt)
    print(f"  {tt}: {count} items")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 4: Execution Loop
# ══════════════════════════════════════════════════════════════════════

TASK_DISPATCH = {
    "interference": cogattention_interference,
}

n_total = len(DATASET)
for i, item in enumerate(DATASET):
    task_fn = TASK_DISPATCH[item["task_type"]]
    print(f"[{i+1}/{n_total}] {item['task_id']} ({item['difficulty']})")
    task_fn.run(
        llm=kbench.llm,
        prompt=item["prompt"],
        gold_json=item["gold_json"],
        task_id=item["task_id"],
        difficulty=item["difficulty"],
    )

print(f"\nCompleted {n_total} items for Attention Capacity")
